In [ ]:
import pandas as pd
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist

import numpy as np
import math
import scipy
import matplotlib.pyplot as plt
import seaborn
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

import os
from scipy.optimize import curve_fit
import re
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler


In [ ]:
# =====================================================================
# GLOBAL FIGURE STYLE — one font family + one font size on every text
# element, and SVG text kept EDITABLE (not outlined) for Illustrator.
# Save any figure as vector with:   fig.savefig("name.svg")
# =====================================================================
import matplotlib as mpl, matplotlib.pyplot as plt, seaborn as sns
sns.set_style("white")                 # grid/colour only (does NOT rescale fonts)
sns.set_context("talk", font_scale=1)
FONT_SIZE = 18                              # <- change this one number to rescale ALL text
mpl.rcParams.update({
    "svg.fonttype": "none",                # export text as editable <text>, not paths
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "font.size":       FONT_SIZE, "axes.titlesize": FONT_SIZE, "axes.labelsize": FONT_SIZE,
    "xtick.labelsize": 16.5, "ytick.labelsize": 16.5,
    "legend.fontsize": FONT_SIZE, "legend.title_fontsize": FONT_SIZE,
    "figure.titlesize": FONT_SIZE, "axes.titleweight": "normal",
    "mathtext.default": "regular",         # ΔG°$_{37}$ etc. match the body font
    "figure.dpi": 600,
})
sns.set_style(rc={"axes.grid": False})
#sns.set_context("talk", font_scale=1.2)

In [ ]:
# === Paper-figure export: one subfolder per panel (figure.svg + data.csv) =====
import os
PAPER_DIR = "Paper_Figures"
def export_panel(name, data=None, save_current_fig=True, note=None):
    """Save the current matplotlib figure and its backing data into Paper_Figures/<name>/."""
    d = os.path.join(PAPER_DIR, name); os.makedirs(d, exist_ok=True)
    if data is not None:
        df = data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
        df.to_csv(os.path.join(d, "data.csv"), index=False)
    if save_current_fig:
        plt.savefig(os.path.join(d, "figure.svg"), bbox_inches="tight")
    if note is not None:
        open(os.path.join(d, "README.txt"), "w").write(note)
    print("exported ->", d)

def export_panel_high(name, data=None, save_current_fig=True):
    """Write the HIGH-reliability variant (figure_high.svg + data_high.csv) into Paper_Figures/<name>/."""
    d = os.path.join(PAPER_DIR, name); os.makedirs(d, exist_ok=True)
    if data is not None:
        df = data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
        df.to_csv(os.path.join(d, "data_high.csv"), index=False)
    if save_current_fig:
        plt.savefig(os.path.join(d, "figure_high.svg"), bbox_inches="tight")
    print("exported (high) ->", d)

In [ ]:
delta_G37 = {
    "AA/TT": -1.00,
    "AT/TA": -0.88,
    "TA/AT": -0.58,
    "CA/GT": -1.45,
    "GT/CA": -1.44,
    "CT/GA": -1.28,
    "GA/CT": -1.30,
    "CG/GC": -2.17,
    "GC/CG": -2.24,
    "GG/CC": -1.84
}
delta_S = {
    "AA/TT": -22.2,
    "AT/TA": -20.4,
    "TA/AT": -21.3,
    "CA/GT": -22.7,
    "GT/CA": -22.4,
    "CT/GA": -21.0,
    "GA/CT": -22.2,
    "CG/GC": -27.2,
    "GC/CG": -24.4,
    "GG/CC": -19.9
}
delta_H = {
    "AA/TT": -7.9,
    "AT/TA": -7.2,
    "TA/AT": -7.2,
    "CA/GT": -8.5,
    "GT/CA": -8.4,
    "CT/GA": -7.8,
    "GA/CT": -8.2,
    "CG/GC": -10.6,
    "GC/CG": -9.8,
    "GG/CC": -8.0
}

init_H = {"AT": 2.3, "GC": 0.1}
init_S = {"AT": 4.1, "GC": -2.8}
init_G = {"AT": 1.03, "GC": 0.98}

sym_H = 0.0
sym_S = -1.4
sym_G = 0.43


In [ ]:
#comp = str.maketrans("ATCG", "TAGC")

comp = str.maketrans("ATCG", "TAGC")

def revcomp(seq):
    return seq.translate(comp)[::-1]


def calc_unified_nn(seq):
    seq = seq.upper()
    rc = revcomp(seq)

    total_H = 0
    total_S = 0
    total_G = 0

    # Sum nearest neighbors
    for i in range(len(seq) - 1):
        top = seq[i:i+2]
        #bottom = rc[len(seq)-i-2:len(seq)-i]  # matching 3'→5' orientation
        bottom = top.translate(comp)
        key = f"{top}/{bottom}"

        if key not in delta_H:
            key_rev = f"{top[::-1]}/{bottom[::-1]}"
            #print(key)
            key = key_rev
            if(key not in delta_H):
                top1 = bottom
                bottom1 = top
                key = f"{top1[::-1]}/{bottom1[::-1]}"
                if(key not in delta_H):
                    raise ValueError(f"Step {key} not found in NN library")

        total_H += delta_H[key]
        total_S += delta_S[key]
        total_G += delta_G37[key]

    # Add initiation terms (two ends)
    for base in (seq[0], seq[-1]):
        pair_type = "AT" if base in "AT" else "GC"
        total_H += init_H[pair_type]
        total_S += init_S[pair_type]
        total_G += init_G[pair_type]

    # Symmetry correction if self-complementary
    if seq == rc:
        total_H += sym_H
        total_S += sym_S
        total_G += sym_G

    return {
        "Delta_H (kcal/mol)": total_H,
        "Delta_S (cal/K/mol)": total_S,
        "Delta_G_37 (kcal/mol)": total_G
    }


In [ ]:
def get_Num_NTs(df):
    get_Num_NTs = list()
    for i in range(len(df)):
        str1 = df['Samples'][i]
        if('A' in str1):
            get_Num_NTs.append('A')
        elif('B' in str1):
            get_Num_NTs.append('B')
        elif('6' in str1):
            get_Num_NTs.append(6)
        elif('7' in str1):
            get_Num_NTs.append(7)
        elif('9' in str1):
            get_Num_NTs.append(9)
        elif('5' in str1):
            get_Num_NTs.append(5)
    return get_Num_NTs

In [ ]:
def saltCorrection(dG_1M,bp_Len):
    Mg_M =0.0114
    Na_M = 0.0
    ionic_term = Na_M + 3.3 * Mg_M

    if ionic_term <= 0:
        raise ValueError("Ionic strength must be > 0")

    correction = 0.114 * bp_Len * math.log(math.sqrt(ionic_term))

    return dG_1M - correction

In [ ]:
def add_Pairs(df):
    dH = list()
    dG = list()
    dS = list()
    helices = list()
    for i in range(0, len(df), 2):
        sumdH = df['dH_kcal_mol'][i]+df['dH_kcal_mol'][i+1]
        sumdG = df['dG37_kcal_mol'][i]+df['dG37_kcal_mol'][i+1]
        sumdS = df['dS_cal_K_mol'][i]+df['dS_cal_K_mol'][i+1]
        helix = int(df['Helix'][i])-1
        dH.append(sumdH)
        dG.append(sumdG)
        dS.append(sumdS)
        helices.append(helix)
    parameters=pd.DataFrame({'Helix':helices,'G':dG,'S':dS,'H':dH})
    return parameters

In [ ]:
def add_nn_thermo_columns(df: pd.DataFrame) -> pd.DataFrame:
    # calculate thermo for every row
    thermo = df["Sequence"].astype(str).apply(calc_unified_nn)  # returns dict per row

    # expand dicts into columns
    thermo_df = pd.json_normalize(thermo)

    # join back
    out = df.join(thermo_df)

    # optional: shorter column names
    out = out.rename(columns={
        "Delta_H (kcal/mol)": "dH_kcal_mol",
        "Delta_S (cal/K/mol)": "dS_cal_K_mol",
        "Delta_G_37 (kcal/mol)": "dG37_kcal_mol",
    })
    return out

#df = add_nn_thermo_columns(df)

In [ ]:
def Get_tm(df, Ct=4.5e-9):
    """
    Predicted Tm for a non-self-complementary bimolecular duplex
        A + B <-> AB :   Tm = dH / (dS + R*ln(C_T/4))
    with C_T = total strand concentration.  Predicted_deltaH is in kcal/mol
    (converted here to cal/mol) and Predicted_deltaS in cal/mol/K, so Tm is in °C.
    """
    R = 1.987204  # gas constant, cal/(mol*K)
    Tm_CList = list()
    for x in range(len(df)):
        dH = df['Predicted_deltaH'][x] * 1000.0   # kcal/mol -> cal/mol
        dS = df['Predicted_deltaS'][x]            # cal/mol/K
        Tm_K = dH / (dS + R * np.log(Ct / 4.0))
        Tm_CList.append(Tm_K - 273.15)
    return Tm_CList


In [ ]:
def Keq(temp, theta, C_T=4.5e-9):
    """
    Equilibrium constant for a non-self-complementary BIMOLECULAR duplex
        A + B <-> AB ,  [A]=[B]=C_T/2,  theta = fraction folded:
        K = [AB]/([A][B]) = 2*theta / ((1-theta)**2 * C_T)      (units 1/M)
    C_T = TOTAL single-strand concentration (both strands summed).  The earlier
    version omitted the factor of 2 (bimolecular stoichiometry); that shifted the
    intercept, hence dS, by R*ln(2).  Returns ln(K) and 1/T (K^-1) for interior
    points 0<theta<1.
    """
    temp = np.asarray(temp, float)
    lnKeq = list(); inverseTemp = list()
    for i in range(len(theta)):
        th = theta[i]
        if 0 < th < 1:
            keq = 2.0 * th / ((1.0 - th)**2 * C_T)
            lnKeq.append(math.log(keq))
            inverseTemp.append(1.0 / (temp[i] + 273.15))
    return lnKeq, inverseTemp


In [ ]:
def boltzmann(x, A1, A2, x0, dx):
    #return A2 + (A1 - A2) / (1 + np.exp((x - x0)/dx))
    return A1 + (A2 - A1) / (1 + np.exp((x0-x)/dx))

# Standard logistic sigmoid function
def logistic(x, a, b, c, d):
    return a / (1 + np.exp(-c * (x - d))) + b

In [ ]:
homeDir = "C:\\Users\\jared\\Downloads\\Tile_TileResults\\"

In [ ]:
######## From measured entropy and enthalpy, convert to Delta G ########
def deltaGCalc(entropy, enthalpy, tempCelcius):
    temp=273.15+tempCelcius
    deltaG=enthalpy-temp*entropy/1000
    return deltaG

In [ ]:
######## Get Predicted Energy ########
def getEnergy_Seq(df):
    EnergyList = list()
    for a in range(len(df)):
        sample1=df['Samples'][a]
        sample1=sample1.partition('.')[0]
        positions = list(sample1)
        energy = 0
        for i in range(len(positions)):
            arr = positions[i]
            bondLen = arr
            #bondLen = bond_lens[i]
            if(bondLen == '5' or bondLen == 'X'):
                #for x in range(len(arr)):
                energy =energy + nt5_2['G_corrected'][int(i)]
            elif(bondLen == '7'):
                #for x in range(len(arr)):
                energy =energy + nt7_2['G_corrected'][int(i)]
            elif(bondLen == '9'):
                #for x in range(len(arr)):
                energy =energy + nt9_2['G_corrected'][int(i)]
            elif(bondLen == '6'):
                #for x in range(len(arr)):
                energy =energy + nt6_2['G_corrected'][int(i)]
            elif(bondLen == '0'):
                energy = energy + 0
            elif(bondLen == 'A'):
                energy = energy + constant_2['G_corrected'][0]
            elif(bondLen == 'B'):
                energy = energy + constant_2['G_corrected'][1]
            else:
                print('Error')
        EnergyList.append(energy)

    return EnergyList
#df_5nt['Predicted_DeltaG25'] = EnergyList_5nt

In [ ]:
######## Get Predicted Energy ########
def getH_Seq(df):
    EnergyList = list()
    for a in range(len(df)):
        sample1=df['Samples'][a]
        sample1=sample1.partition('.')[0]
        positions = list(sample1)
        energy = 0
        for i in range(len(positions)):
            arr = positions[i]
            bondLen = arr
            #bondLen = bond_lens[i]
            if(bondLen == '5' or bondLen == 'X'):
                #for x in range(len(arr)):
                energy =energy + nt5_2['H'][int(i)]
            elif(bondLen == '7'):
                #for x in range(len(arr)):
                energy =energy + nt7_2['H'][int(i)]
            elif(bondLen == '9'):
                #for x in range(len(arr)):
                energy =energy + nt9_2['H'][int(i)]
            elif(bondLen == '6'):
                #for x in range(len(arr)):
                energy =energy + nt6_2['H'][int(i)]
            elif(bondLen == '0'):
                energy = energy + 0
            elif(bondLen == 'A'):
                energy = energy + constant_2['H'][0]
            elif(bondLen == 'B'):
                energy = energy + constant_2['H'][1]
            else:
                print('Error')
        EnergyList.append(energy)

    return EnergyList

In [ ]:
######## Get Predicted Energy ########
def getS_Seq(df):
    EnergyList = list()
    for a in range(len(df)):
        sample1=df['Samples'][a]
        sample1=sample1.partition('.')[0]
        positions = list(sample1)
        energy = 0
        for i in range(len(positions)):
            arr = positions[i]
            bondLen = arr
            #bondLen = bond_lens[i]
            if(bondLen == '5' or bondLen == 'X'):
                #for x in range(len(arr)):
                energy =energy + nt5_2['S'][int(i)]
            elif(bondLen == '7'):
                #for x in range(len(arr)):
                energy =energy + nt7_2['S'][int(i)]
            elif(bondLen == '9'):
                #for x in range(len(arr)):
                energy =energy + nt9_2['S'][int(i)]
            elif(bondLen == '0'):
                energy = energy + 0
            elif(bondLen == '6'):
                #for x in range(len(arr)):
                energy =energy + nt6_2['S'][int(i)]
            elif(bondLen == 'A'):
                energy = energy + constant_2['S'][0]
            elif(bondLen == 'B'):
                energy = energy + constant_2['S'][1]
            else:
                print('Error')
        EnergyList.append(energy)

    return EnergyList

In [ ]:
def getNum_Bonds_Positions(edgeCodeList):
    df=pd.DataFrame({'Samples':edgeCodeList})
    df['Samples'] = df['Samples'].str.split('.', n=1).str[0]
    positions=df['Samples'].apply(lambda x: [i for i, char in enumerate(x) if char != '0'])
    numBonds = list()
    for i in range(len(positions)):
        #positions[i].append(6)
        numBonds.append(len(positions[i]))

    return numBonds, positions

In [ ]:
def getBonds_Positions_Microstates(edgeCodeList):
    df=pd.DataFrame({'Samples':edgeCodeList})
    df['Samples'] = df['Samples'].str.split('.', n=1).str[0]
    positions=df['Samples'].apply(lambda x: [i for i, char in enumerate(x) if char != '0'])
    numBonds = list()
    for i in range(len(positions)):
        numBonds.append(len(positions[i]))
        
    return numBonds, positions

In [ ]:
def getDistances(positions):
    distanceList = list()
    for i in range(len(positions)):
        arr = np.array(positions[i])
        distances = pdist(arr.reshape(-1, 1))
        avgDist =np.mean(distances)
        distanceList.append(avgDist)
    return distanceList

In [ ]:
#A
#print(calc_unified_nn('GACGGCC'))
#print(calc_unified_nn('CGGCAGC'))

#B
#GCGGCTG
#GCGCGGA

Sequence = ['GACGGCC','CGGCAGC','GCGGCTG','GCGCGGA']
constant=pd.DataFrame({'Helix':['0','0','1','1'],'Sequence':Sequence})

In [ ]:
constant_1 = add_nn_thermo_columns(constant)
constant_2 = add_Pairs(constant_1)
constant_2["G_corrected"] = constant_2.apply(lambda row: saltCorrection(
        dG_1M=row['G'],
        bp_Len=14),axis = 1)
constant_2['Helix'] = ['A','B']

In [ ]:
constant_2

In [ ]:
nt7 = pd.read_csv("nt7.csv", index_col=0)

In [ ]:
nt6 = pd.read_csv("nt6.csv", index_col=0)

In [ ]:
nt5 = pd.read_csv("nt5.csv", index_col=0)

In [ ]:
nt9 = pd.read_csv("nt9.csv", index_col=0)

In [ ]:
nt9_stack = pd.read_csv("nt9_stack.csv", index_col=0)

In [ ]:
nt9_stack

In [ ]:
nt9

In [ ]:
nt9_1 = add_nn_thermo_columns(nt9)
nt9_2 = add_Pairs(nt9_1)
nt9_2['G'] = nt9_2['G']+nt9_stack['Stacking']
nt9_2["G_corrected"] = nt9_2.apply(lambda row: saltCorrection(
        dG_1M=row['G'],
        bp_Len=18),axis = 1)

nt7_1 = add_nn_thermo_columns(nt7)
nt7_2 = add_Pairs(nt7_1)
nt7_2["G_corrected"] = nt7_2.apply(lambda row: saltCorrection(
        dG_1M=row['G'],
        bp_Len=14),axis = 1)

nt5_1 = add_nn_thermo_columns(nt5)
nt5_2 = add_Pairs(nt5_1)
nt5_2["G_corrected"] = nt5_2.apply(lambda row: saltCorrection(
        dG_1M=row['G'],
        bp_Len=10),axis = 1)

nt6_1 = add_nn_thermo_columns(nt6)
nt6_2 = add_Pairs(nt6_1)
nt6_2["G_corrected"] = nt6_2.apply(lambda row: saltCorrection(
        dG_1M=row['G'],
        bp_Len=12),axis = 1)

In [ ]:
######## Get Predicted Energy ########
def make_Energy_Array(df):
    A = np.zeros((len(df), 11))
    #EnergyList = list()
    for a in range(len(df)):
        sample1=df['Samples'][a]
        sample1=sample1.partition('.')[0]
        positions = list(sample1)
        energy = 0
        for i in range(len(positions)):
            arr = positions[i]
            bondLen = arr
            #bondLen = bond_lens[i]
            if(bondLen == '5' or bondLen == 'X'):
                #for x in range(len(arr)):
                A[a,i] += nt5_2['G_corrected'][int(i)]

            elif(bondLen == '7'):
                #for x in range(len(arr)):
                A[a,i] += nt7_2['G_corrected'][int(i)]
            elif(bondLen == '9'):
                #for x in range(len(arr)):
                A[a,i] += nt9_2['G_corrected'][int(i)]
            elif(bondLen == '6'):
                #for x in range(len(arr)):
                A[a,i] += nt6_2['G_corrected'][int(i)]
            elif(bondLen == '0'):
                A[a,i] += 0
            elif(bondLen == 'A'):
                A[a,i] += constant_2['G_corrected'][0]
            elif(bondLen == 'B'):
                A[a,i] += constant_2['G_corrected'][1]
            else:
                print('Error')
    return A
#df_5nt['Predicted_DeltaG25'] = EnergyList_5nt

In [ ]:
data_key = {
    "Thermo10_28_2025_Formatted.csv":"Tile_TileResultsKey - 10_28_2025.csv",
    "DNAExtensionThermo10_29_Formatted.csv":"Tile_TileResultsKey - 10_29_2025.csv",
    "DNAExtensionThermo10_30_Formatted.csv":"Tile_TileResultsKey - 10_30_2025.csv",
    "DNAExtensionThermo11_3_Formatted.csv":"Tile_TileResultsKey - 11_3_2025.csv",
    "DNAExtensionThermo11_6_Formatted.csv":"Tile_TileResultsKey - 11_6_2025.csv",
    "DNAExtensionThermo11_14.csv":"Tile_TileResultsKey - 11_14_2025.csv",
    "DNAExtensionThermo11_15.csv":"Tile_TileResultsKey - 11_15_2025.csv",
    "DNAExtensionThermo11_16.txt":"Tile_TileResultsKey - 11_16_2025.csv",
    "DNAExtensionThermo11_17.txt":"Tile_TileResultsKey - 11_17_2025.csv"
}

In [ ]:
def fitNormalized(data, YVar):
    """Same NEW fitted-baseline van't Hoff fit as fit_Not_Normalized."""
    R = 1.987204; C_T = 4.5e-9
    T  = np.asarray(data['T° 569'], float)
    s0 = np.asarray(data.iloc[:, YVar], float)
    ok = np.isfinite(T) & np.isfinite(s0); T, s0 = T[ok], s0[ok]
    if len(T) < 12:
        raise ValueError("too few points")
    # robust two-state fit THROUGH the read noise -> baselines, midpoint, width
    p0 = [np.percentile(s0, 5), np.percentile(s0, 95), np.median(T), 3.0]
    (A1, A2, x0, dx), _ = curve_fit(boltzmann, T, s0, p0=p0, maxfev=20000)
    if not np.isfinite(dx) or dx <= 0 or not (T.min() - 5 < x0 < T.max() + 5):
        raise ValueError("bad sigmoid fit")
    Tm = x0
    # fraction folded from the FITTED baselines (+ light rolling median for read noise)
    s = pd.Series(s0).rolling(7, center=True, min_periods=1).median().values
    theta = (A2 - s) / (A2 - A1) if A2 > A1 else (s - A2) / (A1 - A2)
    theta = np.clip(theta, 1e-4, 1 - 1e-4)
    Tk = T + 273.15
    # The sigmoid midpoint (Tm) above is already valid. If the van't Hoff step
    # fails -- too few points in the linear region, or an unphysical positive
    # dH/dS -- KEEP the Tm and return dH=dS=NaN rather than discarding the curve.
    # Only a failed *sigmoid* fit (handled above) removes a Tm.
    region = (theta > 0.15) & (theta < 0.85)               # linear region of van't Hoff plot
    if region.sum() < 8:
        return [Tm, dx], None, np.nan, np.nan, (0.0, 1.0, Tm, dx)
    x = 1.0 / Tk[region]
    y = np.log(2.0 * theta[region] / ((1.0 - theta[region])**2 * C_T))   # bimolecular K
    o = np.argsort(x); x, y = x[o], y[o]
    m, b = np.polyfit(x, y, 1)
    enthalpy = -m * R / 1000.0        # kcal/mol
    entropy  =  b * R                 # cal/(mol*K)
    if not np.isfinite(enthalpy) or enthalpy >= 0 or entropy >= 0:
        return [Tm, dx], None, np.nan, np.nan, (0.0, 1.0, Tm, dx)   # keep Tm, drop dH/dS
    # normalized boltzmann params (A1=0, A2=1) keep the downstream melting-curve plot valid
    return [Tm, dx], None, entropy, enthalpy, (0.0, 1.0, Tm, dx)


In [ ]:
def fit_Not_Normalized(data, YVar):
    """Droop-corrected two-state fit with a Tm-reliability rule.
    1) flat Boltzmann fit -> transition (x0, dx).
    2) droop trim: cut at the post-transition plateau peak + 3C, REFIT -> droop-corrected Tm=x0
       and its covariance SE.
    3) reliability: midpoint bracketed by data, transition sampled on BOTH sides, SE(Tm)<=1.5C.
       confidence: 'high' if both plateaus seen (>=3 pts each) AND SE<=1.0C, else 'standard'.
    4) van't Hoff dH/dS on the trimmed data over theta in [0.15,0.85] (NaN if too few pts / unphysical).
    Returns [Tm,dx], extras(dict), entropy, enthalpy, (0.0,1.0,Tm,dx). Raises only on an unfittable sigmoid."""
    R = 1.987204; C_T = 4.5e-9
    T  = np.asarray(data['T° 569'], float)
    s0 = np.asarray(data.iloc[:, YVar], float)
    ok = np.isfinite(T) & np.isfinite(s0); T, s0 = T[ok], s0[ok]
    if len(T) < 14:
        raise ValueError("too few points")
    o = np.argsort(T); T, s0 = T[o], s0[o]
    p0 = [np.percentile(s0, 5), np.percentile(s0, 95), np.median(T), 3.0]
    (A1, A2, x0, dx), _ = curve_fit(boltzmann, T, s0, p0=p0, maxfev=20000)
    if not np.isfinite(dx) or dx <= 0 or A2 <= A1 or not (T.min() - 5 < x0 < T.max() + 5):
        raise ValueError("bad sigmoid fit")
    # ---- droop trim: plateau peak (onset) + 3C, then REFIT with covariance ----
    sm = pd.Series(s0).rolling(7, center=True, min_periods=1).median().values
    post = T > (x0 + 1.5 * dx); onset = T.max()
    if post.sum() >= 3:
        idx = np.where(post)[0]; onset = T[idx[np.argmax(sm[idx])]]
    cut = min(onset + 3.0, T.max()); keep = T <= cut
    if keep.sum() < 12: keep = np.ones_like(T, bool)
    Tf, sf = T[keep], s0[keep]
    try:
        (A1, A2, x0, dx), cov = curve_fit(boltzmann, Tf, sf, p0=p0, maxfev=20000)
        if not np.isfinite(dx) or dx <= 0 or A2 <= A1 or not (Tf.min() - 5 < x0 < Tf.max() + 5):
            raise RuntimeError
        se = float(np.sqrt(np.diag(cov))[2]) if np.all(np.isfinite(cov)) else np.inf
    except Exception:
        Tf, sf = T, s0
        (A1, A2, x0, dx), cov = curve_fit(boltzmann, Tf, sf, p0=p0, maxfev=20000)
        se = float(np.sqrt(np.diag(cov))[2]) if np.all(np.isfinite(cov)) else np.inf
    Tm = x0
    # ---- theta from fitted baselines on trimmed data ----
    s = pd.Series(sf).rolling(7, center=True, min_periods=1).median().values
    theta = (A2 - s) / (A2 - A1) if A2 > A1 else (s - A2) / (A1 - A2)
    theta = np.clip(theta, 1e-4, 1 - 1e-4)
    up = int(np.sum(theta < 0.15)); lo = int(np.sum(theta > 0.85))
    plateau_ok = (up >= 3) and (lo >= 3)
    bracketed  = bool(Tf.min() < x0 < Tf.max())
    bothsides  = (np.sum((theta < 0.5) & (theta > 0.15)) >= 2) and (np.sum((theta > 0.5) & (theta < 0.85)) >= 2)
    reliable   = bool(bracketed and bothsides)   # option 3: per-rep keep = bracketed + both-sides; SE is applied per-CONSTRUCT (median) in df_Tm
    confidence = 'high' if (plateau_ok and np.isfinite(se) and se <= 1.0) else 'standard'
    extras = {'se': se, 'plateau_ok': plateau_ok, 'bracketed': bracketed,
              'bothsides': bool(bothsides), 'reliable': reliable, 'confidence': confidence}
    # ---- van't Hoff dH/dS on trimmed data ----
    Tk = Tf + 273.15
    region = (theta > 0.15) & (theta < 0.85)
    entropy = enthalpy = np.nan
    if region.sum() >= 8:
        x = 1.0 / Tk[region]
        y = np.log(2.0 * theta[region] / ((1.0 - theta[region])**2 * C_T))
        oo = np.argsort(x); x, y = x[oo], y[oo]
        m, b = np.polyfit(x, y, 1)
        enth = -m * R / 1000.0; ent = b * R
        if np.isfinite(enth) and enth < 0 and ent < 0:
            entropy, enthalpy = ent, enth
    return [Tm, dx], extras, entropy, enthalpy, (0.0, 1.0, Tm, dx)

In [ ]:
def load_melting2():
    homeDir = "Melting_Results/"
    files=os.listdir(homeDir)
    for file in files:
        if('.txt' in file):
           meltData =pd.read_csv(homeDir + file, sep = '\t', encoding='unicode_escape')
        elif('.csv' in file):
            meltData = pd.read_csv(homeDir + file, encoding='unicode_escape')
        else:
            continue
        samples = meltData.columns[2:,]
        meltData.columns.values[1] = 'T° 569'
        meltData = meltData.drop(meltData.columns[0], axis=1)
    return meltData

In [ ]:
def loadAll_Data2(homeDir):
    files = os.listdir(homeDir); a = 0; table1 = None
    for file in files:
        if '.txt' in file:
            meltData = pd.read_csv(homeDir + file, sep='\t', encoding='unicode_escape')
        elif '.csv' in file:
            meltData = pd.read_csv(homeDir + file, encoding='unicode_escape')
        else:
            continue
        meltData.columns.values[1] = 'T° 569'
        meltData = meltData.drop(meltData.columns[0], axis=1)
        data615 = meltData
        melts=[]; samples=[]; delta=[]; dx=[]; entropyList=[]; enthalpyList=[]
        seL=[]; platL=[]; relL=[]; confL=[]
        for val in range(len(data615.columns)):
            name = data615.columns[val]
            if name in ('T° 569', 'Buffer', 'F Only'):
                continue
            try:
                res1, extras, entropy, enthalpy, boltzmannParam = fit_Not_Normalized(data615, val)
                melts.append(res1[0]); delta.append(res1[1]); dx.append(boltzmannParam[3])
                entropyList.append(entropy); enthalpyList.append(enthalpy); samples.append(name)
                seL.append(extras['se']); platL.append(extras['plateau_ok'])
                relL.append(extras['reliable']); confL.append(extras['confidence'])
            except Exception:
                melts.append('None'); delta.append('None'); dx.append('None')
                entropyList.append('None'); enthalpyList.append('None'); samples.append(name)
                seL.append('None'); platL.append(False); relL.append(False); confL.append('None')
        experiment = np.full(len(melts), file)
        t = pd.DataFrame({'Tm':melts,'Samples':samples,'Steepness':delta,'Entropy':entropyList,
                          'Enthalpy':enthalpyList,'Experiment':experiment,'A1':0.0,'A2':1.0,
                          'x0':melts,'dx':dx,'Tm_SE':seL,'Plateau_OK':platL,
                          'Reliable':relL,'Confidence':confL})
        table1 = t if a == 0 else pd.concat([table1, t], ignore_index=True)
        a += 1
    return table1

In [ ]:
def filterData(df):
    df["Code"] = df["Samples"].astype(str).str.partition(".")[0]

    g = df.groupby("Code")["Tm"]
    counts = g.size()
    
    tol = 0.05
    spread = (g.max() - g.min()) / g.median()
    ok_base = (counts.eq(3)) & (spread.le(tol))
    
    df_ok = df[df["Code"].isin(ok_base[ok_base].index)]
    return df_ok

In [ ]:
def filterData_std(df, std_threshold=2.5):
    """Replicate-reproducibility QC on Tm: keep interfaces (Code) that have
    exactly 3 replicates whose Tm standard deviation is <= std_threshold (°C)."""
    df = df.copy()
    df["Code"] = df["Samples"].astype(str).str.partition(".")[0]
    g = df.dropna(subset=["Tm"]).groupby("Code")["Tm"]
    summary = g.agg(["mean", "std", "count"])
    passing = summary[
        (summary["count"] == 3) &
        (summary["std"] <= std_threshold)
    ]
    df_good = df[df["Code"].isin(passing.index)]
    return df_good

In [ ]:
def filterData_std_H(df):
    df["Code"] = df["code"].astype(str).str.partition(".")[0]

    g = df.groupby("Code")["dH"]
    
    # Compute summary stats
    summary = g.agg(["mean", "std", "count"])
    
    # Define your std threshold
    std_threshold = 10.0  # adjust as needed
    
    # Apply both criteria
    passing = summary[
        (summary["count"] == 3) & 
        (summary["std"] <= std_threshold)
    ]
    
    # Get filtered dataframe
    df_good = df[df["Code"].isin(passing.index)]
    return df_good

In [ ]:
def format_Melting(meltdf, key_df):
    homeDir = "C:\\Users\\jared\\Downloads\\Tile_TileResults\\"
    if(meltdf == "DNAExtensionThermo11_6_Formatted.csv" or meltdf == 'Thermo10_28_2025_Formatted.csv'):
        pandasDF_melt = pd.read_csv(homeDir+ meltdf)
        pandas_key =  pd.read_csv(homeDir+ key_df)
        samples  = pandas_key['Unnamed: 1']
    
    elif(meltdf == "DNAExtensionThermo11_15.csv"):
        pandasDF_melt = pd.read_csv(homeDir+ meltdf, encoding='unicode_escape')
        #samples  = pandas_key['Unnamed: 1']
        samples  = pandasDF_melt.columns[2:]

    elif(meltdf == "DNAExtensionThermo11_14.csv"):
        pandasDF_melt = pd.read_csv(homeDir+ meltdf)
        #samples  = pandas_key['Unnamed: 1']
        samples  = pandasDF_melt.columns[2:]
    
    elif(meltdf == "DNAExtensionThermo11_16.txt" or meltdf == "DNAExtensionThermo11_17.txt"):
        pandasDF_melt = pd.read_csv(homeDir+ meltdf, encoding='unicode_escape', sep ='\t')
        #samples  = pandas_key['Unnamed: 1']
        samples  = pandasDF_melt.columns[2:]
    
    else:
        pandasDF_melt = pd.read_csv(homeDir+ meltdf, encoding='unicode_escape')
        pandas_key =  pd.read_csv(homeDir+ key_df)
        samples  = pandas_key['Unnamed: 1']
    
    pandasDF_Samples = pandasDF_melt.iloc[:,2:]     
    temp = pandasDF_melt['T° 569']
    pandasDF_Samples.columns = samples
    pandasDF_Samples['T° 569'] = temp
    #pandasDF_Samples = pandasDF_Samples.dropna()
    return pandasDF_Samples

In [ ]:
def load_allData(data_key):
    a = 0

    dataReformat = ["DNAExtensionThermo11_14.csv",
    "DNAExtensionThermo11_15.csv",
    "DNAExtensionThermo11_16.txt",
    "DNAExtensionThermo11_17.txt"]
    
    for key in data_key.keys():

        if key in dataReformat:
            meltData=format_Melting(key,data_key[key])
            nt_Count = np.full(len(meltData.columns)-1, 7)
        
        else:
            meltData=format_Melting(key,data_key[key])
            
            if(key == "Thermo10_28_2025_Formatted.csv"):
                nt_Count = np.full(len(meltData.columns)-1, 7)
            
            elif(key == "DNAExtensionThermo10_29_Formatted.csv"):
                meltData.drop('00000X01011', axis=1, inplace = True)
                meltData.drop('00001X00101', axis=1, inplace = True)
                meltData.drop('00001X00110', axis=1, inplace = True)
                meltData.drop('00001X11000', axis=1, inplace = True)
                meltData.drop('00010X00011', axis=1, inplace = True)
    
                nt_Count = np.full(len(meltData.columns)-1, 7)
    
            elif(key == "DNAExtensionThermo10_30_Formatted.csv"):
                nt_Count = np.full(len(meltData.columns)-1, 9)
    
            elif(key == "DNAExtensionThermo11_3_Formatted.csv"):
                nt_Count = np.full(len(meltData.columns)-1, 5)
            
            elif(key == "DNAExtensionThermo11_6_Formatted.csv"):
                legend=data_key[key]
                pandas_key=pd.read_csv(homeDir+legend)
                nt_Count=pandas_key['Unnamed: 3'].apply(lambda x: int(x[0]) if x else None)


        data615 = meltData
        #print(data615)
        melts  = list()
        samples=list()
        delta=list()
        entropyList = list()
        enthalpyList = list()
        for val in range(len(data615.columns)):
            name = data615.columns[val]
            if(name != 'T° 569' and name != 'Buffer' and name != 'F Only'):
                try:
                    res1,day, entropy, enthalpy=fitNormalized(data615,val)
                    entropyList.append(entropy/4184)
                    enthalpyList.append(enthalpy/4184)
                    melts.append(res1[0])
                    delta.append(res1[1])
                    #melts.append(fitAndPlot(data615,name)[0])
                    samples.append(name)
                except:
                    melts.append('None')
                    delta.append('None')
                    entropyList.append('None')
                    enthalpyList.append('None')
                    samples.append(name)
        experiment = np.full(len(melts), key)
        if(a==0):
            table1=pd.DataFrame({'Tm':melts, 'Samples':samples, 'Steepness':delta,'Entropy':entropyList,
                                 'Enthalpy':enthalpyList,'NucleotideBonds':nt_Count,'Experiment':experiment})
        if(a != 0):
            table2=pd.DataFrame({'Tm':melts, 'Samples':samples, 'Steepness':delta,'Entropy':entropyList,
                     'Enthalpy':enthalpyList,'NucleotideBonds':nt_Count,'Experiment':experiment})
            table1 = pd.concat([table1,table2], ignore_index=True)
        a = a+1
    return table1

In [ ]:
fil=os.listdir("Melting_Results/")
meltData =pd.read_csv("Melting_Results/" + fil[0], sep = ',', encoding='unicode_escape')

In [ ]:
res_Table = loadAll_Data2("Melting_Results/")
#res_Table = res_Table[res_Table['Tm'] != 'None']

In [ ]:
dfFilter = pd.read_csv('Filtered_DF_std.csv')
dfFilter.drop(['Unnamed: 0'], axis =1, inplace = True)

In [ ]:
res_Table = res_Table[res_Table['Tm'] != 'None'].copy()
# loadAll_Data2 mixes float fits with 'None' -> object dtype; make numeric columns numeric
for _c in ['Tm', 'Steepness', 'Entropy', 'Enthalpy', 'A1', 'A2', 'x0', 'dx', 'Yield', 'Tm_SE']:
    if _c in res_Table.columns:
        res_Table[_c] = pd.to_numeric(res_Table[_c], errors='coerce')


In [ ]:
# === AUTHORITATIVE Tm TABLE ============================================
# Droop-corrected Tm (onset+3C refit), reliability rule applied per curve in the
# fit (midpoint bracketed, transition both sides, covariance SE(Tm)<=1.5C).
# Reproducible: >=3 reliable reps with Tm std <= 2.5C. Confidence tier per
# construct: 'high' if ALL reps are plateau-complete AND SE<=1.0C, else 'standard'.
# Carries Tm_SE (median), plateau_confident, n_reps. Independent of dH/dS.
_r = res_Table.copy()
_r['Tm']    = pd.to_numeric(_r['Tm'], errors='coerce')
_r['Tm_SE'] = pd.to_numeric(_r['Tm_SE'], errors='coerce')
_r['Reliable']   = _r['Reliable'].apply(lambda v: v is True or str(v) == 'True')
_r['Plateau_OK'] = _r['Plateau_OK'].apply(lambda v: v is True or str(v) == 'True')
_r = _r[_r['Reliable'] & _r['Tm'].notna()].reset_index(drop=True)
_r = _r[~_r['Samples'].astype(str).str.contains('9')].reset_index(drop=True)  # ignore 9nt bonds (stacking handled separately)
# geometry / sequence annotations
_r['NucleotideBonds']               = get_Num_NTs(_r)
_r['Num_Bonds'], _r['Positions']    = getNum_Bonds_Positions(_r['Samples'])
_r['Distances']                     = getDistances(_r['Positions'])
_r['Predicted_deltaS']              = getS_Seq(_r)
_r['Predicted_deltaH']              = getH_Seq(_r)
_r['Predicted_deltaG']              = getEnergy_Seq(_r)
_r['Predicted_tm']                  = Get_tm(_r)
_r['Code']                          = _r['Samples'].astype(str).str.partition('.')[0]
def _mg(pos):
    a = np.sort(np.asarray(pos, float)); g = np.diff(a)
    return g.mean() if len(g) > 0 else np.nan
_r['MeanGap']    = _r['Positions'].map(_mg)
_r['MeanGap_nm'] = _r['MeanGap'] * (88.0/10)
_r['Distances_nm'] = _r['Distances'] * (88.0/10)
# reproducibility: >=3 reliable reps, Tm std <= 2.5C
# option 3: construct kept if >=3 reliable reps, Tm std <= 2.5C, and MEDIAN rep SE <= 1.5C
_ag = _r.groupby('Code').agg(_std=('Tm','std'), _cnt=('Tm','count'), _medse=('Tm_SE','median'))
_good = _ag[(_ag['_cnt'] >= 3) & (_ag['_std'] <= 2.5) & (_ag['_medse'] <= 1.5)].index
df_Tm = _r[_r['Code'].isin(_good)].reset_index(drop=True)
# confidence tier + n_reps
_conf = df_Tm.groupby('Code')['Plateau_OK'].all()
df_Tm['plateau_confident'] = df_Tm['Code'].map(lambda c: 'high' if bool(_conf.get(c, False)) else 'standard')
df_Tm['n_reps'] = df_Tm['Code'].map(df_Tm['Code'].value_counts())
# per-construct median table (for Tm figures)
df_Tm_median = (df_Tm.groupby('Code', as_index=False)
    .agg({'Experiment':'first','NucleotideBonds':'first','Num_Bonds':'first','Positions':'first',
          'Distances':'first','Predicted_deltaG':'first','Predicted_deltaS':'first',
          'Predicted_deltaH':'first','Predicted_tm':'first','MeanGap':'first','MeanGap_nm':'first',
          'Distances_nm':'first','plateau_confident':'first','n_reps':'first',
          'Tm':'median','Tm_SE':'median'}))
df_Tm_median['Samples'] = df_Tm_median['Code']   # alias for make_Energy_Array (ML)
_nhi = (_conf.reindex(_good).fillna(False)).sum()
print(f"Tm TABLE: {df_Tm['Code'].nunique()} interfaces  (high={int(_nhi)}, standard={df_Tm['Code'].nunique()-int(_nhi)}), {len(df_Tm)} curves")
# high-reliability subsets (plateau-complete constructs)
df_Tm_high        = df_Tm[df_Tm['plateau_confident'] == 'high'].reset_index(drop=True)
df_Tm_median_high = df_Tm_median[df_Tm_median['plateau_confident'] == 'high'].reset_index(drop=True)
print(f"  high-reliability subset: {df_Tm_high['Code'].nunique()} interfaces, {len(df_Tm_high)} curves")

In [ ]:
# H/S DATASET input: keep only curves with a valid (physical) dH and dS fit.
# Everything downstream (dH/dS/dG trends) runs on this reproducible set.
df1 = res_Table[res_Table["Entropy"].notna() & res_Table["Enthalpy"].notna()].copy()
#df1 = dfFilter

In [ ]:
dfFilter=filterData_std(df1)

In [ ]:
df1 = dfFilter.reset_index(drop=True)


In [ ]:
df1['Predicted_deltaG'] = getEnergy_Seq(df1)
df1['Predicted_deltaS'] = getS_Seq(df1)
df1['Predicted_deltaH'] = getH_Seq(df1)
df1['Predicted_tm'] = Get_tm(df1)

#getNum_Bonds_Positions(df1['Samples'])
df1['NucleotideBonds']=get_Num_NTs(df1)
df1['Num_Bonds'],df1['Positions'] =getNum_Bonds_Positions(df1['Samples'])
#df1['Predicted_deltaG'] = getEnergy_Seq(df1['Positions'], df1['Num_Bonds'])
df1['Exp_DeltaG'] = deltaGCalc(df1['Entropy'],df1['Enthalpy'], 37)
#df1['factor'] = df1['Exp_DeltaG']/df1['Predicted_deltaG']
df1['factor'] = df1['Predicted_deltaG']-df1['Exp_DeltaG']
df1['Distances']=getDistances(df1['Positions'])
dfFilter=filterData_std(df1)

In [ ]:
df1 = dfFilter.reset_index(drop = True)

In [ ]:
df1["Samples"] = df1["Samples"].str.partition(".")[0]
df1 = df1[~df1["Samples"].str.contains("9")].reset_index(drop=True)  # ignore 9nt bonds

df_median = (
    df1
    .groupby("Samples", as_index=False)
    .agg({
        "Experiment": "first",
        "Predicted_deltaG": "first",
        "NucleotideBonds": "first",
        "Predicted_deltaG": "first",
        "Num_Bonds": "first",
        "Positions": "first",
        "Predicted_deltaS": "first",
        "Predicted_deltaH": "first",
        "Predicted_tm": "first",
        "Tm": "median",
        "Steepness": "median",
        "Entropy": "median",
        "Enthalpy": "median",
        'Exp_DeltaG': "median",
        'factor': "median",
        "A1":"median",
        'A2':"median",
        'dx':"median",
        'x0':'median',
        #'Yield':'median'
    })
)

df1_A = df1[
    df1["Samples"].str.contains("A", na=False) &
    ~df1["Samples"].str.contains("B", na=False)
]

df1_B = df1[
    df1["Samples"].str.contains("B", na=False) &
    ~df1["Samples"].str.contains("A", na=False)
]
df1A_B = df1[
    df1["Samples"].str.contains("B", na=False) &
    df1["Samples"].str.contains("A", na=False)
]

deltaG_Arr=make_Energy_Array(df_median)

In [ ]:
plt.figure(figsize=(6,6))

#df_sorted = df1.sort_values("Tm")
#df_sorted=df_sorted.iloc[np.linspace(35,1030,100)]
#df_top100 = df_sorted.tail(100)
#df_top100 = df_sorted.tail(100).iloc[::-1]

x_fit = np.linspace(
    26,
    55,
    500
)
x=0
for _, row in df_median.iterrows():
    
    A1 = row["A1"]
    A2 = row["A2"]
    Tm = row["Tm"]
    dx = row["dx"]
    if(dx < 1):
        continue
    if(Tm < 30):
        continue
    if(Tm > 50):
        continue
    if(A1 >1):
        continue
    y_curve = boltzmann(x_fit, A1, A2, Tm, dx)
    plt.ylim(0,1)
    plt.xlim(25,55)
    #y_fit=1-y_curve
    fU = (y_curve - A1) / (A2 - A1)
    plt.plot(x_fit, fU, alpha=0.2)
    x=x+1
plt.xlabel("Temperature (°C)")
plt.ylabel("Normalized Signal")
plt.title("Fitted Melting Curves")
plt.ylim(-0.05,1.05)
plt.tight_layout()
#plt.show()
plt.savefig('../Tile_Tole_Figures/Melting Curves.svg')

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, KFold
from scipy.stats import spearmanr, rankdata
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error
import shap


## XGBoost

In [ ]:
feature_names = ['Helix 0', 'Helix 1', 'Helix 2', 'Helix 3', 'Helix 4', 'Helix 5', 'Helix 6', 'Helix 7', 'Helix 8',
                 'Helix 9','Helix 10']
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=0,
    feature_names=feature_names
)

In [ ]:
W=(df_median['Tm']**2)/df_median['Enthalpy']

In [ ]:
dfMini=df_median[df_median['NucleotideBonds'] != 'A']
dfMini=dfMini[dfMini['NucleotideBonds'] != 'B']
dfMini.reset_index(drop=True, inplace=True)
deltaG_Arr_Mini=make_Energy_Array(dfMini)
X_train = deltaG_Arr_Mini
y_train = dfMini['Exp_DeltaG']

df1_A.reset_index(drop=True, inplace=True)

X_test =make_Energy_Array(df1_A) 
y_test = df1_A['Exp_DeltaG']
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(r2_score(y_test, y_pred))
print(spearmanr(y_test, y_pred).correlation)

In [ ]:
# X: (n_samples, n_features), y: (n_samples,)
X = deltaG_Arr

y = df_median['Exp_DeltaG']


kf = KFold(n_splits=10, shuffle=True, random_state=0)

r2_scores = []
spearman_scores = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    r2_scores.append(r2_score(y_test, y_pred))
    spearman_scores.append(spearmanr(y_test, y_pred).correlation)

print("Mean CV R²:", np.mean(r2_scores))
print("Mean CV Spearman ρ:", np.mean(spearman_scores))

In [ ]:

true_rank = np.argsort(np.argsort(y_test))
pred_rank = np.argsort(np.argsort(y_pred))

rho, _ = spearmanr(y_test, y_pred)

plt.figure()
plt.scatter(true_rank, pred_rank)

plt.plot([true_rank.min(), true_rank.max()],
         [true_rank.min(), true_rank.max()])

plt.text(
    0.05, 0.95,
    f"Spearman ρ = {rho:.3f}",
    transform=plt.gca().transAxes,
    verticalalignment='top'
)

plt.xlabel("True Rank")
plt.ylabel("Predicted Rank")
plt.title("Rank vs Rank")
plt.show()

In [ ]:
def oof_predictions(model, X, y, n_splits=10, random_state=0):
    """
    Returns out-of-fold predictions for any sklearn-like model with fit/predict.
    """
    X = np.asarray(X)
    y = np.asarray(y).ravel()

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_oof = np.empty_like(y, dtype=float)

    for train_idx, test_idx in kf.split(X):
        model.fit(X[train_idx], y[train_idx])
        y_oof[test_idx] = model.predict(X[test_idx])

    return y_oof

def plot_rank_vs_rank(y_true, y_pred, title="Rank vs Rank (OOF)"):
    """
    Rank-vs-rank scatter with Spearman rho, all points together.
    """
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    # Use average ranks for ties (standard for Spearman)
    r_true = rankdata(y_true, method="average")
    r_pred = rankdata(y_pred, method="average")

    rho, p = spearmanr(y_true, y_pred)

    plt.figure()
    plt.scatter(r_true, r_pred, alpha = 0.5)

    # Identity line
    lo = min(r_true.min(), r_pred.min())
    hi = max(r_true.max(), r_pred.max())
    plt.plot([lo, hi], [lo, hi])

    #plt.text(
    #    0.05, 1.05,
    #    f"Spearman ρ = {rho:.3f}\n(p = {p:.2g})",
    #    transform=plt.gca().transAxes,
    #    va="center"
    #)
    print(f"Spearman ρ = {rho:.3f}\n(p = {p:.2g})")
    plt.xlabel("True rank")
    plt.ylabel("Predicted rank")
    plt.title(title)
    plt.tight_layout()
    #plt.show()
    plt.savefig('../Tile_Tole_Figures/XGBoost.svg')

# --- usage ---
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=0,
    feature_names=feature_names
)

#dfMini=df_Tm_median   # repointed to authoritative Tm table[df_median['NucleotideBonds'] != 9]
dfMini=df_Tm_median   # repointed to authoritative Tm table
#dfMini=dfMini[dfMini['NucleotideBonds'] != 'B']
dfMini.reset_index(drop=True, inplace=True)
deltaG_Arr_Mini=make_Energy_Array(dfMini)

X = deltaG_Arr_Mini
#y = dfMini['Exp_DeltaG']
y = dfMini['Tm']

df1_A.reset_index(drop=True, inplace=True)


y_oof = oof_predictions(model, X, y, n_splits=10, random_state=0)
plot_rank_vs_rank(y, y_oof)

# --- paper-figure export ---
from scipy.stats import rankdata as _rankdata
export_panel("Fig6b_XGBoost_RankRank",
             data=pd.DataFrame({"true_Tm": np.asarray(y, float).ravel(),
                                "pred_Tm_oof": np.asarray(y_oof, float).ravel(),
                                "true_rank": _rankdata(np.asarray(y, float).ravel()),
                                "pred_rank": _rankdata(np.asarray(y_oof, float).ravel())}),
             note="Fig 6b. XGBoost out-of-fold rank-vs-rank prediction of interface Tm (10-fold CV).")


In [ ]:
plt.figure()
plt.scatter(y_oof, y, alpha = 0.2)
plt.plot([y.min(), y.max()],
         [y.min(), y.max()])  # identity line
plt.xlabel("XGBoost Predicted Tm (C)")
plt.ylabel("Experimental Tm (C)")
#plt.title("Exer vs Predicted")
plt.tight_layout()
plt.savefig('../Tile_Tole_Figures/XGBoost_Scatter.svg')
export_panel("Fig6c_XGBoost_StabilityScatter",
             data=pd.DataFrame({"pred_Tm_oof": np.asarray(y_oof, float).ravel(),
                                "exp_Tm": np.asarray(y, float).ravel()}),
             note="Fig 6c. XGBoost out-of-fold predicted Tm vs experimental Tm (stability prediction).")
plt.show()

residuals = y - y_oof
plt.figure()
plt.scatter(y_oof, residuals)
plt.axhline(0)
plt.xlabel("XGBoost Predicted Tm (C)")
plt.ylabel("Residuals")
#plt.title("Residuals vs Predicted")
plt.show()

#sns.barplot(model.coef_)
#plt.show()
#print("R^2: "+ str(model.score(X, y)))
rho, pval = spearmanr(y, y_oof)
#print("Spearman rho:", rho)

In [ ]:

# Create explainer (TreeExplainer automatically used for XGBoost)
explainer = shap.Explainer(model)

# Compute SHAP values
shap_values = explainer(X)

In [ ]:
fig = plt.figure()

shap.plots.beeswarm(shap_values, show = False)

fig = plt.gcf()
plt.tight_layout()
plt.savefig('../Tile_Tole_Figures/XGBoost_SHAP_Values.svg')
_sv = np.asarray(shap_values.values)
_fn = [f"helix{_i}" for _i in range(_sv.shape[1])]
_shdf = pd.DataFrame(_sv, columns=[f"SHAP_{c}" for c in _fn])
try:
    _dat = np.asarray(shap_values.data)
    for _j, _c in enumerate(_fn): _shdf[f"featval_{_c}"] = _dat[:, _j]
except Exception: pass
export_panel("Fig6e_SHAP_Values", data=_shdf,
             note="Fig 6e. SHAP values from the XGBoost Tm model. SHAP_* = contribution of each helix feature per sample; featval_* = the feature value (NN-predicted dG, 0 if no bond).")
plt.close(fig)

In [ ]:
for i in range(11):
    shap.dependence_plot(i, shap_values.values, X, interaction_index="auto", show=False)
    plt.title("SHAP Dependence: " + str(i))
    plt.tight_layout()
    plt.show()

## Decision Tree

In [ ]:
def oof_predictions(model, X, y, n_splits=10, random_state=0):
    """
    Returns out-of-fold predictions for any sklearn-like model with fit/predict.
    """
    X = np.asarray(X)
    y = np.asarray(y).ravel()

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_oof = np.empty_like(y, dtype=float)

    for train_idx, test_idx in kf.split(X):
        model.fit(X[train_idx], y[train_idx])
        y_oof[test_idx] = model.predict(X[test_idx])

    return y_oof

def plot_rank_vs_rank(y_true, y_pred, title="Rank vs Rank (OOF)"):
    """
    Rank-vs-rank scatter with Spearman rho, all points together.
    """
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    # Use average ranks for ties (standard for Spearman)
    r_true = rankdata(y_true, method="average")
    r_pred = rankdata(y_pred, method="average")

    rho, p = spearmanr(y_true, y_pred)

    plt.figure()
    plt.scatter(r_true, r_pred, alpha = 0.5)

    # Identity line
    lo = min(r_true.min(), r_pred.min())
    hi = max(r_true.max(), r_pred.max())
    plt.plot([lo, hi], [lo, hi])

    #plt.text(
    #    0.05, 1.05,
    #    f"Spearman ρ = {rho:.3f}\n(p = {p:.2g})",
    #    transform=plt.gca().transAxes,
    #    va="center"
    #)
    print(f"Spearman ρ = {rho:.3f}\n(p = {p:.2g})")

    plt.xlabel("True rank")
    plt.ylabel("Predicted rank")
    plt.title(title)
    plt.tight_layout()
    plt.show()

# --- usage ---

tree = DecisionTreeRegressor(
    max_depth=6,              # control complexity
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=0
)


model = tree #...  # e.g., your XGBRegressor() or linear model, etc.
y_oof = oof_predictions(model, X, y, n_splits=10, random_state=0)
plot_rank_vs_rank(y, y_oof)

## Sum of Individual Energies

In [ ]:
def oof_predictions(model, X, y, n_splits=10, random_state=0):
    """
    Returns out-of-fold predictions for any sklearn-like model with fit/predict.
    """
    X = np.asarray(X)
    y = np.asarray(y).ravel()

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_oof = np.empty_like(y, dtype=float)

    for train_idx, test_idx in kf.split(X):
        #model.fit(X[train_idx], y[train_idx])
        #y_oof[test_idx] = model.predict(X[test_idx])
        y_oof[test_idx] = np.sum(X[test_idx], axis=1)

    return y_oof

def plot_rank_vs_rank(y_true, y_pred, title="Rank vs Rank (OOF)"):
    """
    Rank-vs-rank scatter with Spearman rho, all points together.
    """
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    # Use average ranks for ties (standard for Spearman)
    r_true = rankdata(y_true, method="average")
    r_pred = rankdata(y_pred, method="average")

    rho, p = spearmanr(y_true, y_pred)

    plt.figure()
    plt.scatter(r_true, r_pred, alpha = 0.5)

    # Identity line
    lo = min(r_true.min(), r_pred.min())
    hi = max(r_true.max(), r_pred.max())
    plt.plot([lo, hi], [lo, hi])

    #plt.text(
    #    0.05, 1.05,
    #    f"Spearman ρ = {rho:.3f}\n(p = {p:.2g})",
    #    transform=plt.gca().transAxes,
    #    va="center"
    #)
    print(f"Spearman ρ = {rho:.3f}\n(p = {p:.2g})")
    plt.xlabel("True rank")
    plt.ylabel("Predicted rank")
    plt.title(title)
    plt.tight_layout()
    plt.show()

# --- usage ---

tree = DecisionTreeRegressor(
    max_depth=6,              # control complexity
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=0
)


model = tree #...  # e.g., your XGBRegressor() or linear model, etc.
y_oof = oof_predictions(model, X, y, n_splits=10, random_state=0)
plot_rank_vs_rank(y, y_oof)

In [ ]:
def oof_predictions(model, X, y, n_splits=10, random_state=0):
    """
    Returns out-of-fold predictions for any sklearn-like model with fit/predict.
    """
    X = np.asarray(X)
    y = np.asarray(y).ravel()

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_oof = np.empty_like(y, dtype=float)

    for train_idx, test_idx in kf.split(X):
        model.fit(X[train_idx], y[train_idx])
        y_oof[test_idx] = model.predict(X[test_idx])

    return y_oof

def plot_rank_vs_rank(y_true, y_pred, title="Rank vs Rank (OOF)"):
    """
    Rank-vs-rank scatter with Spearman rho, all points together.
    """
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    # Use average ranks for ties (standard for Spearman)
    r_true = rankdata(y_true, method="average")
    r_pred = rankdata(y_pred, method="average")

    rho, p = spearmanr(y_true, y_pred)

    plt.figure()
    plt.scatter(r_true, r_pred, alpha = 0.5)

    # Identity line
    lo = min(r_true.min(), r_pred.min())
    hi = max(r_true.max(), r_pred.max())
    plt.plot([lo, hi], [lo, hi])

    #plt.text(
    #    0.05, 1.05,
    #    f"Spearman ρ = {rho:.3f}\n(p = {p:.2g})",
    #    transform=plt.gca().transAxes,
    #    va="center"
    #)
    print(f"Spearman ρ = {rho:.3f}\n(p = {p:.2g})")

    plt.xlabel("True rank")
    plt.ylabel("Predicted rank")
    plt.title(title)
    plt.tight_layout()
    plt.show()


model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False,interaction_only=True),  # adds x_i, x_i*x_j, and x_i^2
    LinearRegression()
)

y_oof = oof_predictions(model, X, y, n_splits=10, random_state=0)
plot_rank_vs_rank(y, y_oof)

In [ ]:
def oof_predictions(model, X, y, n_splits=10, random_state=0):
    """
    Returns out-of-fold predictions for any sklearn-like model with fit/predict.
    """
    X = np.asarray(X)
    y = np.asarray(y).ravel()

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_oof = np.empty_like(y, dtype=float)

    for train_idx, test_idx in kf.split(X):
        model.fit(X[train_idx], y[train_idx])
        y_oof[test_idx] = model.predict(X[test_idx])

    return y_oof

def plot_rank_vs_rank(y_true, y_pred, title="Rank vs Rank (OOF)"):
    """
    Rank-vs-rank scatter with Spearman rho, all points together.
    """
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    # Use average ranks for ties (standard for Spearman)
    r_true = rankdata(y_true, method="average")
    r_pred = rankdata(y_pred, method="average")

    rho, p = spearmanr(y_true, y_pred)

    plt.figure()
    plt.scatter(r_true, r_pred, alpha = 0.5)

    # Identity line
    lo = min(r_true.min(), r_pred.min())
    hi = max(r_true.max(), r_pred.max())
    plt.plot([lo, hi], [lo, hi])

    #plt.text(
    #    0.05, 1.05,
    #    f"Spearman ρ = {rho:.3f}\n(p = {p:.2g})",
    #    transform=plt.gca().transAxes,
    #    va="center"
    #)
    print(f"Spearman ρ = {rho:.3f}\n(p = {p:.2g})")

    plt.xlabel("True rank")
    plt.ylabel("Predicted rank")
    plt.title(title)
    plt.tight_layout()
    #plt.show()
    plt.savefig('../Tile_Tole_Figures/Linear_Rank.svg')


model = LinearRegression()

y_oof = oof_predictions(model, X, y, n_splits=10, random_state=0)
plot_rank_vs_rank(y, y_oof)

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin

class SumModel(BaseEstimator, RegressorMixin):
    def fit(self, X, y=None):
        return self  # nothing to learn
    
    def predict(self, X):
        return np.sum(X, axis=1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from scipy.stats import spearmanr, rankdata

def cv_spearman_oof(models, X, y, n_splits=10, random_state=0, plot=False):
    """
    For each model:
      - compute out-of-fold predictions
      - compute Spearman rho per fold (on that fold's test set)
      - compute overall Spearman rho using all OOF predictions
    Returns a results dict keyed by model name.
    """
    X = np.asarray(X)
    y = np.asarray(y).ravel()

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    results = {}

    for name, model in models.items():
        y_oof = np.empty_like(y, dtype=float)
        fold_rhos = []

        for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
            model.fit(X[train_idx], y[train_idx])
            y_pred = model.predict(X[test_idx])

            y_oof[test_idx] = y_pred

            rho_fold = spearmanr(y[test_idx], y_pred).correlation
            fold_rhos.append(rho_fold)

        # Overall OOF Spearman across all samples
        rho_oof = spearmanr(y, y_oof).correlation

        results[name] = {
            "fold_rhos": fold_rhos,
            "mean_fold_rho": float(np.nanmean(fold_rhos)),
            "std_fold_rho": float(np.nanstd(fold_rhos)),
            "oof_rho": float(rho_oof),
            "y_oof": y_oof,  # keep if you want plots later
        }

        print(
            f"{name:>20s} | OOF ρ={results[name]['oof_rho']:.3f} "
            f"| mean-fold ρ={results[name]['mean_fold_rho']:.3f} ± {results[name]['std_fold_rho']:.3f}"
        )

        if plot:
            plot_rank_vs_rank(y, y_oof, title=f"Rank vs Rank (OOF): {name}")

    return results


def plot_rank_vs_rank(y_true, y_pred, title="Rank vs Rank (OOF)"):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    r_true = rankdata(y_true, method="average")
    r_pred = rankdata(y_pred, method="average")

    rho, p = spearmanr(y_true, y_pred)

    plt.figure()
    plt.scatter(r_true, r_pred, alpha=0.5)

    lo = min(r_true.min(), r_pred.min())
    hi = max(r_true.max(), r_pred.max())
    plt.plot([lo, hi], [lo, hi])

    plt.xlabel("True rank")
    plt.ylabel("Predicted rank")
    plt.title(title + f" | Spearman ρ={rho:.3f}")
    plt.tight_layout()
    plt.show()


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

In [ ]:
dfMini=df_Tm_median   # repointed to authoritative Tm table
#dfMini=dfMini[dfMini['NucleotideBonds'] != 'B']
dfMini.reset_index(drop=True, inplace=True)
deltaG_Arr_Mini=make_Energy_Array(dfMini)

X = deltaG_Arr_Mini
y = dfMini['Tm']


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    #"Ridge": Ridge(alpha=1.0, random_state=0),
    "Quadratic Regression": make_pipeline(
        PolynomialFeatures(degree=2, include_bias=False,interaction_only=True),  # adds x_i, x_i*x_j, and x_i^2
        LinearRegression()
    ),
    #"Lasso": Lasso(alpha=0.01, random_state=0, max_iter=10000),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=0),
    "XGBoost": XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=0
    ),
    #"Tree" : DecisionTreeRegressor(
    #    max_depth=6,              # control complexity
    #    min_samples_split=10,
     #   min_samples_leaf=5,
     #   random_state=0
    #),
    "Sum of Sequences":SumModel()
}

results = cv_spearman_oof(models, X, y, n_splits=10, random_state=0, plot=False)

# Access list of fold rhos for a model:
linear_fold_rhos = results["Linear Regression"]["fold_rhos"]
print("Linear fold rhos:", linear_fold_rhos)

In [ ]:
def plot_combined_seaborn(results):
    # Convert results dict to DataFrame
    data = []
    for model_name, vals in results.items():
        data.append({
            "Model": model_name,
            "Mean_Fold_Rho": vals["mean_fold_rho"],
            "Std_Fold_Rho": vals["std_fold_rho"],
            "OOF_Rho": vals["oof_rho"]
        })

    df = pd.DataFrame(data)

    # Optional: sort by performance
    df = df.sort_values("Mean_Fold_Rho", ascending=True)

    plt.figure(figsize=(10,6))

    # Barplot (mean fold rho)
    ax = sns.barplot(
        data=df,
        x="Model",color ='#005CAB',
        y="OOF_Rho",
        ci=None   # we'll add our own error bars
    )

    # Add error bars manually
    ax.errorbar(
        x=range(len(df)),
        y=df["Mean_Fold_Rho"],
        yerr=df["Std_Fold_Rho"],
        fmt='none',
        capsize=5,
        color='black'
    )

    # Overlay OOF points
    sns.scatterplot(
        data=df,
        x="Model",
        y="OOF_Rho",
        color="black",
        s=80,
        zorder=5,
        label="OOF ρ"
    )

    plt.ylabel("Spearman ρ")
    plt.title("Model Ranking Performance (10-fold CV)")
    plt.xticks(rotation=45, ha='right')
    plt.legend()
    plt.tight_layout()
    #plt.show()
    plt.savefig('../Tile_Tole_Figures/Model Performances7_28.svg')
plot_combined_seaborn(results)

In [ ]:
def plot_combined_seaborn(results):
    data = []
    for model_name, vals in results.items():
        data.append({
            "Model": model_name,
            "Mean_Fold_Rho": vals["mean_fold_rho"],
            "Std_Fold_Rho": vals["std_fold_rho"],
            "OOF_Rho": vals["oof_rho"]
        })
    df = pd.DataFrame(data)
    df = df.sort_values("Mean_Fold_Rho", ascending=True)

    plt.figure(figsize=(10, 6))

    # Horizontal barplot (OOF rho)
    ax = sns.barplot(
        data=df,
        y="Model",
        x="OOF_Rho",
        color='#005CAB',
        errorbar=None          # ci=None is deprecated in seaborn >=0.12
    )

    # Horizontal error bars on mean fold rho
    ax.errorbar(
        x=df["Mean_Fold_Rho"],
        y=range(len(df)),
        xerr=df["Std_Fold_Rho"],
        fmt='none',
        capsize=5,
        color='black'
    )

    # Overlay OOF points
    sns.scatterplot(
        data=df,
        y="Model",
        x="OOF_Rho",
        color="black",
        s=80,
        zorder=5,
        label="OOF ρ"
    )

    plt.xlabel("Spearman ρ")
    plt.ylabel("Model")
    plt.title("Model Ranking Performance (10-fold CV)")
    plt.legend()
    plt.tight_layout()
    plt.savefig('../Tile_Tole_Figures/HorizontalModel Performances7_28.svg')

plot_combined_seaborn(results)

# --- paper-figure export ---
_dfres6d = pd.DataFrame([{"Model": k,
                          "Mean_Fold_Rho": v["mean_fold_rho"],
                          "Std_Fold_Rho": v["std_fold_rho"],
                          "OOF_Rho": v["oof_rho"]} for k, v in results.items()]).sort_values("Mean_Fold_Rho", ascending=False)
export_panel("Fig6d_ModelComparison_SpearmanRho",
             data=_dfres6d,
             note="Fig 6d. Spearman rho model-comparison (10-fold CV): per-fold mean +/- std and out-of-fold rho for each model.")

## Multiple Linear Regression

In [ ]:
def oof_predictions(model, X, y, n_splits=10, random_state=0):
    """
    Returns out-of-fold predictions for any sklearn-like model with fit/predict.
    """
    X = np.asarray(X)
    y = np.asarray(y).ravel()

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_oof = np.empty_like(y, dtype=float)

    for train_idx, test_idx in kf.split(X):
        model.fit(X[train_idx], y[train_idx])
        y_oof[test_idx] = model.predict(X[test_idx])
        #y_oof[test_idx] = np.sum(X[test_idx], axis=1)

    return y_oof

def plot_rank_vs_rank(y_true, y_pred, title="Rank vs Rank (OOF)"):
    """
    Rank-vs-rank scatter with Spearman rho, all points together.
    """
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    # Use average ranks for ties (standard for Spearman)
    r_true = rankdata(y_true, method="average")
    r_pred = rankdata(y_pred, method="average")

    rho, p = spearmanr(y_true, y_pred)

    plt.figure()
    plt.scatter(r_true, r_pred, alpha = 0.5)

    # Identity line
    lo = min(r_true.min(), r_pred.min())
    hi = max(r_true.max(), r_pred.max())
    plt.plot([lo, hi], [lo, hi])

    plt.text(
        0.05, 0.95,
        f"Spearman ρ = {rho:.3f}\n(p = {p:.2g})",
        transform=plt.gca().transAxes,
        va="top"
    )

    plt.xlabel("True rank")
    plt.ylabel("Predicted rank")
    plt.title(title)
    plt.tight_layout()
    plt.show()

# --- usage ---

tree = LinearRegression()


model = tree #...  # e.g., your XGBRegressor() or linear model, etc.
y_oof = oof_predictions(model, X, y, n_splits=10, random_state=0)
plot_rank_vs_rank(y, y_oof)

In [ ]:
X = deltaG_Arr

y = df_median['Exp_DeltaG']

model = LinearRegression()
model.fit(X, y)
y_Pred=model.predict(X)

#print("Coefficients:", model.coef_)
#print("Intercept:", model.intercept_)

plt.figure()
plt.scatter(y_Pred, y, alpha = 0.2)
plt.plot([y.min(), y.max()],
         [y.min(), y.max()])  # identity line
plt.xlabel("Linear Regression Predicted ΔG (Kcal/mol)")
plt.ylabel("Experimental ΔG (Kcal/mol)")
#plt.title("Actual vs Linear Regression Predicted ΔG(Kcal/mol)")
plt.tight_layout()
plt.savefig('../Tile_Tole_Figures/LinearRegression_Scatter.svg')

residuals = y - y_Pred
plt.figure()
plt.scatter(y_Pred, residuals)
plt.axhline(0)
plt.xlabel("Linear Regression Predicted ΔG (Kcal/mol)")
plt.ylabel("Residuals")
#plt.title("Residuals vs Predicted")
plt.tight_layout()
plt.savefig('../Tile_Tole_Figures/LinearRegression_Residuals.svg')

sns.barplot(model.coef_)
plt.show()
print("R^2: "+ str(model.score(X, y)))
rho, pval = spearmanr(y, y_Pred)
print("Spearman rho:", rho)

## Add Pairwise Interactions

In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# X: (n_samples, n_features), y: (n_samples,)
model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False,interaction_only=True),  # adds x_i, x_i*x_j, and x_i^2
    LinearRegression()
)

model.fit(X, y)
y_pred = model.predict(X)
y_Pred=model.predict(X)
#print(model.score(X, y))
#print("Coefficients:", model.coef_)
#print("Intercept:", model.intercept_)

plt.figure()
plt.scatter(y_Pred, y, alpha = 0.2)
plt.plot([y.min(), y.max()],
         [y.min(), y.max()])  # identity line
plt.xlabel("Quadratic Regression Predicted ΔG (Kcal/mol)")
plt.ylabel("Experimental ΔG (Kcal/mol)")
#plt.title("Actual vs Linear Regression Predicted ΔG(Kcal/mol)")
plt.tight_layout()
plt.savefig('../Tile_Tole_Figures/QuadraticRegression_Scatter.svg')

residuals = y - y_Pred
plt.figure()
plt.scatter(y_Pred, residuals)
plt.axhline(0)
plt.xlabel("Quadratic Regression Predicted ΔG (Kcal/mol)")
plt.ylabel("Residuals")
#plt.title("Residuals vs Predicted")
plt.tight_layout()
plt.savefig('../Tile_Tole_Figures/QuadraticRegression_Residuals.svg')


print("R^2: "+ str(model.score(X, y)))
rho, pval = spearmanr(y, y_Pred)
print("Spearman rho:", rho)

In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# X: (n_samples, n_features), y: (n_samples,)
model = make_pipeline(
    PolynomialFeatures(degree=3, include_bias=False,interaction_only=True),  # adds x_i, x_i*x_j, and x_i^2
    LinearRegression()
)

model.fit(X, y)
y_pred = model.predict(X)
y_Pred=model.predict(X)
#print(model.score(X, y))
#print("Coefficients:", model.coef_)
#print("Intercept:", model.intercept_)

plt.figure()
plt.scatter(y_Pred, y, alpha = 0.2)
plt.plot([y.min(), y.max()],
         [y.min(), y.max()])  # identity line
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Actual vs Predicted")
plt.show()

residuals = y - y_Pred
plt.figure()
plt.scatter(y_Pred, residuals)
plt.axhline(0)
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted")
plt.show()

print("R^2: "+ str(model.score(X, y)))

In [ ]:
model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False,interaction_only=True),  # adds x_i, x_i*x_j, and x_i^2
    LinearRegression()
)

model.fit(X, y)
y_pred = model.predict(X)
y_Pred=model.predict(X)
#print(model.score(X, y))
#print("Coefficients:", model.coef_)
#print("Intercept:", model.intercept_)

plt.figure()
plt.scatter(y_Pred, y, alpha = 0.2)
plt.plot([y.min(), y.max()],
         [y.min(), y.max()])  # identity line
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Actual vs Predicted")
plt.show()

residuals = y - y_Pred
plt.figure()
plt.scatter(y_Pred, residuals)
plt.axhline(0)
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted")
plt.show()

print("R^2: "+ str(model.score(X, y)))

In [ ]:
model = make_pipeline(
    PolynomialFeatures(degree=2),
    LinearRegression()
)

cv_r2 = cross_val_score(model, X, y, cv=5, scoring='r2')

print("Mean CV R²:", cv_r2.mean())

In [ ]:
model = make_pipeline(
    PolynomialFeatures(degree=1),
    LinearRegression()
)

cv_r2 = cross_val_score(model, X, y, cv=5, scoring='r2')

print("Mean CV R²:", cv_r2.mean())

In [ ]:
model = make_pipeline(
    PolynomialFeatures(degree=0),
    LinearRegression()
)

cv_r2 = cross_val_score(model, X, y, cv=5, scoring='r2')

print("Mean CV R²:", cv_r2.mean())

In [ ]:
model = make_pipeline(
    PolynomialFeatures(degree=3),
    LinearRegression()
)

cv_r2 = cross_val_score(model, X, y, cv=5, scoring='r2')

print("Mean CV R²:", cv_r2.mean())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline

def plot_synergy_heatmap(X, y, feature_names=None, alphas=(1e-3, 1e-2, 1e-1, 1, 10, 100), use_abs=False):
    """
    Fit y ~ main effects + pairwise interactions (no squares) and plot a heatmap of interaction coefficients.

    Parameters
    ----------
    X : array-like, shape (n_samples, n_features)
    y : array-like, shape (n_samples,)
    feature_names : list[str] or None
        Names for columns of X. If None, uses x0, x1, ...
    alphas : iterable
        Ridge regularization strengths for CV selection.
    use_abs : bool
        If True, plot absolute interaction strength |beta_ij| instead of signed beta_ij.
    """
    X = np.asarray(X)
    y = np.asarray(y).ravel()
    n, p = X.shape

    if feature_names is None:
        feature_names = [f"x{i}" for i in range(p)]
    else:
        if len(feature_names) != p:
            raise ValueError("feature_names must have length equal to number of columns in X")

    # Interaction-only: includes x_i and x_i*x_j, excludes x_i^2
    poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)

    model = Pipeline([
        ("poly", poly),
        ("ridge", RidgeCV(alphas=alphas))
    ])

    model.fit(X, y)

    # Get names of expanded features and corresponding coefficients
    expanded_names = model.named_steps["poly"].get_feature_names_out(feature_names)
    coefs = model.named_steps["ridge"].coef_

    # Build p x p synergy matrix from interaction coefficients
    S = np.zeros((p, p), dtype=float)

    name_to_idx = {name: i for i, name in enumerate(feature_names)}

    for fname, beta in zip(expanded_names, coefs):
        # interaction terms look like "A B" (space-separated) in sklearn's naming
        parts = fname.split(" ")
        if len(parts) == 2:
            a, b = parts
            i = name_to_idx[a]
            j = name_to_idx[b]
            S[i, j] = beta
            S[j, i] = beta  # symmetric

    # Optionally use absolute synergy strength
    M = np.abs(S) if use_abs else S

    # Hide diagonal (no self-interactions in interaction_only)
    M_plot = M.copy()
    np.fill_diagonal(M_plot, np.nan)
    M_plot = np.ma.array(M_plot, mask=np.eye(M_plot.shape[0], dtype=bool))
#H_masked = np.ma.array(H, mask=np.eye(H.shape[0], dtype=bool))

    # Symmetric color scaling
    max_abs = np.max(np.abs(M_plot))
    
    # Choose diverging colormap and set masked color to black
    cmap = plt.cm.RdBu_r.copy()
    cmap.set_bad(color='black')

    
    plt.figure(figsize=(0.6*p + 3, 0.6*p + 3))
    im = plt.imshow(M_plot, aspect="auto",cmap = cmap, vmin= - max_abs*0.95,vmax= max_abs*0.95)  # default colormap
    plt.colorbar(im, label="Interaction coefficient" + (" (abs)" if use_abs else ""))
    #plt.cmap('RdBu_r')
    plt.xticks(range(p), feature_names, rotation=90)
    plt.yticks(range(p), feature_names)

    plt.title("Feature synergy (pairwise interaction coefficients)")
    plt.tight_layout()
    plt.show()

    return S, model  # return signed synergy matrix and fitted model


In [ ]:
synergy_ax, model = plot_synergy_heatmap(X,y)

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

main_model = LinearRegression()

int_model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    LinearRegression()
)

r2_main = cross_val_score(main_model, X, y, cv=5, scoring="r2").mean()
r2_int  = cross_val_score(int_model, X, y, cv=5, scoring="r2").mean()

print("CV R^2 main:", r2_main)
print("CV R^2 + interactions:", r2_int)
print("ΔR^2:", r2_int - r2_main)

In [ ]:
residuals = y - y_Pred
plt.figure()
plt.scatter(y_Pred, residuals)
plt.axhline(0)
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted")
plt.show()

In [ ]:
listSamps = ['000075A0000','0000750A000','00007500A00','000075000A0']
sub_dfA = df1_A[df1_A['Samples'].isin(listSamps)]

In [ ]:
fig, ax = plt.subplots()

#sub_dfA = df1_A[df1_A['Experiment'] == 'DNAExtensionThermo11_15.csv']
sns.barplot(data = sub_dfA, x = 'Samples', y ='Tm')
#plt.xticks(rotation = 45)
ax.set_xticklabels(["0","+1","+2","+3"])
plt.ylim(40,52)
plt.ylabel('Tm (°C)')
plt.xlabel('Bond Position')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots()

sub_dfA = df1_A[df1_A['Experiment'] == 'DNAExtensionThermo11_15.csv']
sns.barplot(data = sub_dfA, x = 'Samples', y ='Steepness')
#plt.xticks(rotation = 45)
ax.set_xticklabels(["-3", "-2", "-1","0","+1","+2","+3"])
#plt.ylim(40,52)
plt.ylabel('Tm (C)')
plt.xlabel('Bond Position')
plt.tight_layout()
plt.show()
#plt.savefig('../Tile_Tole_Figures/ConstantSequence.svg')


In [ ]:
fig, ax = plt.subplots()

sub_dfA = df1_A[df1_A['Experiment'] == 'DNAExtensionThermo11_15.csv']
sns.barplot(data = sub_dfA, x = 'Samples', y ='Tm')
#plt.xticks(rotation = 45)
ax.set_xticklabels(["-3", "-2", "-1","0","+1","+2","+3"])
plt.ylim(40,52)
plt.ylabel('Tm (C)')
plt.xlabel('Bond Position')
plt.tight_layout()
plt.savefig('../Tile_Tole_Figures/ConstantSequence.svg')

# --- paper-figure export ---
_s3e = sub_dfA.reset_index(drop=True)
_uorder3e = list(dict.fromkeys(_s3e["Samples"].tolist()))
_poslbl3e = ["-3","-2","-1","0","+1","+2","+3"]
_posmap3e = {_sn:(_poslbl3e[_k] if _k < len(_poslbl3e) else str(_k)) for _k,_sn in enumerate(_uorder3e)}
export_panel("Fig3e_Tm_vs_BondPosition_ConstantSeq",
             data=pd.DataFrame({"Bond_Position": [_posmap3e[_sn] for _sn in _s3e["Samples"]],
                                "Samples": _s3e["Samples"].values,
                                "Tm": _s3e["Tm"].values}),
             note="Fig 3e. Tm as a single bond of identical sequence is moved away from two constant bonds (experiment DNAExtensionThermo11_15). Bond position relative to constant bonds.")

# --- repoint Fig3e to authoritative df_Tm ---
_a3e = df_Tm_median[df_Tm_median["Samples"].str.contains("A",na=False) & ~df_Tm_median["Samples"].str.contains("B",na=False)
             & (df_Tm_median["Experiment"]=="DNAExtensionThermo11_15.csv")].copy()
if len(_a3e) >= 2:
    plt.figure(); sns.barplot(data=_a3e, x="Code", y="Tm")
    plt.ylabel("Tm (°C)"); plt.xlabel("Bond Position"); plt.xticks(rotation=45); plt.tight_layout()
    _uo=list(dict.fromkeys(_a3e["Code"].tolist())); _lbl=["-3","-2","-1","0","+1","+2","+3"]
    _pm={s:(_lbl[k] if k<len(_lbl) else str(k)) for k,s in enumerate(_uo)}
    export_panel("Fig3e_Tm_vs_BondPosition_ConstantSeq",
                 data=pd.DataFrame({"Bond_Position":[_pm[s] for s in _a3e["Code"]],"Samples":_a3e["Samples"].values,
                                    "Tm":_a3e["Tm"].values,"Tm_SE":_a3e["Tm_SE"].values}),
                 note="Fig 3e. Tm vs bond position (constant-seq series, expt 11_15). Source: authoritative Tm table.")
    plt.close()
else:
    print("Fig3e: too few reliable A-series curves in df_Tm; kept prior export")

# --- Fig3e HIGH-reliability variant ---
_a3eh = df_Tm_median_high[df_Tm_median_high["Samples"].str.contains("A",na=False) & ~df_Tm_median_high["Samples"].str.contains("B",na=False)
                   & (df_Tm_median_high["Experiment"]=="DNAExtensionThermo11_15.csv")].copy()
if len(_a3eh) >= 2:
    plt.figure(); sns.barplot(data=_a3eh, x="Code", y="Tm")
    plt.ylabel("Tm (°C)"); plt.xlabel("Bond Position"); plt.xticks(rotation=45); plt.tight_layout()
    _uoh=list(dict.fromkeys(_a3eh["Code"].tolist())); _lblh=["-3","-2","-1","0","+1","+2","+3"]
    _pmh={s:(_lblh[k] if k<len(_lblh) else str(k)) for k,s in enumerate(_uoh)}
    export_panel_high("Fig3e_Tm_vs_BondPosition_ConstantSeq",
                      pd.DataFrame({"Bond_Position":[_pmh[s] for s in _a3eh["Code"]],"Samples":_a3eh["Samples"].values,
                                    "Tm":_a3eh["Tm"].values,"Tm_SE":_a3eh["Tm_SE"].values}))
    plt.close()
else:
    print("Fig3e high: too few high-reliability A-series curves")

In [ ]:
sub_dfA = df1_A[df1_A['Experiment'] == 'DNAExtensionThermo11_15.csv']
sns.barplot(data = sub_dfA, x = 'Samples', y ='Entropy')
plt.xticks(rotation = 45)
#ax.set_xticklabels(["-3", "-2", "-1","0","+1","+2","+3"])
#plt.ylim(40,52)

In [ ]:
sub_dfA = df1_A[df1_A['Experiment'] == 'DNAExtensionThermo11_15.csv']
sns.barplot(data = sub_dfA, x = 'Samples', y ='Steepness')
plt.xticks(rotation = 45)
#ax.set_xticklabels(["-3", "-2", "-1","0","+1","+2","+3"])
#plt.ylim(40,52)

In [ ]:
fig, ax = plt.subplots()

#sub_dfA = df1_A[df1_A['Experiment'] == 'DNAExtensionThermo11_15.csv']
sns.barplot(data = df1A_B, x = 'Samples', y ='Tm')
plt.xticks(rotation = 45)
#ax.set_xticklabels(["-3", "-2", "-1","-0","+0","+1","+2","+3"])
plt.ylim(45,55)

In [ ]:
fig, ax = plt.subplots()

sub_dfA = df1_A[df1_A['Experiment'] == 'DNAExtensionThermo11_17.txt']
sns.barplot(data = sub_dfA, x = 'Samples', y ='Tm')
plt.xticks(rotation = 90)
#ax.set_xticklabels(["-3", "-2", "-1","-0","+0","+1","+2","+3"])
plt.ylim(35,52)

In [ ]:
df1_mini = df1[df1['NucleotideBonds'] == 5]
sns.scatterplot(data = df1_mini, x = 'Distances', y ='factor', hue='Num_Bonds',palette='colorblind')
#plt.ylim(0,0.7)
plt.title('5nt Long Bonds')

In [ ]:
df1_mini = df1[df1['NucleotideBonds'] == 7]
df1_mini = df1_mini[df1_mini['factor'] >0]
sns.scatterplot(data = df1_mini, x = 'Distances', y ='factor', hue='Num_Bonds',palette='colorblind')
#plt.ylim(0,1.4)
plt.title('7nt Long Bonds')

In [ ]:
df1_mini = df1[df1['NucleotideBonds'] == 9]
sns.scatterplot(data = df1_mini, x = 'Distances', y ='factor', hue='Num_Bonds',palette='colorblind')
#plt.ylim(0,0.4)
plt.title('9nt Long Bonds')

In [ ]:
#df1_mini = df_median[df_median['factor'] >0]
df1_mini = df_median[df_median['Num_Bonds'] !=2]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='A']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='B']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=9]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=6]
plt.figure(figsize=(10, 6))
ax = sns.catplot(
    data=df1_mini, x="Num_Bonds", y="Tm", col="NucleotideBonds",
    kind="swarm", height=4, aspect=.6,palette='colorblind',sharex=False)
plt.ylim(30,50)
ax.set_titles("{col_name}nts")
plt.ylabel('Tm (°C)')
plt.xlabel('Binding Sites')
ax.set_axis_labels("Binding Sites", "Tm (°C)")
plt.savefig('../Tile_Tole_Figures/NumBonds_Tm_SwarmPlot.svg')

In [ ]:
#df1_mini = df_median[df_median['factor'] >0]
df1_mini = df_median[df_median['Num_Bonds'] !=2]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='A']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='B']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=9]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=6]
plt.figure(figsize=(10, 6))
ax = sns.catplot(
    data=df1_mini, x="Num_Bonds", y="Tm", col="NucleotideBonds",
    kind="strip", height=4, aspect=.6,palette='colorblind',sharex=False,alpha = 0.5)
plt.ylim(30,55)
ax.set_titles("{col_name}nts")
plt.ylabel('Tm (°C)')
plt.xlabel('Binding Sites')
ax.set_axis_labels("Binding Sites", "Tm (°C)")
plt.show()

In [ ]:
import numpy as np

g = sns.catplot(
    data=df1_mini, x="Num_Bonds", y="Tm", col="NucleotideBonds",
    kind="strip", height=4, aspect=.6, palette='colorblind',jitter=0.2,
    sharex=False, alpha=0.5,
)

# overlay per-facet, per-category medians as black dashes
g.map_dataframe(
    sns.pointplot, x="Num_Bonds", y="Tm",
    estimator=np.median, errorbar=None,        # seaborn ≥0.12; use ci=None if older
    color='k', markers='_', linestyle='none',  # linestyle='none' replaces join=False (≥0.13)
    markersize=18, markeredgewidth=1.5, zorder=5,
)

g.set_titles("{col_name}nts")
g.set_axis_labels("Binding Sites", "Tm (°C)")
g.set(ylim=(30, 50))
#plt.show()
plt.savefig('../Tile_Tole_Figures/NumBonds_Tm_Stripplot.svg', dpi = 600)

# --- paper-figure export ---
export_panel("Fig2a_Tm_vs_BindingSites_x_Length",
             data=df1_mini[["Samples","NucleotideBonds","Num_Bonds","Tm"]],
             note="Fig 2a. Measured Tm (per replicate) vs number of binding sites, faceted by bond length (nt). Source: Tm reproducibility-filtered data.")

# --- repoint Fig2a to authoritative df_Tm ---
_g2a = sns.catplot(data=df_Tm_median, x="Num_Bonds", y="Tm", col="NucleotideBonds",
                   kind="strip", height=4, aspect=.6, palette='colorblind', jitter=0.2, sharex=False, alpha=0.5)
_g2a.map_dataframe(sns.pointplot, x="Num_Bonds", y="Tm", estimator=np.median, errorbar=None,
                   color='k', markers='_', linestyle='none', markersize=18, markeredgewidth=1.5, zorder=5)
_g2a.set_titles("{col_name}nts"); _g2a.set_axis_labels("Binding Sites", "Tm (°C)")
export_panel("Fig2a_Tm_vs_BindingSites_x_Length",
             data=df_Tm_median[["Code","NucleotideBonds","Num_Bonds","Tm","Tm_SE","plateau_confident"]],
             note="Fig 2a. Droop-corrected median Tm (one point per interface) vs binding sites, faceted by bond length. Source: authoritative Tm table (reliability rule; Tm_SE + plateau_confident included).")
plt.close('all')

# --- Fig2a HIGH-reliability variant ---
_g2ah = sns.catplot(data=df_Tm_median_high, x="Num_Bonds", y="Tm", col="NucleotideBonds",
                    kind="strip", height=4, aspect=.6, palette='colorblind', jitter=0.2, sharex=False, alpha=0.5)
_g2ah.map_dataframe(sns.pointplot, x="Num_Bonds", y="Tm", estimator=np.median, errorbar=None,
                    color='k', markers='_', linestyle='none', markersize=18, markeredgewidth=1.5, zorder=5)
_g2ah.set_titles("{col_name}nts"); _g2ah.set_axis_labels("Binding Sites", "Tm (°C)")
export_panel_high("Fig2a_Tm_vs_BindingSites_x_Length",
                  df_Tm_median_high[["Code","NucleotideBonds","Num_Bonds","Tm","Tm_SE"]])
plt.close('all')

In [ ]:
#df1_mini = df_median[df_median['factor'] >0]
df1_mini = df_median[df_median['Num_Bonds'] !=2]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='A']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='B']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=9]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=6]
plt.figure(figsize=(10, 6))
ax = sns.catplot(
    data=df1_mini, x="Num_Bonds", y="Tm", col="NucleotideBonds",
    kind="violin", height=4, aspect=.6,palette='colorblind',sharex=False,alpha = 0.5)
plt.ylim(30,55)
ax.set_titles("{col_name}nts")
plt.ylabel('Tm (°C)')
plt.xlabel('Binding Sites')
ax.set_axis_labels("Binding Sites", "Tm (°C)")
plt.show()
#plt.savefig('../Tile_Tole_Figures/NumBonds_Tm_SwarmPlot.svg')

In [ ]:
import numpy as np
from scipy import stats

def welch_tm(tm_a, tm_b, label_a="Class A", label_b="Class B", alpha=0.05):
    """
    Welch's unequal-variance t-test comparing melting temperatures of two DNA classes.
    Reports the mean difference (ΔTm) with a CI, not just a p-value.

    Parameters
    ----------
    tm_a, tm_b : 1D array-like of Tm values (°C), one value per experimental unit
                 (per construct/sequence — NOT per technical melt if those are
                 repeated measures of the same construct; aggregate first if so).
    """
    a = np.asarray(tm_a, dtype=float)
    b = np.asarray(tm_b, dtype=float)
    a = a[~np.isnan(a)]
    b = b[~np.isnan(b)]

    na, nb = a.size, b.size
    ma, mb = a.mean(), b.mean()
    # ddof=1 -> sample variance (n-1), matches Welch's formulation
    va, vb = a.var(ddof=1), b.var(ddof=1)

    diff = ma - mb                       # ΔTm = mean(A) - mean(B)
    se = np.sqrt(va / na + vb / nb)      # Welch standard error of the difference

    # Welch–Satterthwaite degrees of freedom (non-integer, as expected)
    df = se**4 / ((va / na)**2 / (na - 1) + (vb / nb)**2 / (nb - 1))

    t_stat, p_val = stats.ttest_ind(a, b, equal_var=False)  # Welch's

    t_crit = stats.t.ppf(1 - alpha / 2, df)
    ci_low, ci_high = diff - t_crit * se, diff + t_crit * se

    # Standardized effect size robust to unequal variance (Glass's Δ, using B as reference).
    # Swap the denominator to a's SD if A is your control/reference class.
    glass_delta = diff / np.sqrt(vb)

    print(f"{label_a}: n={na}, mean={ma:.3f} °C, sd={np.sqrt(va):.3f}")
    print(f"{label_b}: n={nb}, mean={mb:.3f} °C, sd={np.sqrt(vb):.3f}")
    print("-" * 52)
    print(f"ΔTm ({label_a} − {label_b}) = {diff:+.3f} °C")
    print(f"{100*(1-alpha):.0f}% CI: [{ci_low:+.3f}, {ci_high:+.3f}] °C")
    print(f"Welch t({df:.1f}) = {t_stat:.3f},  p = {p_val:.3g}")
    print(f"Glass's Δ = {glass_delta:.3f}")

    return {
        "diff": diff, "ci": (ci_low, ci_high), "se": se,
        "t": t_stat, "df": df, "p": p_val, "glass_delta": glass_delta,
    }

# --- example usage: replace with your Tm arrays ---
#tm_class1 = [64.2, 65.1, 63.8, 66.0, 64.7, 65.3]
#tm_class2 = [61.5, 62.3, 60.9, 63.1, 61.8]
df5_mini=df1_mini[df1_mini['NucleotideBonds'] == 5]
tm_class2=df5_mini[df5_mini['Num_Bonds'] ==9]['Tm'].values
tm_class1=df5_mini[df5_mini['Num_Bonds'] ==10]['Tm'].values

res = welch_tm(tm_class1, tm_class2, label_a="10 Bonds", label_b="9 Bonds")

In [ ]:
df5_mini=df1_mini[df1_mini['NucleotideBonds'] ==7]
tm_class2=df5_mini[df5_mini['Num_Bonds'] ==5]['Tm'].values
tm_class1=df5_mini[df5_mini['Num_Bonds'] ==4]['Tm'].values

res = welch_tm(tm_class1, tm_class2, label_a="4 Bonds", label_b="15 Bonds")

In [ ]:
#df1_mini = df_median[df_median['factor'] >0]
df1_mini = df_median[df_median['Num_Bonds'] !=2]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='A']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='B']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=9]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=6]
plt.figure(figsize=(10, 6))
ax = sns.catplot(
    data=df1_mini, x="Num_Bonds", y="Tm", col="NucleotideBonds",
    kind="bar", height=4, aspect=.6,palette='colorblind',sharex=False)
plt.ylim(30,50)
ax.set_titles("{col_name}nts")
plt.ylabel('Tm (°C)')
plt.xlabel('Binding Sites')
ax.set_axis_labels("Binding Sites", "Tm (°C)")
plt.show()
#plt.savefig('../Tile_Tole_Figures/NumBonds_Tm.svg')

In [ ]:
#sns.scatterplot(x = df1_mini['MeanGap'],y = df1_mini['Steepness'])

df1_mini = df_median[df_median['Num_Bonds'] !=2]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='A']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='B']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=9]

#df1_mini = df1_mini[df1_mini['Num_Bonds'] !=2]
#df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=9]

ax = sns.catplot(
    data=df1_mini, x="Num_Bonds", y="Steepness", col="NucleotideBonds",
    kind="bar", height=4, aspect=.6,palette='colorblind')
#plt.ylim(30,50)
ax.set_titles("{col_name}nts")
plt.tight_layout()

In [ ]:
#sns.scatterplot(x = df1_mini['MeanGap'],y = df1_mini['Steepness'])

df1_mini = df_median[df_median['Num_Bonds'] !=2]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='A']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='B']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=6]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=9]

#df1_mini = df1_mini[df1_mini['Num_Bonds'] !=2]
#df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=9]

ax = sns.catplot(
    data=df1_mini, x="Num_Bonds", y="Steepness", col="NucleotideBonds",
    kind="bar", height=4, aspect=.6,palette='colorblind',sharex=False)
#plt.ylim(30,50)
plt.tight_layout()
plt.ylim(0.5,2.5)
ax.set_titles("{col_name}nts")
ax.set_xlabels('Binding Sites')
plt.savefig('../Tile_Tole_Figures/Steepness_BondLength.svg')
#plt.savefig('NumBonds_Steepness.eps', format='eps', dpi=1000)

In [ ]:
df1_mini = df_median[df_median['Num_Bonds'] !=2]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='A']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='B']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=9]

#for x in np.unique(df1_mini['NucleotideBonds']):
#    df2Mini = df1_mini[df1_mini['NucleotideBonds'] == x]
#    sns.barplot(data = df2Mini, x ='Num_Bonds',y = 'factor')
#    plt.ylim(0.15,0.4)
#    plt.show()
ax = sns.catplot(
    data=df1_mini, x="Num_Bonds", y="factor", col="NucleotideBonds",
    kind="bar", height=4, aspect=.6,palette='colorblind', sharex=False,sharey = True)
#p#lt.ylim(0,1)
#plt.ylim(0.15,0.4)
ax.set_titles("{col_name}nts")


In [ ]:
df1_mini = df_median[df_median['Num_Bonds'] !=2]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='A']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='B']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=9]

#for x in np.unique(df1_mini['NucleotideBonds']):
#    df2Mini = df1_mini[df1_mini['NucleotideBonds'] == x]
#    sns.barplot(data = df2Mini, x ='Num_Bonds',y = 'factor')
#    plt.ylim(0.15,0.4)
#    plt.show()
ax = sns.catplot(
    data=df1_mini, x="Num_Bonds", y="factor", col="NucleotideBonds",
    kind="swarm", height=4, aspect=.6,palette='colorblind', sharex=False,sharey = True)
#p#lt.ylim(0,1)
#plt.ylim(0.15,0.4)
ax.set_titles("{col_name}nts")


In [ ]:
#df_median[df_median['Num_Bonds'] == 9]
#df_median['H_S'] = df_median['Predicted_deltaH']/df_median['Predicted_deltaS']
x1 = df_median[['Predicted_tm']]
y1= df_median['Tm']
#sns.scatterplot(x=x1, y=y1)
lin_model = LinearRegression().fit(x1, y1)
y_lin = lin_model.predict(x1)


# ----- Linear Fit -----
#x_reshaped = x.reshape(-1,1)
#lin_model = LinearRegression().fit(x_reshaped, y)
#y_lin = lin_model.predict(x_reshaped)


r2_lin = r2_score(y1, y_lin)

# Residual sum of squares
rss_lin = np.sum((y1 - y_lin)**2)
#x_fit = np.linspace(x.min(), x.max(), 300)

print("Linear R²:", r2_lin)
#print("Exp R²:", r2_exp)
print("Linear RSS:", rss_lin)
#print("Exp RSS:", rss_exp)

x_fit = np.linspace(x1.min(), x1.max(), 300)
#y_fit = exp_decay(x_fit, *params)

# ----- Plot -----
sns.scatterplot(x=df_median['Predicted_tm'], y= y1, label='Data')
plt.plot(x1, y_lin ,label='Linear Fit')
#plt.plot(x_fit, y_fit, label='Exponential')
plt.xlabel('Sequence Predicted Tm')
plt.ylim(25,55)
plt.legend()
plt.savefig('Linear_Sequence_Melting.svg', dpi = 600)

In [ ]:
df1_mini = df_median[df_median['Num_Bonds'] !=2]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='A']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='B']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=9]

#for x in np.unique(df1_mini['NucleotideBonds']):
#    df2Mini = df1_mini[df1_mini['NucleotideBonds'] == x]
#    sns.barplot(data = df2Mini, x ='Num_Bonds',y = 'factor')
#    plt.ylim(0.15,0.4)
#    plt.show()
ax = sns.catplot(
    data=df1_mini, x="Num_Bonds", y="Tm", col="NucleotideBonds",size = 5,
    kind="swarm", height=4, aspect=.6,palette='colorblind', sharex=False,sharey = True)
#p#lt.ylim(0,1)
#plt.ylim(0.15,0.4)
ax.set_titles("{col_name}nts")


In [ ]:
df1_mini = df_median[df_median['Num_Bonds'] !=2]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='A']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='B']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=9]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=6]

#for x in np.unique(df1_mini['NucleotideBonds']):
#    df2Mini = df1_mini[df1_mini['NucleotideBonds'] == x]
#    sns.barplot(data = df2Mini, x ='Num_Bonds',y = 'factor')
#    plt.ylim(0.15,0.4)
#    plt.show()
ax = sns.catplot(
    data=df1_mini, x="Num_Bonds", y="Entropy", col="NucleotideBonds",
    kind="bar", height=4, aspect=.6,palette='colorblind', sharex=False,sharey = True)
#p#lt.ylim(0,1)
#plt.ylim(0.15,0.4)
ax.set_titles("{col_name}nts")


In [ ]:
df1_mini = df_median[df_median['Num_Bonds'] !=2]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='A']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='B']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=9]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=5]

factors = list()
numBonds = list()
for x in np.unique(df1_mini['NucleotideBonds']):
    df2Mini = df1_mini[df1_mini['NucleotideBonds'] == x]
    for y in np.unique(df2Mini['Num_Bonds']):
        df3Mini=df2Mini[df2Mini['Num_Bonds'] == y]
        df3Mini.reset_index(drop=True, inplace = True)
        #print(y)
        for z in range(len(df3Mini)):
            factors.append(df3Mini['factor'][z])
            numBonds.append(y)
        #sns.barplot(data = df2Mini, x ='Num_Bonds',y = 'factor')
    #plt.ylim(0.15,0.4)
    #plt.show()
#ax = sns.catplot(
#    data=df1_mini, x="Num_Bonds", y="factor", col="NucleotideBonds",
#    kind="bar", height=4, aspect=.6,palette='colorblind', sharex=False,sharey = True)
#p#lt.ylim(0,1)
#plt.ylim(0.15,0.4)
#ax.set_titles("{col_name}nts")


In [ ]:
for y in np.unique(df_median['Num_Bonds']):
    df3Mini=df_median[df_median['Num_Bonds'] == y]
    print(np.median(df3Mini['factor']))
    print(y)

In [ ]:
dfFactor=pd.DataFrame({'Bond Count':numBonds,'Factor':factors})

In [ ]:
dfFactor=dfFactor.sort_values(by='Factor', ascending = False)

In [ ]:
#dfFactor=dfFactor[dfFactor['Bond Count'] > 8]
slope, intercept, r_val, p_val, std_err = stats.linregress(dfFactor['Bond Count'], dfFactor['Factor'])
#print(slope)
sns.swarmplot(data = dfFactor, x='Bond Count', y='Factor')
plt.title('5nt bonds')
plt.ylabel('ΔΔG')
#plt.ylim(-55,-42)
plt.xlabel('Binding Sites')

In [ ]:
from scipy import stats

In [ ]:
#dfFactor=dfFactor[dfFactor['Bond Count'] < 6]
slope, intercept, r_val, p_val, std_err = stats.linregress(dfFactor['Bond Count'], dfFactor['Factor'])
print(slope)
sns.regplot(data = dfFactor, x='Bond Count', y='Factor')
plt.title('7nt bonds')
plt.ylabel('ΔΔG')
#plt.ylim(-55,-35)
plt.xlabel('Binding Sites')

In [ ]:
decays = list()
for i in range(len(dfFactor)-1):
    decay = dfFactor['Factor'][i+1]/dfFactor['Factor'][i]
    decays.append(decay)

In [ ]:
decays2 = list()
for i in range(len(dfFactor)-1):
    decay = dfFactor['Factor'][i+1]/dfFactor['Factor'][i]
    decays.append(decay)

In [ ]:
def exp_decay(x, A, k, C):
    return A * np.exp(-k * x) + C
x=numBonds
y = factors



x_reshaped = np.array(x).reshape(-1,1)

# Fit model
model = LinearRegression()
model.fit(x_reshaped, y)

# Predict
y_fitLinear = model.predict(x_reshaped)
#plt.scatter(x, y, label="Data")
#plt.plot(x, y_fit, label="Linear fit", linewidth=2)
#plt.xlabel("X")
#plt.ylabel("Y")
#plt.legend()
#plt.show()




# Initial guesses (important!)
p0 = [max(y), 0.1, min(y)]

# Fit
params, cov = curve_fit(exp_decay, x, y, p0=p0)

A, k, C = params

# Generate fit curve
x_fit = np.linspace(min(x), max(x), 200)
y_fit = exp_decay(x_fit, A, k, C)

# Plot
plt.scatter(x, y, alpha = 0.5, label = 'Data')
plt.plot(x_fit, y_fit, label = 'Exponential fit', color = 'green')
#plt.plot(x, y_fitLinear, label="Linear fit", linewidth=2, color = 'red')
plt.ylabel('Normalized Binding Strength')
plt.xlabel('Number of Binding Positions')
plt.legend()
plt.tight_layout()
#plt.savefig('../Tile_Tole_Figures/Decaying_Factor.svg')


#print("A =", A)
#print("k =", k)
#print("C =", C)

# --- paper-figure export (CORRECTED) ---
# Paper "normalized binding strength" = measured dG / NN-predicted dG (methods: measured
# interaction free energies normalized BY the sum of NN-predicted bond contributions).
# This is a dimensionless ratio in ~(0,1) that decays with binder count; the notebook's
# `factor` (Predicted - Exp) is the cooperativity *deficit* in kcal/mol and is NOT this.
# Pool all bond types (that is the point of normalizing) and all N.
_f2d = df_median.copy()
_f2d["_pred"] = pd.to_numeric(_f2d["Predicted_deltaG"], errors="coerce")
_f2d["_exp"]  = pd.to_numeric(_f2d["Exp_DeltaG"], errors="coerce")
_f2d["_nbs"]  = pd.to_numeric(_f2d["Num_Bonds"], errors="coerce")
_f2d = _f2d[(_f2d["_pred"].abs() > 1e-6) & _f2d["_exp"].notna() & _f2d["_nbs"].notna()].copy()
_f2d["NormStrength"] = _f2d["_exp"] / _f2d["_pred"]
_x2d = _f2d["_nbs"].to_numpy(float); _y2d = _f2d["NormStrength"].to_numpy(float)
def _expdecay2d(xx, A, k, C): return A*np.exp(-k*xx) + C
_p02d = [max(_y2d)-min(_y2d), 0.3, min(_y2d)]
try:
    _par2d, _ = curve_fit(_expdecay2d, _x2d, _y2d, p0=_p02d, maxfev=20000)
except Exception:
    _par2d = None
plt.figure()
plt.scatter(_x2d, _y2d, alpha=0.4, label="Data")
if _par2d is not None:
    _xf2d = np.linspace(_x2d.min(), _x2d.max(), 200)
    plt.plot(_xf2d, _expdecay2d(_xf2d, *_par2d), color="green", label="Exponential fit")
plt.xlabel("Number of Binding Positions")
plt.ylabel("Normalized Binding Strength\n(measured ΔG / NN-predicted ΔG)")
plt.legend(); plt.tight_layout()
export_panel("Fig2d_NormalizedBindingStrength",
             data=pd.DataFrame({"Samples": _f2d["Samples"].values,
                                "Num_Binding_Positions": _x2d,
                                "Normalized_Binding_Strength": _y2d}),
             note="Fig 2d. Normalized binding strength = measured dG / NN-predicted dG (per construct, all bond types), vs number of binding positions. Green curve = exponential-decay fit. Dimensionless ratio decaying from ~0.5 (N=2) to ~0.2 (N=10) = diminishing per-bond returns.")
plt.close()

In [ ]:
dfFactor=dfFactor.sort_values(by='Factor', ascending = False)

In [ ]:
x=np.array(dfFactor['Bond Count'])
y = np.array(dfFactor['Factor'])

# ----- Linear Fit -----
x_reshaped = x.reshape(-1,1)
lin_model = LinearRegression().fit(x_reshaped, y)
y_lin = lin_model.predict(x_reshaped)

# ----- Exponential Fit -----
def exp_decay(x, A, k, C):
    return A * np.exp(-k * x) + C

p0 = [max(y), 0.1, min(y)]
params, _ = curve_fit(exp_decay, x, y, p0=p0)
y_exp = exp_decay(x, *params)

# ----- Compare Metrics -----
r2_lin = r2_score(y, y_lin)
r2_exp = r2_score(y, y_exp)

# Residual sum of squares
rss_lin = np.sum((y - y_lin)**2)
rss_exp = np.sum((y - y_exp)**2)
#x_fit = np.linspace(x.min(), x.max(), 300)

print("Linear R²:", r2_lin)
print("Exp R²:", r2_exp)
print("Linear RSS:", rss_lin)
print("Exp RSS:", rss_exp)

x_fit = np.linspace(x.min(), x.max(), 300)
y_fit = exp_decay(x_fit, *params)

# ----- Plot -----
plt.scatter(x, y, label='Data')
#plt.plot(x, y_lin, label='Linear')
plt.plot(x_fit, y_fit, label='Exponential')
plt.legend()
plt.show()

In [ ]:
n = len(y)
AIC_lin = n*np.log(rss_lin/n) + 2*2
AIC_exp = n*np.log(rss_exp/n) + 2*3

print("AIC Linear:", AIC_lin)
print("AIC Exp:", AIC_exp)

In [ ]:
df1_mini = df_median[df_median['Num_Bonds'] !=2]
#df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='A']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='B']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=9]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=6]

ax = sns.catplot(
    data=df1_mini, x="Num_Bonds", y="Tm", col="NucleotideBonds",
    kind="bar", height=4, aspect=.6,palette='colorblind')
#plt.ylim(0,1)
ax.set_titles("{col_name} nts")

In [ ]:
df1_mini = df_median[df_median['Num_Bonds'] !=2]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='A']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='B']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=9]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] ==6]

ax = sns.catplot(
    data=df1_mini, x="Num_Bonds", y="factor", col="NucleotideBonds",
    kind="bar", height=4, aspect=.6,palette='colorblind')
plt.ylim(0,1)
ax.set_titles("{col_name} nts")
#plt.savefig('NumBonds_Factor.eps', format='eps', dpi=1000)

In [ ]:
df1_mini = df_median[df_median['Num_Bonds'] !=2]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='A']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='B']
#df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=9]
ax = sns.catplot(
    data=df1_mini, x="Num_Bonds", y="Entropy", col="NucleotideBonds",
    kind="bar", height=4, aspect=.6,palette='colorblind'
)
#plt.ylim(0,1)
#plt.savefig('NumBonds_Factor.eps', format='eps', dpi=1000)

In [ ]:
df1_mini = df_median[df_median['Num_Bonds']==4]
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='A']
df1_mini = df1_mini[df1_mini['NucleotideBonds'] !='B']
#df1_mini = df1_mini[df1_mini['NucleotideBonds'] !=9]
#ax = sns.catplot(
#    data=df1_mini, x="Num_Bonds", y="Entropy", col="NucleotideBonds",
#    kind="bar", height=4, aspect=.6,palette='colorblind'
#)
sns.barplot(data = df1_mini, x = 'NucleotideBonds', y = 'Entropy')
plt.xlabel('Bond length (nts)')
plt.ylabel('Measured ΔS (kcal/mol)')
plt.tight_layout()
#plt.ylim(0,1)
plt.savefig('../Tile_Tole_Figures/Entropy7vs9.svg')

In [ ]:
dfMini = df_median[df_median['NucleotideBonds'] !=9]
sns.scatterplot(x=dfMini['Entropy'],y =df_median['factor'])

In [ ]:
df1_mini = df1[df1['Num_Bonds'] == 4]
df1_mini=df1_mini[df1_mini['NucleotideBonds'] == 7]

In [ ]:
df4=df_median[df_median['Num_Bonds'] ==4]

In [ ]:

def spacing_metrics(df, sample_var=False):
    df = df.reset_index(drop = True)
    meanGaps = list()
    Vars = list()
    for i in range(len(df)):
        arr=df.loc[i,'Positions']
        arr.sort()
        arr = np.asarray(arr)
        gaps = np.diff(arr)
        mean_nn = gaps.mean()
        var_gaps = gaps.var(ddof=1) #NEED TO CHECKKK
        meanGaps.append(mean_nn)
        Vars.append(var_gaps)
    return meanGaps, Vars

mean_nn, var_gaps = spacing_metrics(df_median)
df_median['MeanGap'] = mean_nn
df_median['VarGap'] = var_gaps
df_median['MeanGap_nm'] = df_median['MeanGap'] * (88.0/10)   # 8.8 nm per site (88 nm / 10 intervals)
if 'Distances' in df_median.columns:
    df_median['Distances_nm'] = df_median['Distances'] * (88.0/10)


In [ ]:
def skew(df):
    df = df.reset_index(drop = True)
    skew = list()
    #Vars = list()
    for i in range(len(df)):
        arr=df.loc[i,'Positions']
        arr.sort()
        arr = np.asarray(arr)
        skewTot = 0
        for a in arr:
            skew1 = abs(a - 5)
            skewTot = skewTot + skew1
        skew.append(skewTot)
    return skew

df4['SkewAbs'] = skew(df4)
#df_median['MeanGap'] = mean_nn
#df_median['VarGap'] = var_gaps

In [ ]:
def skewDi(df):
    df = df.reset_index(drop = True)
    skew = list()
    #Vars = list()
    for i in range(len(df)):
        arr=df.loc[i,'Positions']
        arr.sort()
        arr = np.asarray(arr)
        skewTot = 0
        for a in arr:
            skew1 = a - 5
            skewTot = skewTot + skew1
        skew.append(skewTot)
    return skew

df4['SkewDi'] = skewDi(df4)

In [ ]:
df4=df4[df4['NucleotideBonds'] == 7]

In [ ]:
mean_nn, var_gaps = spacing_metrics(df4)
df4['MeanGap'] = mean_nn
df4['VarGap'] = var_gaps
df4['MeanGap_nm'] = df4['MeanGap'] * (88.0/10)


In [ ]:
mean_nn, var_gaps = spacing_metrics(df_median)
df_median['MeanGap'] = mean_nn
df_median['VarGap'] = var_gaps
df_median['MeanGap_nm'] = df_median['MeanGap'] * (88.0/10)   # 8.8 nm per site (88 nm / 10 intervals)
if 'Distances' in df_median.columns:
    df_median['Distances_nm'] = df_median['Distances'] * (88.0/10)


In [ ]:
gaps=np.unique(df4['MeanGap'])
for gap in gaps:
    df4Mini = df4[df4['MeanGap'] == gap]
    sns.scatterplot(data = df4Mini, x = 'SkewDi', y = 'Tm')
    plt.show()

In [ ]:
_NM = 88.0/10
plt.figure(); sns.scatterplot(x=df_median['MeanGap'], y=df_median['Steepness']); plt.xlabel('Mean gap (bond sites)'); plt.show()
plt.figure(); sns.scatterplot(x=df_median['MeanGap']*_NM, y=df_median['Steepness']); plt.xlabel('Mean gap (nm)'); plt.ylabel('Steepness'); plt.show()


In [ ]:
gap = pd.to_numeric(df_median['MeanGap'], errors='coerce')
entropy = pd.to_numeric(df_median['Entropy'], errors='coerce')
sns.regplot(x = gap,y = entropy)
plt.ylim(-500,0)
plt.show()

In [ ]:
gap = pd.to_numeric(df_median['MeanGap'], errors='coerce')
entropy = pd.to_numeric(df_median['Enthalpy'], errors='coerce')
sns.regplot(x = gap,y = entropy)
plt.ylim(-200,0)
plt.show()

In [ ]:
import numpy as np, pandas as pd
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import seaborn as sns

# ---- data  (note: 'Enthalpy' column -> this is dH, not entropy) ---------
gap = pd.to_numeric(df_median['MeanGap'],      errors='coerce')   # mean inter-site distance d_bar (= mean of d_k)
dH  = pd.to_numeric(df_median['Enthalpy'],     errors='coerce')   # experimental dH_eff per construct
N   = pd.to_numeric(df_median['Num_Bonds'], errors='coerce')   # <-- set to YOUR site-count column

m = gap.notna() & dH.notna() & N.notna()
gap, dH, N = gap[m].to_numpy(), dH[m].to_numpy(), N[m].to_numpy()

# ---- model from the Tm formula ------------------------------------------
# Distance enters ONLY through g_k = exp(-d_k / y).  With a per-bond dH_NN:
#   dH_eff = dH_NN * sum_k exp(-d_k/y)  ~  dH_NN * N * exp(-d_bar/y)   (regular spacing)
# No additive offset for enthalpy; y (the decay length) is the ONLY nonlinear parameter.
def dH_model(X, dH_NN, y):
    d, n = X
    return dH_NN * n * np.exp(-d / y)

p0 = [dH.mean() / N.mean(), 1.0]                      # dH_NN ~ per-site level, y ~ a distance
popt, pcov = curve_fit(dH_model, (gap, N), dH, p0=p0,
                       bounds=([-np.inf, 1e-6], [np.inf, np.inf]), maxfev=20000)
dH_NN, y = popt
perr = np.sqrt(np.diag(pcov))
print(f"dH_NN = {dH_NN:6.2f} ± {perr[0]:.2f}  kcal/mol per bond")
print(f"y     = {y:6.3f} ± {perr[1]:.3f}  (decay length, MeanGap units)")

# ---- plot PER-SITE so all N collapse onto one exponential ---------------
xf = np.linspace(gap.min(), gap.max(), 200)
plt.figure(figsize=(6, 4))
sns.scatterplot(x=gap, y=dH / N, hue=N, palette='viridis')
plt.plot(xf, dH_NN * np.exp(-xf / y), 'k-', lw=2,
         label=fr'$\Delta H_{{NN}}\,e^{{-\bar d/y}}$  ($\Delta H_{{NN}}$={dH_NN:.1f}, $y$={y:.2f})')
plt.xlabel(r'mean inter-site distance  $\bar d$  (MeanGap)')
plt.ylabel(r'per-site enthalpy  $\Delta H_{\rm eff}/N$')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
def exponential_func(x, a, b, c):
    return a * np.exp(b * x)
    #return a * np.log(x) + b
initial_guess = [1.0, 1.0, 1.0]
#popt, pcov = curve_fit(exponential_func, gap, entropy, p0=initial_guess)

gap = pd.to_numeric(df_median['MeanGap'], errors='coerce')
entropy = pd.to_numeric(df_median['Enthalpy'], errors='coerce')

popt, pcov = curve_fit(exponential_func, gap, entropy, p0=initial_guess)
a_opt, b_opt, c_opt = popt

x_fit = np.linspace(1, 4, 100)
y_fit = exponential_func(x_fit, *popt)

plt.plot(x_fit, y_fit, color='blue', label=f'Fit: {a_opt:.2f}*e^({b_opt:.2f}*x) + {c_opt:.2f}', linewidth=2)

sns.scatterplot(x = gap,y = entropy)
plt.ylim(-200,0)
plt.show()

In [ ]:
def exponential_func(x, a, b, c):
    return a * np.exp(b * x)
    #return a * np.log(x) + b
initial_guess = [1.0, 1.0, 1.0]
#popt, pcov = curve_fit(exponential_func, gap, entropy, p0=initial_guess)

gap = pd.to_numeric(df_median['MeanGap'], errors='coerce')
entropy = pd.to_numeric(df_median['Entropy'], errors='coerce')

popt, pcov = curve_fit(exponential_func, gap, entropy, p0=initial_guess)
a_opt, b_opt, c_opt = popt

x_fit = np.linspace(1, 4, 100)
y_fit = exponential_func(x_fit, *popt)

plt.plot(x_fit, y_fit, color='blue', label=f'Fit: {a_opt:.2f}*e^({b_opt:.2f}*x) + {c_opt:.2f}', linewidth=2)

sns.scatterplot(x = gap,y = entropy)
plt.ylim(-500,0)
plt.show()

In [ ]:
gap = pd.to_numeric(df_median['MeanGap'], errors='coerce')
entropy = pd.to_numeric(df_median['Steepness'], errors='coerce')
sns.regplot(x = gap,y = entropy)
#plt.ylim(-200,0)
plt.show()

In [ ]:
from collections import defaultdict

pos_to_indices = defaultdict(list)

for i, p in enumerate(df1_mini['Positions'].values):
    for pos in p:
        pos_to_indices[pos].append(i)


In [ ]:
df1_mini = df_median

In [ ]:
_NM = 88.0/10
r_s = df1_mini["MeanGap"].corr(df1_mini["Tm"], method="spearman")
plt.figure(); sns.scatterplot(data=df1_mini, x="MeanGap", y="Tm"); plt.xlabel("Mean gap (bond sites)"); plt.title(f"Spearman ρ = {r_s:.2f}"); plt.show()
plt.figure(); sns.scatterplot(x=df1_mini["MeanGap"]*_NM, y=df1_mini["Tm"]); plt.xlabel("Mean gap (nm)"); plt.ylabel("Tm"); plt.title(f"Spearman ρ = {r_s:.2f}"); plt.show()


In [ ]:
from scipy.stats import linregress

In [ ]:
df1 = df1_mini
df1["Tm"] = pd.to_numeric(df1["Tm"], errors="coerce")

In [ ]:
df_median_7nt = df_median[df_median['NucleotideBonds'] == 7]
for bond in np.unique(df_median_7nt['Num_Bonds']):
    df1_mini = df_median_7nt[df_median_7nt['Num_Bonds'] == bond]
    xPlot = pd.to_numeric(df1_mini['MeanGap'], errors="coerce").to_numpy()
    yPlot = pd.to_numeric(df1_mini['Steepness'], errors="coerce").to_numpy()
    sns.regplot(x = xPlot,y=yPlot)
    plt.title("Number of Bonds: " + str(bond))
    plt.xlabel('Mean Gap Between Bonds')
    plt.ylabel('Steepness')
    plt.show()

In [ ]:
df_median_7nt = df_median[df_median['NucleotideBonds'] == 7]
for bond in np.unique(df_median_7nt['Num_Bonds']):
    df1_mini = df_median_7nt[df_median_7nt['Num_Bonds'] == bond]
    xPlot = pd.to_numeric(df1_mini['MeanGap'], errors="coerce").to_numpy()
    yPlot = pd.to_numeric(df1_mini['Entropy'], errors="coerce").to_numpy()
    sns.regplot(x = xPlot,y=yPlot)
    plt.title("Number of Bonds: " + str(bond))
    plt.xlabel('Mean Gap Between Bonds')
    plt.ylabel('Entropy')
    plt.show()

In [ ]:
df_median_7nt = df_median[df_median['NucleotideBonds'] == 7]
for bond in np.unique(df_median_7nt['Num_Bonds']):
    df1_mini = df_median_7nt[df_median_7nt['Num_Bonds'] == bond]
    xPlot = pd.to_numeric(df1_mini['Steepness'], errors="coerce").to_numpy()
    yPlot = pd.to_numeric(df1_mini['Entropy'], errors="coerce").to_numpy()
    sns.regplot(x = xPlot,y=yPlot)
    plt.title("Number of Bonds: " + str(bond))
    plt.xlabel('Mean Gap Between Bonds')
    plt.ylabel('Entropy')
    plt.show()

In [ ]:
df_median_7nt = df_median[df_median['NucleotideBonds'] == 7]
for bond in np.unique(df_median_7nt['Num_Bonds']):
    if(bond ==5):
        continue
    df1_mini = df_median_7nt[df_median_7nt['Num_Bonds'] == bond]
    xPlot = pd.to_numeric(df1_mini['MeanGap'], errors="coerce").to_numpy()
    yPlot = pd.to_numeric(df1_mini['Steepness'], errors="coerce").to_numpy()
    sns.regplot(x = xPlot,y=yPlot)
    plt.title("Number of Bonds: " + str(bond))
    plt.xlabel('Mean Gap Between Bonds')
    plt.ylabel('Boltzmann Curve Steepness')
    plt.tight_layout()
    #plt.savefig('../Tile_Tole_Figures/Steepness_Regplot.svg')
    plt.show()

In [ ]:
df_median_7nt = df_median[df_median['NucleotideBonds'] == 7]
for bond in np.unique(df_median_7nt['Num_Bonds']):
    if(bond == 5):
        continue
    df1_mini = df_median_7nt[df_median_7nt['Num_Bonds'] == bond]
    xPlot = pd.to_numeric(df1_mini['MeanGap_nm'], errors="coerce").to_numpy()
    yPlot = pd.to_numeric(df1_mini['Tm'], errors="coerce").to_numpy()
    sns.regplot(x = xPlot,y=yPlot,line_kws={"color": "red"})
    #plt.title("Number of Binding Sites: " + str(bond))
    plt.xlabel('Mean Gap Between Sites')
    plt.ylabel('Tm (°C)')
    #plt.suptitle('7nt bonds', y=0.9)
    plt.tight_layout()
    plt.savefig('../Tile_Tole_Figures/Tm_distance_Regplot.svg')
    plt.show()

In [ ]:
df_median_7nt = df_median[df_median['NucleotideBonds'] == 7]
for bond in np.unique(df_median_7nt['Num_Bonds']):
    if(bond == 5):
        continue
    df1_mini = df_median_7nt[df_median_7nt['Num_Bonds'] == bond]
    xPlot = pd.to_numeric(df1_mini['MeanGap_nm'], errors="coerce").to_numpy()
    yPlot = pd.to_numeric(df1_mini['Tm'], errors="coerce").to_numpy()
    sns.regplot(x = xPlot,y=yPlot,line_kws={"color": "red"})
    #plt.title("Number of Binding Sites: " + str(bond))
    plt.xlabel('Mean Gap Between Sites')
    plt.ylabel('Tm (°C)')
    #plt.suptitle('7nt bonds', y=0.9)
    plt.tight_layout()
    plt.savefig('../Tile_Tole_Figures/Tm_distance_Regplot.svg')
    plt.show()

# --- paper-figure export (clean 4-bond, 7nt panel) ---
_sub3b = df_median_7nt[df_median_7nt["Num_Bonds"] == 4].copy()
_x3b = pd.to_numeric(_sub3b["MeanGap_nm"], errors="coerce")
_y3b = pd.to_numeric(_sub3b["Tm"], errors="coerce")
plt.figure()
sns.regplot(x=_x3b, y=_y3b, line_kws={"color": "red"})
plt.xlabel("Mean Gap Between Sites (nm)"); plt.ylabel("Tm (°C)"); plt.tight_layout()
export_panel("Fig3b_Tm_vs_MeanGap_7nt_4bonds",
             data=_sub3b[["Samples","MeanGap_nm","Tm"]],
             note="Fig 3b. Tm vs mean bond gap for interfaces with 7nt bonds and 4 binding sites. Red line = linear regression.")
plt.close()

# --- repoint Fig3b to authoritative df_Tm_median ---
_m7 = df_Tm_median[(df_Tm_median["NucleotideBonds"]==7) & (df_Tm_median["Num_Bonds"]==4)].copy()
plt.figure()
sns.regplot(x=pd.to_numeric(_m7["MeanGap_nm"],errors="coerce"), y=pd.to_numeric(_m7["Tm"],errors="coerce"), line_kws={"color":"red"})
plt.xlabel("Mean Gap Between Sites (nm)"); plt.ylabel("Tm (°C)"); plt.tight_layout()
export_panel("Fig3b_Tm_vs_MeanGap_7nt_4bonds",
             data=_m7[["Code","MeanGap_nm","Tm","Tm_SE"]],
             note="Fig 3b. Tm vs mean bond gap (7nt, 4 sites). Source: authoritative Tm table (per-construct median).")
plt.close()

# --- Fig3b HIGH-reliability variant ---
_m7h = df_Tm_median_high[(df_Tm_median_high["NucleotideBonds"]==7) & (df_Tm_median_high["Num_Bonds"]==4)].copy()
plt.figure()
sns.regplot(x=pd.to_numeric(_m7h["MeanGap_nm"],errors="coerce"), y=pd.to_numeric(_m7h["Tm"],errors="coerce"), line_kws={"color":"red"})
plt.xlabel("Mean Gap Between Sites (nm)"); plt.ylabel("Tm (°C)"); plt.tight_layout()
export_panel_high("Fig3b_Tm_vs_MeanGap_7nt_4bonds", _m7h[["Code","MeanGap_nm","Tm","Tm_SE"]])
plt.close()

In [ ]:
_NM = 88.0/10
for bond in df1['NucleotideBonds'].value_counts().index:
    df1_mini = df1[df1['NucleotideBonds'] == bond]
    slope, intercept, r_value, p_value, std_err = linregress(df1_mini["MeanGap"], df1_mini["Tm"])
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
    sns.regplot(data=df1_mini, x="MeanGap", y="Tm", scatter_kws={"alpha":0.6}, line_kws={"color":"red"}, ax=axs[0])
    axs[0].set(xlabel='Mean gap (bond sites)', ylabel='Melting Temperature')
    sns.regplot(x=df1_mini["MeanGap"]*_NM, y=df1_mini["Tm"], scatter_kws={"alpha":0.6}, line_kws={"color":"red"}, ax=axs[1])
    axs[1].set(xlabel='Mean gap (nm)', ylabel='Melting Temperature')
    fig.suptitle(bond); plt.tight_layout(); plt.show()


In [ ]:
_NM = 88.0/10
fig, axs = plt.subplots(1, 2, figsize=(12, 4.6))
sns.boxplot(data=df1_mini, x='MeanGap', y='factor', ax=axs[0]); axs[0].set_xlabel('Mean gap (bond sites)'); axs[0].tick_params(axis='x', rotation=45)
_d = df1_mini.assign(MeanGap_nm=(df1_mini['MeanGap']*_NM).round(1))
sns.boxplot(data=_d, x='MeanGap_nm', y='factor', ax=axs[1]); axs[1].set_xlabel('Mean gap (nm)'); axs[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()


In [ ]:
df1_mini = df_median[df_median['Num_Bonds'] == 4]
sns.barplot(data = df1_mini, x = 'NucleotideBonds', y = 'Entropy')
#ax = sns.catplot(
#    data=df1_mini, x="Num_Bonds", y="Entropy", col="NucleotideBonds",
#    kind="bar", height=4, aspect=.6,palette='colorblind'
#)
plt.xlabel('Length of Bonds (nt)')
plt.tight_layout()
#plt.savefig('Entropy7vs9.eps', format='eps', dpi=1000)

In [ ]:
df1_mini = df1[df1['Num_Bonds'] == 5]
sns.barplot(data = df1_mini, x = 'NucleotideBonds', y = 'Entropy')
#ax = sns.catplot(
#    data=df1_mini, x="Num_Bonds", y="Entropy", col="NucleotideBonds",
#    kind="bar", height=4, aspect=.6,palette='colorblind'
#)
plt.xlabel('Length of Bonds (nt)')
plt.tight_layout()

In [ ]:
df1_mini = df1[df1['NucleotideBonds'] == 5]
sns.scatterplot(data = df1_mini, x = 'Predicted_deltaG', y ='Tm', hue='Num_Bonds',palette='colorblind')
#plt.ylim(0,0.7)
plt.title('5nt Long Bonds')

In [ ]:
df1_mini = df1[df1['NucleotideBonds'] == 7]
sns.scatterplot(data = df1_mini, x = 'Predicted_deltaG', y ='Tm', hue='Num_Bonds',palette='colorblind')
#plt.ylim(0,0.7)
plt.title('7nt Long Bonds')

In [ ]:
df1_mini = df1[df1['NucleotideBonds'] == 9]
sns.scatterplot(data = df1_mini, x = 'Predicted_deltaG', y ='Tm', hue='Num_Bonds',palette='colorblind')
#plt.ylim(0,0.7)
plt.title('9nt Long Bonds')

In [ ]:
df1_mini = df1[df1['NucleotideBonds'] == 9]
sns.scatterplot(data = df1_mini, x = 'factor', y ='Entropy', hue='Num_Bonds',palette='colorblind')
#plt.ylim(0,0.7)
plt.title('9nt Long Bonds')

In [ ]:
df1_mini = df1[df1['NucleotideBonds'] == 7]
df1_mini = df1_mini[df1_mini['factor'] >0]
sns.scatterplot(data = df1_mini, x = 'factor', y ='Entropy', hue='Num_Bonds',palette='colorblind')
#plt.ylim(0,0.7)
plt.xlim(0,1)
plt.title('7nt Long Bonds')

In [ ]:
df1_mini = df1[df1['NucleotideBonds'] == 5]
sns.scatterplot(data = df1_mini, x = 'factor', y ='Entropy', hue='Num_Bonds',palette='colorblind')
#plt.ylim(0,0.7)
plt.title('5nt Long Bonds')

In [ ]:
df1_mini = df1[df1['NucleotideBonds'] == 5]
sns.scatterplot(data = df1_mini, x = 'factor', y ='Enthalpy', hue='Num_Bonds',palette='colorblind')
#plt.ylim(0,0.7)
plt.title('5nt Long Bonds')

In [ ]:
df1_mini = df1[df1['NucleotideBonds'] == 7]
sns.scatterplot(data = df1_mini, x = 'factor', y ='Enthalpy', hue='Num_Bonds',palette='colorblind')
#plt.ylim(0,0.7)
plt.title('7nt Long Bonds')

In [ ]:
df1_mini = df1[df1['NucleotideBonds'] == 9]
sns.scatterplot(data = df1_mini, x = 'factor', y ='Enthalpy', hue='Num_Bonds',palette='colorblind')
#plt.ylim(0,0.7)
plt.title('9nt Long Bonds')

In [ ]:
df1_mini = df1[df1['NucleotideBonds'] == 9]
sns.scatterplot(data = df1_mini, x = 'factor', y ='Exp_DeltaG', hue='Num_Bonds',palette='colorblind')
#plt.ylim(0,0.7)
plt.title('9nt Long Bonds')

In [ ]:
df1_mini = df1[df1['NucleotideBonds'] == 9]
sns.scatterplot(data = df1_mini, x = 'Predicted_deltaG', y ='Exp_DeltaG', hue='Num_Bonds',palette='colorblind')
#plt.ylim(0,0.7)
plt.title('9nt Long Bonds')

In [ ]:
df1_mini = df1[df1['NucleotideBonds'] == 7]
sns.scatterplot(data = df1_mini, x = 'Predicted_deltaG', y ='Exp_DeltaG', hue='Num_Bonds',palette='colorblind')
plt.ylim(-25,-10)
plt.title('7nt Long Bonds')

In [ ]:
df1_mini = df1[df1['NucleotideBonds'] == 5]
sns.scatterplot(data = df1_mini, x = 'Predicted_deltaG', y ='Exp_DeltaG', hue='Num_Bonds',palette='colorblind')
plt.ylim(-25,-5)
plt.title('5nt Long Bonds')

In [ ]:
import numpy as np
import itertools
from scipy.special import logsumexp

R_kcal = 0.0019872041  # kcal/(mol K)

def _get_single_G(i, single_site_dict):
    # accept int keys or string keys
    return single_site_dict.get(i, single_site_dict.get(str(i), 0.0))

def _get_pair_G(i, j, config_dict):
    # accept either ordering
    i = str(i)
    j = str(j)
    if (i,j) in config_dict:
        return config_dict[(i,j)]
    if (j,i) in config_dict:
        return config_dict[(j,i)]
    return 0.0

def predict_deltaG_partition(seq, single_site_dict, config_dict, T_C=25.0, concentration_M=4.5E-9):
    """
    Compute ΔG_pred (kcal/mol) for the ensemble (all microstates compatible with seq).
    seq: string of '0'/'1' length L (e.g. '00000100001')
    single_site_dict: mapping index->ΔG_site (kcal/mol) (keys can be int or str)
    config_dict: mapping (i,j)->ΔG_conf (kcal/mol) for pair (i<->j)
                 (if pair missing assumed 0)
    T_C: temperature in Celsius used to evaluate beta and convert units
    Returns: ΔG_pred (kcal/mol), and optionally W (bound weight)
    """
    T_K = T_C + 273.15
    beta = 1.0 / (R_kcal * T_K)   # 1/(RT) in 1/(kcal/mol)
    ones = [i for i,ch in enumerate(seq) if ch == '1']
    if len(ones) == 0:
        # no possible bound microstates
        return np.nan, 0.0

    logw_list = []
    # enumerate all non-empty subsets S of 'ones'
    for r in range(1, len(ones)+1):
        for S in itertools.combinations(ones, r):
            # sum single-site ΔG
            G_S = sum(_get_single_G(i, single_site_dict) for i in S)
            # add pairwise ΔG_conf for every pair in S
            if len(S) >= 2:
                for i,j in itertools.combinations(S, 2):
                    G_S += _get_pair_G(i, j, config_dict)
            # log weight = -beta * G_S
            logw_list.append(-beta * G_S)

    # log-sum-exp -> log W
    logW = logsumexp(np.array(logw_list))
    W = np.exp(logW)
    # if W is zero (numerically) return very large positive DG
    if W <= 0 or not np.isfinite(logW):
        return np.nan, W
    
    K_eff = concentration_M * W
    #DG_at_C = - (1.0 / beta) * np.log(K_eff)
    # predicted ΔG for association (standard state): -RT ln W (kcal/mol)
    DG_pred = - (1.0 / beta) * logW +11.38  # note: 1/beta = R*T in kcal/mol units
    return float(DG_pred), float(W)

# Corrected van't Hoff thermodynamics & cooperativity analysis

This section recomputes the experimental thermodynamics (T$_m$, ΔG°, ΔS°, ΔH°) with the
issues below fixed, then asks whether the multivalent tile–tile bonds are **positively or
negatively cooperative**. It is self-contained (it only needs the setup cells at the top:
imports, the nearest-neighbour helpers, and `constant_2`).

**Corrections applied**
1. **Bimolecular equilibrium constant.** Two tiles associate as a non-self-complementary
   duplex $A+B\rightleftharpoons AB$, so $K=[AB]/([A][B])=2\theta/((1-\theta)^2 C_T)$ with
   $C_T$ = total strand concentration. The previous code dropped the factor of 2.
2. **Robust baselines.** The raw fluorescence is very noisy; keying the folded/unfolded
   levels off single extreme points (MinMax) corrupts $\theta$. We fit a two-state sigmoid
   through the noise and take $\theta$ from the *fitted* baselines (+ a light rolling median).
3. **Linear region of the van't Hoff plot.** $\ln K$ vs $1/T$ is straight only in the
   transition region, so the slope is fit over $\theta\in[0.15,0.85]$ and the curved plateau
   tails are excluded. Each fit is scored by $R^2$.
4. **Quality control & units.** Fits with the wrong sign (ΔH ≥ 0) or too few points are
   rejected. Units: **ΔH kcal/mol, ΔS cal/mol/K, ΔG kcal/mol, T$_m$ °C**, all consistent.
   $T_m=\Delta H/(\Delta S+R\ln(C_T/4))$ is self-consistent with the same $C_T$.

**Per-interface table `vh`** (one row per interface) now also records, for each interface:
- the **linear region around T$_m$** actually used for the van't Hoff slope (`lin_T_lo`, `lin_T_hi`, `npts`);
- **replicate standard deviations** `dH_sd`, `dS_sd`, `coop_dG_sd` (std of ΔΔG), plus `Tm_sd`, `dG37_sd`, `Steepness_sd`; and
- all previous **interface-characterization features**: `Num_Bonds`, `NucleotideBonds`, `MeanGap`, `VarGap`, `SkewAbs`, `SkewDi`, `Distances`, `Steepness`, `Positions`.


In [ ]:
# === Corrected van't Hoff fit ================================================
import numpy as np, pandas as pd, os
from scipy.optimize import curve_fit
#sns.set_style('whitegrid')

R_CAL = 1.987204          # gas constant, cal/(mol*K)
C_T   = 4.5e-9            # total strand concentration, M (both strands summed)
T37   = 37 + 273.15       # reference temperature, K

def _sigmoid(x, A1, A2, x0, dx):
    z = np.clip((x0 - x) / dx, -500, 500)
    return A1 + (A2 - A1) / (1 + np.exp(z))

def vant_hoff_fit(T, signal, theta_lo=0.15, theta_hi=0.85, smooth=7, min_pts=8):
    """Robust van't Hoff fit of one (noisy) melting curve.
    Tm is the two-state sigmoid midpoint (°C); Tm_vh is the van't Hoff Tm.
    Returns dict Tm(°C), dH(kcal/mol), dS(cal/mol/K), dG37(kcal/mol),
    dH_shape(kcal/mol cross-check), R2, npts, and lin_T_lo/lin_T_hi (the
    temperature bounds of the linear region around Tm) -- or None if unusable."""
    T = np.asarray(T, float); s0 = np.asarray(signal, float)
    ok = np.isfinite(T) & np.isfinite(s0); T, s0 = T[ok], s0[ok]
    if len(T) < 12:
        return None
    # 1) robust two-state fit THROUGH the read noise -> Tm, baselines, width
    try:
        p0 = [np.percentile(s0, 5), np.percentile(s0, 95), np.median(T), 3.0]
        (A1, A2, x0, dx), _ = curve_fit(_sigmoid, T, s0, p0=p0, maxfev=20000)
    except Exception:
        return None
    if not np.isfinite(dx) or dx <= 0 or not (T.min() - 5 < x0 < T.max() + 5):
        return None
    # 2) fraction folded from the FITTED baselines (not single noisy extrema)
    s = pd.Series(s0).rolling(smooth, center=True, min_periods=1).median().values
    theta = (A2 - s) / (A2 - A1) if A2 > A1 else (s - A2) / (A1 - A2)  # folded = low-T
    theta = np.clip(theta, 1e-4, 1 - 1e-4)
    Tk = T + 273.15
    m = (theta > theta_lo) & (theta < theta_hi)                       # 3) linear region
    if m.sum() < min_pts:
        return None
    x = 1.0 / Tk[m]
    y = np.log(2.0 * theta[m] / ((1.0 - theta[m])**2 * C_T))
    o = np.argsort(x); x, y = x[o], y[o]
    slope, intercept = np.polyfit(x, y, 1)
    yhat = slope * x + intercept
    ss_res = np.sum((y - yhat)**2); ss_tot = np.sum((y - y.mean())**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    dH = -slope * R_CAL / 1000.0        # kcal/mol   (folding -> < 0)
    dS =  intercept * R_CAL             # cal/mol/K  (folding -> < 0)
    if not np.isfinite(r2) or dH >= 0 or dS >= 0:                     # physical-sign QC
        return None
    Tm_vh = dH * 1000.0 / (dS + R_CAL * np.log(C_T / 4.0)) - 273.15   # van't Hoff Tm (cross-check)
    dG37 = dH - T37 * dS / 1000.0                                     # kcal/mol
    dH_shape = -1.5 * R_CAL * (x0 + 273.15)**2 / dx / 1000.0          # kcal/mol cross-check
    # linear region actually used for the van't Hoff slope (temperatures around Tm)
    return dict(Tm=x0, Tm_vh=Tm_vh, dx=dx, dH=dH, dS=dS, dG37=dG37,
                dH_shape=dH_shape, R2=r2, npts=int(m.sum()),
                lin_T_lo=float(T[m].min()), lin_T_hi=float(T[m].max()))

In [ ]:
# === Fit every melting curve; build the per-interface table ===================
# Per-bond energies from the sequence CSVs (reuses the top-of-notebook NN helpers).
_nt5 = pd.read_csv("nt5.csv", index_col=0); _nt6 = pd.read_csv("nt6.csv", index_col=0)
_nt7 = pd.read_csv("nt7.csv", index_col=0); _nt9 = pd.read_csv("nt9.csv", index_col=0)
_nt9_stack = pd.read_csv("nt9_stack.csv", index_col=0)
def _bond_table(df, L, stack=None):
    a = add_Pairs(add_nn_thermo_columns(df))
    if stack is not None:
        a['G'] = a['G'] + stack['Stacking']
    a['G_corrected'] = a.apply(lambda r: saltCorrection(r['G'], L), axis=1)
    return a
_TAB = {'5': _bond_table(_nt5, 5), '6': _bond_table(_nt6, 6),
        '7': _bond_table(_nt7, 7), '9': _bond_table(_nt9, 9, _nt9_stack)}
_TAB['X'] = _TAB['5']                       # central anchor: treat as a 5-nt bond

def additive_from_code(code, col):
    """Sum of independent sticky-end bonds. col='G_corrected'/'H' (kcal/mol) or 'S' (cal/mol/K)."""
    tot = 0.0
    for i, ch in enumerate(code):
        if   ch == '0':    continue
        elif ch in _TAB:   tot += _TAB[ch][col][i]
        elif ch in 'AB':   tot += constant_2[col][0 if ch == 'A' else 1]
        else:              return np.nan
    return tot

# interface-geometry features (same definitions as the earlier characterization cells)
def _spacing(positions):
    a = np.sort(np.asarray(positions)); g = np.diff(a)
    return (float(g.mean()), float(g.var(ddof=1))) if len(g) > 0 else (np.nan, np.nan)
def _skew_abs(positions): return int(sum(abs(p - 5) for p in positions))   # |pos - centre|
def _skew_di(positions):  return int(sum(p - 5 for p in positions))        # signed skew
def _bond_len(code):
    for ch in ('A', 'B', '6', '7', '9', '5'):
        if ch in code: return ch if ch in 'AB' else int(ch)
    return None

# ---- fit every replicate melting curve ----
_VALID = set('05679XAB'); _rows = []
for _f in os.listdir("Melting_Results/"):
    if   _f.endswith(".txt"): _d = pd.read_csv("Melting_Results/" + _f, sep="\t", encoding="unicode_escape")
    elif _f.endswith(".csv"): _d = pd.read_csv("Melting_Results/" + _f, encoding="unicode_escape")
    else: continue
    _d.columns.values[1] = "T"; _T = _d[_d.columns[1]].values
    for _c in _d.columns[2:]:
        _code = str(_c).split('.')[0]
        if len(_code) != 11 or not set(_code) <= _VALID:
            continue
        _r = vant_hoff_fit(_T, _d[_c].values)
        if _r:
            _r.update(code=_code, exp=_f); _rows.append(_r)
vh_reps = pd.DataFrame(_rows)
# per-replicate additive prediction & cooperativity (add_dG is constant per construct)
_addG = {c: additive_from_code(c, 'G_corrected') for c in vh_reps['code'].unique()}
_addH = {c: additive_from_code(c, 'H')           for c in vh_reps['code'].unique()}
vh_reps['add_dG']  = vh_reps['code'].map(_addG)
vh_reps['add_dH']  = vh_reps['code'].map(_addH)
vh_reps['coop_dG'] = vh_reps['dG37'] - vh_reps['add_dG']          # ΔΔG per replicate
print("replicate melting curves fit (physical signs):", len(vh_reps),
      "| median R^2 = %.3f" % vh_reps.R2.median())

# ---- keep good linear-region fits, aggregate to ONE ROW PER INTERFACE ----
# median = best estimate; *_sd = std across the (>=2) replicates of that interface.
_good = vh_reps[vh_reps.R2 >= 0.85]
vh = (_good.groupby("code").agg(
        n_rep=('Tm', 'size'),
        Tm=('Tm', 'median'),            Tm_sd=('Tm', 'std'),
        dH=('dH', 'median'),            dH_sd=('dH', 'std'),        # std of ΔH  (kcal/mol)
        dS=('dS', 'median'),            dS_sd=('dS', 'std'),        # std of ΔS  (cal/mol/K)
        dG37=('dG37', 'median'),        dG37_sd=('dG37', 'std'),
        coop_dG=('coop_dG', 'median'),  coop_dG_sd=('coop_dG', 'std'),   # std of ΔΔG (kcal/mol)
        dH_shape=('dH_shape', 'median'),
        Steepness=('dx', 'median'),     Steepness_sd=('dx', 'std'),
        R2=('R2', 'median'),
        lin_T_lo=('lin_T_lo', 'median'), lin_T_hi=('lin_T_hi', 'median'), npts=('npts', 'median'))
      .reset_index())
vh = vh[vh.n_rep >= 2].reset_index(drop=True)

# ---- additive prediction & cooperativity ----
vh['add_dG'] = vh['code'].map(_addG)
vh['add_dH'] = vh['code'].map(_addH)

# ---- interface-characterization features (skew, mean gap, distances, valency, ...) ----
vh['Positions']       = vh['code'].map(lambda c: [i for i, ch in enumerate(c) if ch != '0'])
vh['Num_Bonds']       = vh['Positions'].map(len)
vh['n_bonds']         = vh['Num_Bonds']                 # alias used by the plots below
vh['NucleotideBonds'] = vh['code'].map(_bond_len)
_sm = vh['Positions'].map(_spacing)
vh['MeanGap'] = _sm.map(lambda t: t[0])
vh['VarGap']  = _sm.map(lambda t: t[1])
vh['SkewAbs'] = vh['Positions'].map(_skew_abs)
vh['SkewDi']  = vh['Positions'].map(_skew_di)
vh['Distances'] = getDistances(list(vh['Positions']))   # mean pairwise bond spacing
# nanometre scale: 11 sites evenly span an 88 nm interface -> 88/(11-1) = 8.8 nm per site interval
NM_PER_SITE = 88.0 / (11 - 1)
vh['MeanGap_nm']   = vh['MeanGap']   * NM_PER_SITE
vh['Distances_nm'] = vh['Distances'] * NM_PER_SITE
vh['coop_dG_per_bond'] = vh['coop_dG'] / vh['Num_Bonds']

# tidy column order: id -> thermo (+std) -> cooperativity -> QC/region -> geometry
_order = ['code', 'n_rep', 'NucleotideBonds', 'Num_Bonds',
          'Tm', 'Tm_sd', 'dH', 'dH_sd', 'dS', 'dS_sd', 'dG37', 'dG37_sd',
          'add_dG', 'add_dH', 'coop_dG', 'coop_dG_sd', 'coop_dG_per_bond',
          'dH_shape', 'Steepness', 'Steepness_sd', 'R2', 'lin_T_lo', 'lin_T_hi', 'npts',
          'MeanGap', 'MeanGap_nm', 'VarGap', 'SkewAbs', 'SkewDi', 'Distances', 'Distances_nm', 'Positions']
vh = vh[[c for c in _order if c in vh.columns] + [c for c in vh.columns if c not in _order]]

print("interfaces with >=2 good replicates:", len(vh), "| columns:", len(vh.columns))
print("median replicate std:  ΔH %.1f kcal/mol | ΔS %.1f cal/mol/K | ΔΔG %.2f kcal/mol | Tm %.2f °C"
      % (vh.dH_sd.median(), vh.dS_sd.median(), vh.coop_dG_sd.median(), vh.Tm_sd.median()))
vh.head()


## 1. Van't Hoff diagnostics — is the fit in the linear region?

Left→right: the noisy melting curve with its two-state fit; the fraction folded with the
green **linear region** $\theta\in[0.15,0.85]$; and the van't Hoff plot, where the slope is
fit only to the green (linear) points and the curved plateau tails (open circles) are excluded.

In [ ]:
# representative van't Hoff linear-region diagnostic
_d = pd.read_csv("Melting_Results/Thermo10_28_2025_Formatted.csv", encoding="unicode_escape")
_d.columns.values[1] = "T"; _T = _d["T"].values.astype(float); _sig = _d[_d.columns[2]].values.astype(float)
(_A1,_A2,_x0,_dx), _ = curve_fit(_sigmoid,_T,_sig,
        p0=[np.percentile(_sig,5),np.percentile(_sig,95),np.median(_T),3],maxfev=20000)
_ss = pd.Series(_sig).rolling(7,center=True,min_periods=1).median().values
_th = np.clip((_A2-_ss)/(_A2-_A1),1e-4,1-1e-4); _Tk=_T+273.15
_m = (_th>0.15)&(_th<0.85)
_x = 1/_Tk; _y = np.log(2*_th/((1-_th)**2*C_T))
_sl,_ic = np.polyfit(_x[_m],_y[_m],1)
_r2 = 1-np.sum((_y[_m]-(_sl*_x[_m]+_ic))**2)/np.sum((_y[_m]-_y[_m].mean())**2)
fig,ax = plt.subplots(1,3,figsize=(18,5))
ax[0].plot(_T,_sig,'.',color="0.7",ms=5,label="raw signal")
_xs=np.linspace(_T.min(),_T.max(),300); ax[0].plot(_xs,_sigmoid(_xs,_A1,_A2,_x0,_dx),'r-',lw=2,label="two-state fit")
ax[0].set(xlabel="Temperature (°C)",ylabel="Fluorescence (a.u.)",title="Melting curve  %s"%_d.columns[2]); ax[0].legend()
ax[1].plot(_T,_th,'.',color="0.7",ms=5); ax[1].axhspan(0.15,0.85,color="tab:green",alpha=.12,label="linear region θ∈[0.15,0.85]")
ax[1].axhline(0.5,color="k",lw=.6); ax[1].set(xlabel="Temperature (°C)",ylabel="Fraction folded θ",title="Fraction folded"); ax[1].legend()
ax[2].plot(_x[~_m],_y[~_m],'o',mfc="none",mec="0.6",ms=6,label="excluded (curved tails)")
ax[2].plot(_x[_m],_y[_m],'o',color="tab:green",ms=6,label="linear region")
_xr=np.linspace(_x[_m].min(),_x[_m].max(),50); ax[2].plot(_xr,_sl*_xr+_ic,'r-',lw=2)
ax[2].set(xlabel="1/T (K$^{-1}$)",ylabel="ln K",title="van't Hoff plot (slope → ΔH)")
ax[2].text(.04,.05,"ΔH = %.0f kcal/mol\nR² = %.3f"%(-_sl*R_CAL/1000,_r2),transform=ax[2].transAxes,
           va="bottom",bbox=dict(fc="w",ec="0.5"))
ax[2].legend(loc="upper right"); plt.tight_layout(); plt.show()

# fit quality across all curves + independent ΔH cross-check
fig,ax = plt.subplots(1,2,figsize=(13,5))
ax[0].hist(vh_reps.R2,bins=30,color="tab:blue",alpha=.85); ax[0].axvline(0.85,color="r",ls="--",label="QC cut 0.85")
ax[0].set(xlabel="van't Hoff fit R²",ylabel="replicate curves",title="Linear-region fit quality"); ax[0].legend()
_sc=ax[1].scatter(vh_reps.dH_shape,vh_reps.dH,s=16,c=vh_reps.R2,cmap="viridis"); _lim=[-350,0]
ax[1].plot(_lim,_lim,'k--',lw=1); ax[1].set(xlim=_lim,ylim=_lim,
        xlabel="ΔH shape method (kcal/mol)",ylabel="ΔH van't Hoff slope (kcal/mol)",
        title="Two independent ΔH estimates\nconverge for clean (high-R²) fits")
plt.colorbar(_sc,ax=ax[1]).set_label("R²"); plt.tight_layout(); plt.show()

## 2. Cooperativity — positive or negative?

Compare each construct's **measured** ΔG°$_{37}$ to the **additive** prediction (the sum of
its independent sticky-end duplex energies). Points **above** the $y=x$ line are *sub-additive*
(the assembly is weaker than the sum of its parts) → **negative cooperativity**; points below
would be super-additive → positive cooperativity. The gap $\Delta\Delta G$ = measured − additive
is then plotted against the number of bonds (valency).

In [ ]:
fig,ax = plt.subplots(1,3,figsize=(18,5.4))
_sc=ax[0].scatter(vh.add_dG,vh.dG37,c=vh.n_bonds,cmap="plasma",s=70,ec="k",lw=.4)
ax[0].errorbar(vh.add_dG, vh.dG37, yerr=vh.dG37_sd, fmt="none", ecolor="0.55", elinewidth=.8, alpha=.6, zorder=0)  # replicate std of measured ΔG (≈ std of ΔΔG)
_lim=[min(vh.add_dG.min(),vh.dG37.min())-3, max(vh.add_dG.max(),vh.dG37.max())+3]
ax[0].plot(_lim,_lim,'k--',lw=1,label="additive (y=x)")
ax[0].set(xlim=_lim,ylim=_lim,xlabel="Additive ΔG°$_{37}$  Σ bonds (kcal/mol)",
          ylabel="Measured ΔG°$_{37}$ (kcal/mol)",title="Measured vs sum-of-parts")
ax[0].text(.5,.06,"above line → sub-additive\n(negative cooperativity)",transform=ax[0].transAxes,
           ha="center",color="darkred")
plt.colorbar(_sc,ax=ax[0]).set_label("# bonds"); ax[0].legend(loc="upper left")
sns.boxplot(data=vh,x="n_bonds",y="coop_dG",ax=ax[1],color="lightsteelblue")
sns.stripplot(data=vh,x="n_bonds",y="coop_dG",ax=ax[1],color="0.25",size=4,alpha=.6)
ax[1].axhline(0,color="r",lw=1.2)
ax[1].set(xlabel="# bonds (valency)",ylabel="ΔΔG = measured − additive (kcal/mol)",title="Cooperativity vs valency")
ax[1].text(.02,.97,"ΔΔG>0 → negative cooperativity",transform=ax[1].transAxes,va="top",color="darkred")
sns.boxplot(data=vh,x="n_bonds",y="coop_dG_per_bond",ax=ax[2],color="lightsteelblue")
sns.stripplot(data=vh,x="n_bonds",y="coop_dG_per_bond",ax=ax[2],color="0.25",size=4,alpha=.6)
ax[2].axhline(0,color="r",lw=1.2)
ax[2].set(xlabel="# bonds (valency)",ylabel="ΔΔG per bond (kcal/mol)",title="Per-bond cooperativity")
plt.tight_layout(); plt.show()

print("Median ΔΔG (measured − additive) by valency [kcal/mol]:")
print(vh.groupby("n_bonds")["coop_dG"].median().round(1).to_string())
print("\nAll constructs are sub-additive (ΔΔG > 0): the tile–tile assemblies are less stable")
print("than the sum of their isolated sticky-end duplexes, and the shortfall grows with valency")
print("-> NEGATIVE cooperativity / diminishing returns per added bond.")

## 3. Other relationships in the data

Enthalpy–entropy compensation (the near-linear ΔH–ΔS trend, with slope $\approx T_c$, is partly
imposed by the fixed-T$_m$ constraint), stability vs valency, and T$_m$ vs measured free energy.

In [ ]:
fig,ax = plt.subplots(1,3,figsize=(18,5.4))
_sl,_ic = np.polyfit(vh.dS,vh.dH,1)
ax[0].scatter(vh.dS,vh.dH,c=vh.n_bonds,cmap="plasma",s=60,ec="k",lw=.3)
_xs=np.linspace(vh.dS.min(),vh.dS.max(),50); ax[0].plot(_xs,_sl*_xs+_ic,'r-',lw=2)
ax[0].set(xlabel="ΔS° (cal/mol/K)",ylabel="ΔH° (kcal/mol)",
          title="Enthalpy–entropy compensation\n(T$_c$≈%.0f K; slope constrained by T$_m$)"%(_sl*1000))
sns.boxplot(data=vh,x="n_bonds",y="Tm",ax=ax[1],color="lightsteelblue")
sns.stripplot(data=vh,x="n_bonds",y="Tm",ax=ax[1],color="0.25",size=4,alpha=.6)
ax[1].set(xlabel="# bonds (valency)",ylabel="T$_m$ (°C)",title="Stability vs valency")
ax[2].scatter(vh.dG37,vh.Tm,c=vh.n_bonds,cmap="plasma",s=60,ec="k",lw=.3)
ax[2].set(xlabel="Measured ΔG°$_{37}$ (kcal/mol)",ylabel="T$_m$ (°C)",title="T$_m$ vs free energy")
plt.tight_layout(); plt.show()

## 4. Enthalpy and entropy vs interface geometry

ΔH° and ΔS° for each interface plotted against mean gap between bonds, valency (`Num_Bonds`), and bond length (`NucleotideBonds`). Longer bonds (9 nt) give the most negative ΔH°/ΔS°, as expected; error bars on the gap panels are the replicate std.

In [ ]:
# ΔH° and ΔS° as a function of interface geometry (mean gap, valency, bond length)
_vp   = vh.assign(NucleotideBonds=vh["NucleotideBonds"].astype(str))
_feats = [("MeanGap", "Mean gap between bonds (sites)"),
          ("Num_Bonds", "# bonds (valency)"),
          ("NucleotideBonds", "Bond length (nt)")]
_rows  = [("dH", "dH_sd", "ΔH° (kcal/mol)"), ("dS", "dS_sd", "ΔS° (cal/mol/K)")]
fig, ax = plt.subplots(2, 3, figsize=(18, 10))
for j, (col, xlab) in enumerate(_feats):
    for r, (yc, ysd, ylab) in enumerate(_rows):
        a = ax[r, j]
        if col == "MeanGap":                       # continuous -> scatter + trend + error bars
            sc = a.scatter(vh[col], vh[yc], c=vh["Num_Bonds"], cmap="viridis",
                           s=55, ec="k", lw=.3, zorder=3)
            a.errorbar(vh[col], vh[yc], yerr=vh[ysd], fmt="none", ecolor="0.6",
                       elinewidth=.8, alpha=.6, zorder=2)
            sns.regplot(data=vh, x=col, y=yc, ax=a, scatter=False, color="tab:red",
                        ci=None, line_kws=dict(lw=1.5))
            plt.colorbar(sc, ax=a, shrink=.85, pad=.02).set_label("# bonds")
            _axnm = a.secondary_xaxis('top', functions=(lambda v: v*8.8, lambda v: v/8.8))
            _axnm.set_xlabel("mean gap (nm)")   # 8.8 nm per site (88 nm / 10)
        else:                                       # discrete -> box + strip
            order = sorted(_vp[col].unique(), key=lambda v: (len(str(v)), str(v)))
            sns.boxplot(data=_vp, x=col, y=yc, ax=a, color="lightsteelblue", order=order, width=.6)
            sns.stripplot(data=_vp, x=col, y=yc, ax=a, color="0.25", size=4, alpha=.6, order=order)
        a.set_xlabel(xlab); a.set_ylabel(ylab)
        a.set_title(("ΔH°" if r == 0 else "ΔS°") + " vs " + xlab)
fig.suptitle("Enthalpy and entropy vs interface geometry", y=1.005)
plt.tight_layout(); plt.show()


## 5. Cooperativity — Tm-reproducibility filtered

Cooperativity for **all** interfaces, keeping only those with **exactly 3 replicates** whose melting temperatures agree within **std ≤ 1.5 °C** — the same criterion as `dfFilter = filterData_std(df1)`. This is a reproducibility cut on Tm only (no R² requirement), so it retains more interfaces than `vh` and the negative-cooperativity trend with valency is seen across the full 2–10-bond range.

In [ ]:
# === Cooperativity for ALL interfaces passing the Tm-reproducibility filter ===
# filterData_std keeps only interfaces with EXACTLY 3 replicates whose Tm values
# agree within std <= 1.5 °C  -- identical criterion to  dfFilter = filterData_std(df1).
# (This is a Tm-reproducibility cut only; it does NOT require the R^2>=0.85 van't Hoff
#  linearity used for `vh`, so more interfaces survive.)
_repsF = vh_reps.copy()
_repsF["Samples"] = _repsF["code"]
_keptF = filterData_std(_repsF)                       # replicate rows for passing interfaces
vhF = (_keptF.groupby("code").agg(
          n_rep=('Tm', 'size'), Tm=('Tm', 'median'), Tm_sd=('Tm', 'std'),
          dG37=('dG37', 'median'), coop_dG=('coop_dG', 'median'),
          coop_dG_sd=('coop_dG', 'std'), add_dG=('add_dG', 'first'))
       .reset_index())
vhF['n_bonds']          = vhF['code'].map(lambda c: sum(ch != '0' for ch in c))
vhF=vhF[vhF['n_bonds'] !=3]
vhF=vhF[vhF['n_bonds'] !=6]
vhF=vhF[vhF['n_bonds'] !=7]
vhF=vhF[~vhF['code'].str.contains('9|6|8')]
vhF['coop_dG_per_bond'] = vhF['coop_dG'] / vhF['n_bonds']
print("interfaces passing the Tm filter (3 reps, Tm std <= 1.5 °C):", len(vhF),
      " | for comparison, %d passed the stricter R^2>=0.85 filter used in `vh`" % len(vh))
fig, ax = plt.subplots(1, 3, figsize=(18, 5.4))
_sc = ax[0].scatter(vhF.add_dG, vhF.dG37, c=vhF.n_bonds, cmap="plasma", s=70, ec="k", lw=.4)
ax[0].errorbar(vhF.add_dG, vhF.dG37, yerr=vhF.coop_dG_sd, fmt="none", ecolor="0.55",
               elinewidth=.8, alpha=.6, zorder=0)
_lim = [min(vhF.add_dG.min(), vhF.dG37.min()) - 3, max(vhF.add_dG.max(), vhF.dG37.max()) + 3]
ax[0].plot(_lim, _lim, 'k--', lw=1, label="additive (y=x)")
ax[0].set(xlim=_lim, ylim=_lim, xlabel="Additive ΔG°$_{37}$  Σ bonds (kcal/mol)",
          ylabel="Measured ΔG°$_{37}$ (kcal/mol)", title="Measured vs sum-of-parts")
ax[0].text(.5, .06, "above line → sub-additive\n(negative cooperativity)",
           transform=ax[0].transAxes, ha="center", color="darkred")
plt.colorbar(_sc, ax=ax[0]).set_label("# bonds"); ax[0].legend(loc="upper left")
sns.boxplot(data=vhF, x="n_bonds", y="coop_dG", ax=ax[1], color="lightsteelblue")
sns.stripplot(data=vhF, x="n_bonds", y="coop_dG", ax=ax[1], color="0.25", size=4, alpha=.6)
ax[1].axhline(0, color="r", lw=1.2)
ax[1].set(xlabel="# bonds (valency)", ylabel="ΔΔG = measured − additive (kcal/mol)",
          title="Cooperativity vs valency")
sns.boxplot(data=vhF, x="n_bonds", y="coop_dG_per_bond", ax=ax[2], color="lightsteelblue")
sns.stripplot(data=vhF, x="n_bonds", y="coop_dG_per_bond", ax=ax[2], color="0.25", size=4, alpha=.6)
ax[2].axhline(0, color="r", lw=1.2)
ax[2].set(xlabel="# bonds (valency)", ylabel="ΔΔG per bond (kcal/mol)", title="Per-bond cooperativity")
fig.suptitle("Cooperativity — interfaces passing the Tm-reproducibility filter "
             "(3 reps, Tm std ≤ 1.5 °C; n=%d)" % len(vhF), y=1.03)
plt.tight_layout(); plt.show()

print("\nMedian ΔΔG (measured − additive) by valency [kcal/mol]:")
print(vhF.groupby("n_bonds")["coop_dG"].median().round(1).to_string())


In [ ]:
vhF = (_keptF.groupby("code").agg(
          n_rep=('Tm', 'size'), Tm=('Tm', 'median'), Tm_sd=('Tm', 'std'),
          dG37=('dG37', 'median'), coop_dG=('coop_dG', 'median'),
          coop_dG_sd=('coop_dG', 'std'), add_dG=('add_dG', 'first'))
       .reset_index())
vhF['n_bonds']          = vhF['code'].map(lambda c: sum(ch != '0' for ch in c))

vhF[vhF['n_bonds'] == 9]

## 6. Curves the new method rejects (that the old fit kept)

The switch to the new fitted-baseline fit drops ~290 melting curves the previous MinMax fit had accepted. Here every one of those curves is plotted (grey points) with the old sigmoid overlaid, coloured by why the new method rejects it: **bad-sigmoid** (no clean two-state midpoint in the raw signal) or **unphysical** (van't Hoff slope of the wrong sign). Most are genuinely noisy/near-monotonic traces with no real transition.

In [ ]:
# === Diagnostic: melting curves the OLD fit accepted but the NEW method rejects ===
# The pipeline uses the new fitted-baseline method everywhere; the previous MinMax fit
# is reconstructed HERE ONLY to show which curves the stricter new method drops.
from sklearn.preprocessing import MinMaxScaler
import math as _math

def _old_fit_minmax(T, sig):
    """PREVIOUS method: MinMax-normalize, fit sigmoid, van't Hoff slope over theta in [0.15,0.85]."""
    T = np.asarray(T, float); s = np.asarray(sig, float)
    ok = np.isfinite(T) & np.isfinite(s); T, s = T[ok], s[ok]
    if len(T) < 12: return None
    nd = MinMaxScaler().fit_transform((1 - s).reshape(-1, 1)).ravel()
    try: (A1, A2, x0, dx), _ = curve_fit(boltzmann, T, nd, p0=[0, 1, np.median(T), 2], maxfev=10000)
    except Exception: return None
    th = pd.Series(nd).rolling(7, center=True, min_periods=1).median().values
    ln = []; it = []
    for i in range(len(th)):
        v = th[i]
        if 0 < v < 1: ln.append(_math.log(2*v/((1-v)**2*C_T))); it.append(1/(T[i]+273.15))
    thin = th[(th > 0) & (th < 1)]; reg = (thin > 0.15) & (thin < 0.85)
    if reg.sum() < 8: return None
    m, b = np.polyfit(np.array(it)[reg], np.array(ln)[reg], 1)
    if not np.isfinite(-m) or -m*R_CAL/1000 >= 0: return None
    return dict(x0=x0, dx=dx)

def _new_fit_reason(T, sig):
    """NEW method; returns (result_or_None, reason) so rejections can be categorised."""
    T = np.asarray(T, float); s0 = np.asarray(sig, float)
    ok = np.isfinite(T) & np.isfinite(s0); T, s0 = T[ok], s0[ok]
    if len(T) < 12: return None, 'few-points'
    try:
        p0 = [np.percentile(s0, 5), np.percentile(s0, 95), np.median(T), 3.0]
        (A1, A2, x0, dx), _ = curve_fit(boltzmann, T, s0, p0=p0, maxfev=20000)
    except Exception: return None, 'sigmoid-fail'
    if not np.isfinite(dx) or dx <= 0 or not (T.min()-5 < x0 < T.max()+5): return None, 'bad-sigmoid'
    s = pd.Series(s0).rolling(7, center=True, min_periods=1).median().values
    th = (A2 - s)/(A2 - A1) if A2 > A1 else (s - A2)/(A1 - A2); th = np.clip(th, 1e-4, 1-1e-4)
    Tk = T + 273.15; reg = (th > 0.15) & (th < 0.85)
    if reg.sum() < 8: return None, 'few-region'
    x = 1/Tk[reg]; y = np.log(2*th[reg]/((1-th[reg])**2*C_T)); o = np.argsort(x)
    m, b = np.polyfit(x[o], y[o], 1)
    if -m*R_CAL/1000 >= 0 or b*R_CAL >= 0: return None, 'unphysical'
    return dict(x0=x0, dx=dx), 'ok'

# scan every melting curve
_lost = []; _cache = {}
def _getmelt(f):
    if f not in _cache:
        if f.endswith(".txt"): _cache[f] = pd.read_csv("Melting_Results/"+f, sep="\t", encoding="unicode_escape")
        else: _cache[f] = pd.read_csv("Melting_Results/"+f, encoding="unicode_escape")
        _cache[f].columns.values[1] = "T"
    return _cache[f]
for _f in os.listdir("Melting_Results/"):
    if not (_f.endswith(".txt") or _f.endswith(".csv")): continue
    _d = _getmelt(_f); _T = _d["T"].values
    for _ci in range(2, len(_d.columns)):
        if str(_d.columns[_ci]) in ("Buffer", "F Only"): continue
        _o = _old_fit_minmax(_T, _d.iloc[:, _ci].values)
        _n, _r = _new_fit_reason(_T, _d.iloc[:, _ci].values)
        if _o is not None and _n is None:
            _lost.append((_f, _ci, _r, _o))
_cnt = pd.Series([x[2] for x in _lost]).value_counts()
print("Curves accepted by the OLD fit but rejected by the NEW method:", len(_lost))
print("Reason the new method rejects them:", _cnt.to_dict())

# plot them all, coloured by rejection reason, with the old sigmoid overlaid
_col = {'bad-sigmoid':'tab:red', 'unphysical':'tab:orange', 'few-region':'tab:purple',
        'sigmoid-fail':'tab:brown', 'few-points':'0.5'}
_ordv = {'bad-sigmoid':0, 'unphysical':1, 'few-region':2, 'sigmoid-fail':3, 'few-points':4}
_lost = sorted(_lost, key=lambda x: (_ordv.get(x[2], 9), x[0]))
_n = len(_lost); _nc = 12; _nr = int(np.ceil(_n/_nc))
_fig, _ax = plt.subplots(_nr, _nc, figsize=(_nc*1.5, _nr*1.25)); _ax = np.atleast_2d(_ax)
for _k in range(_nr*_nc):
    _a = _ax[_k//_nc, _k % _nc]; _a.set_xticks([]); _a.set_yticks([])
    if _k >= _n: _a.axis("off"); continue
    _f, _ci, _r, _o = _lost[_k]; _dd = _getmelt(_f)
    _Tt = _dd["T"].values.astype(float); _ss = _dd.iloc[:, _ci].values.astype(float)
    _mm = np.isfinite(_Tt) & np.isfinite(_ss); _Tt, _ss = _Tt[_mm], _ss[_mm]
    _a.plot(_Tt, _ss, '.', color="0.6", ms=1.5)
    _xs = np.linspace(_Tt.min(), _Tt.max(), 200)
    _yb = boltzmann(_xs, 0, 1, _o['x0'], _o['dx']); _yb = _ss.min() + _yb*(_ss.max()-_ss.min())
    _a.plot(_xs, _yb, '-', color=_col.get(_r, 'k'), lw=1)
    for _sp in _a.spines.values(): _sp.set_edgecolor(_col.get(_r, 'k')); _sp.set_linewidth(1.2)
import matplotlib.patches as _mpatch
_hn = [_mpatch.Patch(color=_col[k], label="%s (%d)" % (k, v)) for k, v in _cnt.items()]
_fig.legend(handles=_hn, loc="upper center", ncol=len(_hn), frameon=False, bbox_to_anchor=(.5, 1.006))
_fig.suptitle("Melting curves accepted by the OLD (MinMax) fit but rejected by the NEW method (n=%d)" % _n,
              y=1.012)
plt.tight_layout(); plt.show()


## 7. Trends: thermodynamics vs interface geometry

Systematic scan of the five responses (**ΔH°, ΔS°, cooperativity ΔΔG, sigmoid steepness, T$_m$**)
against interface properties (number of bonds, bond length, distance/span between bonds,
clustering: adjacency runs & fill fraction, and off-centre skew). Reported as Spearman ρ, and —
because most geometry metrics are confounded with valency — **partial** correlations that control
for number and length of bonds.

**Trends found**

1. **Stability is set by bond *clustering* (strongest geometric effect).** For the *same* number and
   length of bonds, interfaces whose bonds are packed close together are markedly more stable:
   partial ρ(T$_m$, Span)=−0.47, ρ(T$_m$, Distance)=−0.42; within 5-bond interfaces ρ(T$_m$, Distance)=−0.64.
   Spreading the same bonds out lowers T$_m$ by several °C.
2. **Cooperativity is driven by valency, then by bond *position*.** ΔΔG (sub-additivity, i.e. negative
   cooperativity) rises steeply with the number of bonds (ρ=+0.69). Once valency and length are
   controlled, the clustering metrics (adjacency, gaps, fill) fall away, but **off-centre bonds stay
   more sub-additive** (partial ρ(ΔΔG, SkewAbs)=+0.35).
3. **ΔH°/ΔS° track bond length and compensate.** Longer bonds give more negative ΔH°/ΔS° (ρ≈−0.33);
   ΔH° and ΔS° are almost perfectly collinear (enthalpy–entropy compensation, T$_c$≈310 K ≈ the melting range).
4. **Transitions sharpen as bonds spread out.** Steepness dx shrinks (sharper melt) with larger
   span/distance (partial ρ≈−0.3); clustered interfaces melt more gradually.
5. **Valency also raises stability modestly** (ρ(T$_m$, #bonds)=+0.20) — much weaker than the clustering effect.


In [ ]:
# === Trend analysis: thermodynamics vs interface geometry ======================
# Build a per-interface table (Tm-reproducibility filter) with geometry + clustering
# features, then correlate the five responses against them (marginal + partial).
from scipy.stats import spearmanr, rankdata, pearsonr

def _iface_features(code):
    pos = np.array(sorted(i for i, ch in enumerate(code) if ch != '0'))
    gaps = np.diff(pos); n = len(pos); span = int(pos.max() - pos.min())
    bl = next((int(ch) for ch in ('6', '7', '9', '5') if ch in code), np.nan)
    runs = mx = 1
    for gth in gaps:
        runs = runs + 1 if gth == 1 else 1
        mx = max(mx, runs)
    dist = np.mean([abs(a - b) for i, a in enumerate(pos) for b in pos[i+1:]]) if n > 1 else np.nan
    return dict(Num_Bonds=n, BondLen=bl,
                MeanGap=gaps.mean() if len(gaps) else np.nan,
                VarGap=gaps.var(ddof=1) if len(gaps) > 1 else 0.0,
                Distances=dist, Span=span, FillFrac=n/(span+1),
                N_Adjacent=int(np.sum(gaps == 1)), MaxRun=mx,
                SkewAbs=int(sum(abs(p-5) for p in pos)), SkewDi=int(sum(p-5 for p in pos)),
                has_AB=('A' in code) or ('B' in code))

# aggregate the Tm-reproducibility-filtered replicates to one row per interface
_reps = vh_reps.copy(); _reps["Samples"] = _reps["code"]
_kept = filterData_std(_reps)
_agg = (_kept.groupby("code").agg(
            Enthalpy=('dH', 'median'), Entropy=('dS', 'median'), dG37=('dG37', 'median'),
            Steepness=('dx', 'median'), Tm=('Tm', 'median')).reset_index())
_agg['Coop'] = _agg['dG37'] - _agg['code'].map(lambda c: additive_from_code(c, 'G_corrected'))
TR = pd.concat([_agg, pd.DataFrame([_iface_features(c) for c in _agg['code']])], axis=1)
TR = TR[~TR['has_AB']].reset_index(drop=True)          # numeric bond-length subset
print("interfaces analysed:", len(TR))

_resp = ['Enthalpy', 'Entropy', 'Coop', 'Steepness', 'Tm']
_feat = ['Num_Bonds', 'BondLen', 'Distances', 'Span', 'MeanGap', 'VarGap',
         'FillFrac', 'N_Adjacent', 'MaxRun', 'SkewAbs']
_M = pd.DataFrame(index=_resp, columns=_feat, dtype=float)
for r in _resp:
    for f in _feat:
        x = pd.to_numeric(TR[f], errors='coerce'); y = pd.to_numeric(TR[r], errors='coerce')
        m = x.notna() & y.notna(); _M.loc[r, f] = spearmanr(x[m], y[m])[0]

def _partial(y, x, ctrl=('Num_Bonds', 'BondLen')):
    sub = TR[[y, x, *ctrl]].apply(pd.to_numeric, errors='coerce').dropna()
    Rk = {c: rankdata(sub[c]) for c in sub.columns}
    C = np.column_stack([np.ones(len(sub))] + [Rk[c] for c in ctrl])
    res = lambda v: v - C @ np.linalg.lstsq(C, v, rcond=None)[0]
    return pearsonr(res(Rk[x]), res(Rk[y]))

# ---- figure 1: correlation heatmap ----
fig, ax = plt.subplots(figsize=(13, 4.6))
sns.heatmap(_M.astype(float), annot=True, fmt="+.2f", cmap="RdBu_r", center=0, vmin=-.7, vmax=.7,
            cbar_kws=dict(label="Spearman ρ"), linewidths=.5, ax=ax)
ax.set(title="Thermodynamic responses vs interface geometry (Spearman ρ, n=%d)" % len(TR),
       xlabel="interface property")
plt.tight_layout(); plt.show()

# ---- figure 2: the notable trends ----
fig, ax = plt.subplots(2, 3, figsize=(19, 11))
sns.boxplot(data=TR, x="Num_Bonds", y="Coop", ax=ax[0, 0], color="lightsteelblue")
sns.stripplot(data=TR, x="Num_Bonds", y="Coop", ax=ax[0, 0], color="0.25", size=4, alpha=.6)
ax[0, 0].set(xlabel="# bonds (valency)", ylabel="Cooperativity ΔΔG (kcal/mol)",
             title="Cooperativity ↑ with valency  (ρ=+0.69)")
_pr = _partial('Coop', 'SkewAbs')
sns.regplot(data=TR, x="SkewAbs", y="Coop", ax=ax[0, 1],
            scatter_kws=dict(s=35, alpha=.6, color="tab:purple"), line_kws=dict(color="k"))
ax[0, 1].set(xlabel="SkewAbs  (Σ|pos−centre|)", ylabel="Cooperativity ΔΔG (kcal/mol)",
             title="Off-centre bonds → more sub-additive\npartial ρ=%+.2f (vs valency & length)" % _pr[0])
import matplotlib as _mpl
_norm = _mpl.colors.Normalize(TR.Num_Bonds.min(), TR.Num_Bonds.max())
sc = ax[0, 2].scatter(TR.Distances, TR.Tm, c=TR.Num_Bonds, cmap="viridis", norm=_norm, s=45, ec="k", lw=.3)
for nb in [4, 5]:
    s = TR[TR.Num_Bonds == nb]
    if len(s) > 6:
        m, b = np.polyfit(s.Distances, s.Tm, 1); xs = np.linspace(s.Distances.min(), s.Distances.max(), 20)
        ax[0, 2].plot(xs, m*xs+b, color=_mpl.cm.viridis(_norm(nb)), lw=3, label="%d bonds" % nb)
plt.colorbar(sc, ax=ax[0, 2]).set_label("# bonds"); ax[0, 2].legend(title="fit")
_pr2 = _partial('Tm', 'Span')
ax[0, 2].set(xlabel="Mean distance between bonds", ylabel="T$_m$ (°C)",
             title="Clustered bonds → more stable\npartial ρ(Tm,Span)=%+.2f" % _pr2[0])
s5 = TR[TR.Num_Bonds == 5]
sns.regplot(data=s5, x="Distances", y="Tm", ax=ax[1, 0],
            scatter_kws=dict(s=40, alpha=.7, color="tab:green"), line_kws=dict(color="k"))
ax[1, 0].set(xlabel="Mean distance between bonds", ylabel="T$_m$ (°C)",
             title="Within 5-bond interfaces (n=%d):  ρ=%+.2f" % (len(s5), spearmanr(s5.Distances, s5.Tm)[0]))
_DB = TR.dropna(subset=["BondLen"]); _ordb = sorted(_DB.BondLen.unique())
sns.boxplot(data=_DB, x="BondLen", y="Enthalpy", ax=ax[1, 1], color="lightsteelblue", order=_ordb)
sns.stripplot(data=_DB, x="BondLen", y="Enthalpy", ax=ax[1, 1], color="0.25", size=4, alpha=.6, order=_ordb)
ax[1, 1].set(xlabel="Bond length (nt)", ylabel="ΔH° (kcal/mol)", title="Longer bonds → more negative ΔH°  (ρ=−0.33)")
_sl, _ic = np.polyfit(TR.Entropy, TR.Enthalpy, 1)
ax[1, 2].scatter(TR.Entropy, TR.Enthalpy, c=TR.Num_Bonds, cmap="viridis", s=40, ec="k", lw=.3)
_xs = np.linspace(TR.Entropy.min(), TR.Entropy.max(), 30); ax[1, 2].plot(_xs, _sl*_xs+_ic, 'r-', lw=2)
ax[1, 2].set(xlabel="ΔS° (cal/mol/K)", ylabel="ΔH° (kcal/mol)",
             title="Enthalpy–entropy compensation  (T$_c$≈%d K)" % (_sl*1000))
plt.tight_layout(); plt.show()


## 8. Mechanism: strain vs transient (effective-concentration) occupancy

Comparing the **van't Hoff** (cooperative) ΔH° to the **additive/calorimetric** ΔH° (Σ of the
NN bond enthalpies) discriminates the two hypotheses.

**Result — the data favour transient / partial occupancy (low effective concentration), not strain.**
- The van't Hoff ΔH° is only ~**30 %** of the additive ΔH° (shape method ~37 %): the interface melts far from two-state.
- The van't Hoff cooperative ΔH° is essentially **constant (~−110 kcal/mol ≈ one helix) regardless of valency**, while the additive total scales steeply with the number of bonds (ρ=−0.62). The cooperative melting unit does **not grow** as bonds are added — they melt largely **independently**, roughly one at a time. Strain would keep the bonds melting together, so ΔH°$_{vH}$ would scale with valency; it does not.
- The huge enthalpy deficit is **compensated** by a matching entropy deficit (slope ≈ T$_m$), so |ΔΔH|/|ΔΔG| ≈ 8: the "missing" bonds cost lots of enthalpy+entropy but almost no free energy — exactly what unoccupied bonds *near their own melting point* would cost. Strain would be an **uncompensated** free-energy penalty (ΔΔG ≈ ΔΔH), which is not seen.
- Consistent with occupancy, **clustered** bonds (higher local effective concentration) are more stable and more occupied (Section 7), and cooperativity ΔΔG grows with valency as each added bond contributes less than a free duplex.

**Non-linearity:** ΔH° and ΔS° vs bond *separation* are weak and essentially **linear** (quadratic terms add ~1 % R²) — no strain-like ∝distance² curvature. The clear non-linear relation is ΔH°$_{vH}$/ΔH°$_{add}$ vs valency (falls ≈ as 1/N because the cooperative ΔH° is fixed while the total grows).

*Caveat:* part of the ΔH°–ΔS° compensation is imposed by single-concentration van't Hoff analysis; the load-bearing, non-artefactual evidence is the ~0.3 van't Hoff/calorimetric ratio and its constancy with valency.


In [ ]:
# === Mechanism: strain vs transient-occupancy (effective concentration) ========
# Compare the van't Hoff (cooperative) ΔH° to the additive/calorimetric ΔH° (Σ bonds),
# per interface, and test how the deficit behaves.
from scipy.stats import spearmanr
_reps = vh_reps.copy(); _reps["Samples"] = _reps["code"]
_kept = filterData_std(_reps)
MEC = (_kept.groupby("code").agg(dH=('dH','median'), dH_shape=('dH_shape','median'),
        dS=('dS','median'), dG37=('dG37','median'), Tm=('Tm','median'), R2=('R2','median')).reset_index())
def _sep(c):
    p = np.array(sorted(i for i,ch in enumerate(c) if ch!='0'))
    return len(p), (np.mean([abs(a-b) for i,a in enumerate(p) for b in p[i+1:]]) if len(p)>1 else np.nan)
MEC[['Num_Bonds','Distance']] = pd.DataFrame([_sep(c) for c in MEC['code']], index=MEC.index)
MEC['add_dH'] = MEC['code'].map(lambda c: additive_from_code(c,'H'))
MEC['add_dS'] = MEC['code'].map(lambda c: additive_from_code(c,'S'))
MEC['add_dG'] = MEC['code'].map(lambda c: additive_from_code(c,'G_corrected'))
MEC = MEC[~MEC['code'].str.contains('A|B')].reset_index(drop=True)
MEC['f_shape'] = MEC['dH_shape']/MEC['add_dH']
MEC['def_H'] = MEC['add_dH']-MEC['dH']; MEC['def_S'] = MEC['add_dS']-MEC['dS']; MEC['def_G'] = MEC['add_dG']-MEC['dG37']
H = MEC[MEC.R2>=0.85].copy()
print("van't Hoff ΔH° / additive ΔH°  (median) = %.2f   (shape method %.2f)"%((H.dH/H.add_dH).median(),(H.dH_shape/H.add_dH).median()))
print("|ΔΔH| / |ΔΔG| = %.1f   (large ⇒ deficit is compensated, costs little free energy)"%(abs(H.def_H.median())/abs(H.def_G.median())))
print("van't Hoff ΔH° ~ valency: rho=%+.2f (flat) | additive ΔH° ~ valency: rho=%+.2f (scales)"
      %(spearmanr(H.Num_Bonds,H.dH)[0], spearmanr(H.Num_Bonds,H.add_dH)[0]))

fig,ax=plt.subplots(2,3,figsize=(19,11))
sc=ax[0,0].scatter(H.add_dH,H.dH,c=H.Num_Bonds,cmap="viridis",s=55,ec="k",lw=.3)
lim=[H.add_dH.min()-20,0]; ax[0,0].plot(lim,lim,'k--',lw=1,label="two-state (y=x)")
ax[0,0].plot(lim,[0.29*v for v in lim],'r-',lw=1.5,label="f≈0.29")
ax[0,0].set(xlim=lim,ylim=[H.dH.min()-20,0],xlabel="Additive (calorimetric) ΔH° (kcal/mol)",
            ylabel="van't Hoff ΔH° (kcal/mol)",title="van't Hoff ΔH° ≈ 30% of additive → NOT two-state")
plt.colorbar(sc,ax=ax[0,0]).set_label("# bonds"); ax[0,0].legend(loc="lower right")
gb=H.groupby("Num_Bonds"); xs=sorted(H.Num_Bonds.unique())
ax[0,1].plot(xs,[gb.get_group(n).dH.median() for n in xs],'o-',color="tab:blue",ms=9,label="van't Hoff ΔH° (cooperative)")
ax[0,1].plot(xs,[gb.get_group(n).add_dH.median() for n in xs],'^-',color="tab:red",ms=9,label="additive ΔH° (Σ bonds)")
ax[0,1].axhspan(-135,-87,color="0.7",alpha=.3,label="one 7-nt helix")
ax[0,1].set(xlabel="# bonds (valency)",ylabel="ΔH° (kcal/mol)",title="Cooperative unit ≈ 1 helix, independent of valency")
ax[0,1].legend(); ax[0,1].set_xticks(xs)
sns.boxplot(data=H,x="Num_Bonds",y="f_shape",ax=ax[0,2],color="lightsteelblue")
sns.stripplot(data=H,x="Num_Bonds",y="f_shape",ax=ax[0,2],color="0.25",size=4,alpha=.6)
ax[0,2].set(xlabel="# bonds",ylabel="cooperative fraction ΔH°$_{vH}$/ΔH°$_{add}$",title="Cooperativity fraction falls with valency")
sl,ic=np.polyfit(H.def_S,H.def_H,1)
ax[1,0].scatter(H.def_S,H.def_H,c=H.Num_Bonds,cmap="viridis",s=50,ec="k",lw=.3)
xr=np.linspace(H.def_S.min(),H.def_S.max(),20); ax[1,0].plot(xr,sl*xr+ic,'r-',lw=2)
ax[1,0].set(xlabel="entropy deficit (cal/mol/K)",ylabel="enthalpy deficit (kcal/mol)",
            title="Missing ΔH° compensated by missing ΔS° (slope=%d K≈T$_m$)"%(sl*1000))
ax[1,0].text(.05,.9,"|ΔΔH|/|ΔΔG| = %.1f"%(abs(H.def_H.median())/abs(H.def_G.median())),transform=ax[1,0].transAxes,color="darkred")
for j,(yv,lab) in enumerate([('dH',"van't Hoff ΔH° (kcal/mol)"),('dS',"ΔS° (cal/mol/K)")]):
    a=ax[1,1+j]; d=MEC.dropna(subset=['Distance',yv]); xg=np.linspace(d.Distance.min(),d.Distance.max(),100)
    a.scatter(d.Distance,d[yv],s=22,alpha=.5,color="0.4")
    p1=np.polyfit(d.Distance,d[yv],1); p2=np.polyfit(d.Distance,d[yv],2)
    r1=1-np.sum((d[yv]-np.polyval(p1,d.Distance))**2)/np.sum((d[yv]-d[yv].mean())**2)
    r2=1-np.sum((d[yv]-np.polyval(p2,d.Distance))**2)/np.sum((d[yv]-d[yv].mean())**2)
    a.plot(xg,np.polyval(p1,xg),'b-',lw=2,label="linear R²=%.2f"%r1); a.plot(xg,np.polyval(p2,xg),'r--',lw=2,label="quadratic R²=%.2f"%r2)
    a.set(xlabel="mean distance between bonds",ylabel=lab,title="%s vs separation: weak & ~linear"%('ΔH°' if j==0 else 'ΔS°')); a.legend()
plt.tight_layout(); plt.show()


## 9. What actually sets stability: GC content and bond position

Two further findings, both robust after controlling for valency and length.

**GC content of the sticky ends is a dominant, hidden determinant.** Even though the additive NN
prediction already uses the sequences, the measured stability falls short of it in an
**AT-dependent** way: within the well-measured 7-nt set, the ΔG° deficit vs GC has partial
ρ=**−0.79** (p≈1e-32) and T$_m$ vs GC partial ρ=**+0.50**. AT-rich sticky ends realise far less
than their nominal NN stability — consistent with terminal fraying / poor occupancy of weak AT
ends, amplified in the tile context. It is **not** a measurement-window artefact (it holds after
dropping the low-T$_m$ constructs).

**Bond length is non-monotonic.** T$_m$ goes 5-nt (34°C) → 7-nt (42°C) ≈ 6-nt (43°C) → but
**9-nt collapses to 30°C** — the *lowest* of any length, despite the longest, predicted-strongest
bonds. The reason is composition: the 9-nt ends are the most AT-rich (GC≈0.37). Length and GC
trade off, and here AT-fraying wins. **Design takeaway: GC-rich sticky ends beat longer AT-rich ones.**

**Positional load-bearing map.** A per-position ridge fit of T$_m$ shows bonds at the **edges**
(helix positions 0,1,9,10) contribute *negatively* to stability while bonds flanking the central
anchor (7,8) contribute positively — again consistent with the effective-concentration picture:
edge bonds are farthest from their neighbours, least occupied, and add configurational cost
without much stabilisation.


In [ ]:
# === Sequence composition (GC) and bond position govern realised stability =====
from scipy.stats import spearmanr, rankdata, pearsonr
from sklearn.linear_model import Ridge

# per-interface table (Tm-reproducibility filter)
_reps = vh_reps.copy(); _reps["Samples"] = _reps["code"]
GCT = (filterData_std(_reps).groupby("code")
       .agg(Tm=('Tm','median'), dG37=('dG37','median')).reset_index())
GCT = GCT[~GCT['code'].str.contains('A|B')].reset_index(drop=True)
GCT['add_dG'] = GCT['code'].map(lambda c: additive_from_code(c,'G_corrected'))
GCT['def_G']  = GCT['add_dG'] - GCT['dG37']
GCT['Num_Bonds'] = GCT['code'].map(lambda c: sum(ch!='0' for ch in c))
GCT['BL'] = GCT['code'].map(lambda c: next((int(x) for x in '9765' if x in c), np.nan))

# GC fraction of each construct's sticky-end sequences (two seqs per occupied helix)
_seqcsv = {'5': pd.read_csv("nt5.csv", index_col=0)['Sequence'].astype(str).tolist(),
           '6': pd.read_csv("nt6.csv", index_col=0)['Sequence'].astype(str).tolist(),
           '7': pd.read_csv("nt7.csv", index_col=0)['Sequence'].astype(str).tolist(),
           '9': pd.read_csv("nt9.csv", index_col=0)['Sequence'].astype(str).tolist()}
def _gc(s): return sum(c in 'GC' for c in s)/len(s) if s and set(s) <= set('ACGT') else np.nan
def _construct_gc(code):
    v = []
    for i, ch in enumerate(code):
        tbl = _seqcsv['5'] if ch == 'X' else _seqcsv.get(ch)
        if tbl is None: continue
        for r in (2*i, 2*i+1):
            if r < len(tbl):
                g = _gc(tbl[r])
                if not np.isnan(g): v.append(g)
    return np.mean(v) if v else np.nan
GCT['GC'] = GCT['code'].map(_construct_gc)

def _partial(df, y, x, ctrl):
    sub = df[[y, x, *ctrl]].apply(pd.to_numeric, errors='coerce').dropna()
    Rk = {c: rankdata(sub[c]) for c in sub.columns}
    C = np.column_stack([np.ones(len(sub))] + [Rk[c] for c in ctrl])
    res = lambda v: v - C @ np.linalg.lstsq(C, v, rcond=None)[0]
    return pearsonr(res(Rk[x]), res(Rk[y]))
_s7 = GCT[GCT.BL == 7]
print("within 7-nt (n=%d):  def_G ~ GC | #bonds  partial rho=%+.2f | Tm ~ GC | #bonds  partial rho=%+.2f"
      % (len(_s7), _partial(_s7, 'def_G', 'GC', ['Num_Bonds'])[0], _partial(_s7, 'Tm', 'GC', ['Num_Bonds'])[0]))
print("Tm by bond length (median):", GCT.groupby('BL')['Tm'].median().round(1).to_dict(),
      " | GC by length:", GCT.groupby('BL')['GC'].median().round(2).to_dict())

fig, ax = plt.subplots(2, 2, figsize=(15, 11))
sc = ax[0, 0].scatter(GCT.GC, GCT.def_G, c=GCT.BL, cmap="autumn_r", s=55, ec="k", lw=.3)
xs = np.linspace(_s7.GC.min(), _s7.GC.max(), 20); m, b = np.polyfit(_s7.GC, _s7.def_G, 1)
ax[0, 0].plot(xs, m*xs+b, 'k-', lw=2, label="7-nt trend")
ax[0, 0].set(xlabel="GC fraction of sticky ends", ylabel="ΔG° deficit (additive − measured)",
             title="AT-rich bonds fall far short of NN prediction\ndef_G ~ GC | #bonds : partial ρ=−0.79 (within 7-nt)")
plt.colorbar(sc, ax=ax[0, 0]).set_label("bond length (nt)"); ax[0, 0].legend()
ax[0, 1].scatter(GCT.GC, GCT.Tm, c=GCT.BL, cmap="autumn_r", s=55, ec="k", lw=.3)
m, b = np.polyfit(_s7.GC, _s7.Tm, 1); ax[0, 1].plot(xs, m*xs+b, 'k-', lw=2, label="7-nt trend")
ax[0, 1].set(xlabel="GC fraction of sticky ends", ylabel="T$_m$ (°C)",
             title="GC-rich interfaces are more stable\nTm ~ GC | #bonds : partial ρ=+0.50"); ax[0, 1].legend()
_order = [5, 6, 7, 9]
sns.boxplot(data=GCT, x="BL", y="Tm", ax=ax[1, 0], color="lightsteelblue", order=_order)
sns.stripplot(data=GCT, x="BL", y="Tm", ax=ax[1, 0], color="0.25", size=4, alpha=.6, order=_order)
for k, bl in enumerate(_order):
    ax[1, 0].text(k, GCT[GCT.BL == bl].Tm.max()+0.7, "GC=%.2f" % GCT[GCT.BL == bl].GC.median(),
                  ha="center", color="darkred")
ax[1, 0].set(xlabel="bond length (nt)", ylabel="T$_m$ (°C)", title="Length is non-monotonic: 9-nt (most AT-rich) is weakest")
def _occ(code): return [1 if ch != '0' else 0 for ch in code]
_X = np.array([_occ(c) for c in GCT['code']]); _y = pd.to_numeric(GCT.Tm).values
_keep = [j for j in range(11) if _X[:, j].std() > 0]
_coef = Ridge(alpha=1.0).fit(_X[:, _keep], _y).coef_
_bar = np.full(11, np.nan)
for j, c in zip(_keep, _coef): _bar[j] = c
_cols = ['tab:red' if (not np.isnan(v) and v < 0) else 'tab:green' for v in _bar]
ax[1, 1].bar(range(11), np.nan_to_num(_bar), color=_cols, ec="k")
ax[1, 1].axhline(0, color="k", lw=.8); ax[1, 1].axvline(5, color="tab:blue", ls=":", lw=2)
ax[1, 1].set(xlabel="helix position (0–10, 5 = central anchor)", ylabel="per-position T$_m$ contribution (°C)",
             title="Load-bearing map: edge bonds destabilise, centre bonds stabilise"); ax[1, 1].set_xticks(range(11))
plt.tight_layout(); plt.show()


## 10. Does occupancy decay with bond distance? — Yes

Direct test of the occupancy prediction. Occupancy isn't measured, but under the
effective-concentration model a bond's *contribution* to stability is proportional to how much
it is engaged, so I decomposed the measured stabilities into **per-bond marginal contributions**
(multiple regression on the counts of bonds at each nearest-neighbour distance), with bootstrap CIs.

**Result — supported.** A bond's contribution decays monotonically with its distance to the
nearest other bond:
- **ΔG° penalty per bond:** ≈0 for clustered bonds (≤18 nm) → **+0.56 kcal/mol [95% CI +0.14,+0.96]**
  for isolated bonds (≥40 nm) — the CI excludes zero.
- Converting that penalty to occupancy, ΔΔG = −RT·ln θ, gives **relative occupancy
  θ(d)/θ(9 nm) = 1.00 → 0.98 → 0.86 → 0.47** as the nearest neighbour moves 9 → 18 → 26 → 40 nm away:
  the most isolated bonds spend **~half** as much time engaged.
- The per-bond **T$_m$** contribution shows the same decay (−1.0 → −2.8 °C).

So bonds do behave as if less occupied when they are more separated, with a length scale of
~20–25 nm (2–3 sites) beyond which engagement drops off. Caveat: the signal comes from the
minority of isolated bonds (49 of ~830 sit at ≥4 sites apart), so the *interface-average* effect
is diluted — which is why plain Tm-vs-mean-gap correlations look modest even though the per-bond
decay is clear.


In [ ]:
# === Testing the occupancy prediction: does engagement decay with bond distance? ==
# Decompose measured stability into per-bond marginal contributions, binned by each
# bond's distance to its nearest neighbour. If occupancy falls with distance, isolated
# bonds should contribute less (bigger ΔG penalty, less Tm).
from sklearn.linear_model import LinearRegression
np.random.seed(0); _RT = 0.616  # kcal/mol at 37 °C
_reps = vh_reps.copy(); _reps["Samples"] = _reps["code"]
OCC = (filterData_std(_reps).groupby("code")
       .agg(Tm=('Tm','median'), dG37=('dG37','median')).reset_index())
OCC = OCC[~OCC['code'].str.contains('A|B')].reset_index(drop=True)
OCC = OCC[~OCC['code'].str.contains('5|6|9')].reset_index(drop=True)
def _nnhist(code, nb=5):
    pos = [i for i, ch in enumerate(code) if ch != '0']; h = np.zeros(nb)
    for p in pos:
        d = min(abs(p-q) for q in pos if q != p); h[min(d-1, nb-1)] += 1
    return h
_H = np.array([_nnhist(c) for c in OCC['code']])
_dnm = np.array([1, 2, 3, 4, 5]) * (88.0/10); _labels = ['1', '2', '3', '4', '≥5']
def _boot(resp, nb=4):
    y = pd.to_numeric(OCC[resp]).values; n = len(y); C = []
    for _ in range(nb):
        idx = np.random.randint(0, n, n); C.append(LinearRegression().fit(_H[idx], y[idx]).coef_)
    C = np.array(C); return C.mean(0), np.percentile(C, 2.5, 0), np.percentile(C, 97.5, 0), C
_gmu, _glo, _ghi, _gC = _boot('dG37'); _tmu, _tlo, _thi, _tC = _boot('Tm')
_theta = np.exp(-(_gmu - _gmu[0]) / _RT)
_thetaB = np.exp(-(_gC - _gC[:, [0]]) / _RT)
_tl, _th = np.percentile(_thetaB, 2.5, 0), np.percentile(_thetaB, 97.5, 0)
print("bonds per bin (NN dist 1,2,3,>=4):", _H.sum(0).astype(int).tolist())
print("relative occupancy θ(d)/θ(9nm):", dict(zip(_labels, _theta.round(2))))

fig, ax = plt.subplots(1, 3, figsize=(19, 5.4))
ax[0].errorbar(_dnm, _gmu-_gmu[0], yerr=[(_gmu-_gmu[0])-(_glo-_gmu[0]), (_ghi-_gmu[0])-(_gmu-_gmu[0])],
               fmt='o-', ms=9, color="tab:red", capsize=4, lw=2); ax[0].axhline(0, color="k", lw=.7)
ax[0].set(xlabel="distance to nearest bond (nm)", ylabel="extra ΔG° penalty per bond (kcal/mol)",
          title="Isolated bonds pay a free-energy penalty")
ax[1].errorbar(_dnm, _theta, yerr=[_theta-_tl, _th-_theta], fmt='o-', ms=10, color="tab:blue", capsize=4, lw=2)
ax[1].axhline(1, color="0.6", ls="--")
ax[1].set(ylim=(0, 1.25), xlabel="distance to nearest bond (nm)", ylabel="relative occupancy θ(d)/θ(9 nm)",
          title="Occupancy decays with bond distance → SUPPORTED")
for d, t in zip(_dnm, _theta): ax[1].annotate("%.2f" % t, (d, t), textcoords="offset points", xytext=(6, -14))
ax[2].errorbar(_dnm, _tmu, yerr=[_tmu-_tlo, _thi-_tmu], fmt='o-', ms=9, color="tab:green", capsize=4, lw=2)
ax[2].set(xlabel="distance to nearest bond (nm)", ylabel="per-bond T$_m$ contribution (°C)",
          title="Isolated bonds contribute less T$_m$")
for a in ax:
    a.secondary_xaxis('top', functions=(lambda v: v/(88.0/10), lambda v: v*(88.0/10))).set_xlabel("distance (bond sites)")
plt.tight_layout(); plt.show()


In [ ]:
# === Testing the occupancy prediction: does engagement decay with bond distance? ==
# Decompose measured stability into per-bond marginal contributions, binned by each
# bond's distance to its nearest neighbour. If occupancy falls with distance, isolated
# bonds should contribute less (bigger ΔG penalty, less Tm).
from sklearn.linear_model import LinearRegression
np.random.seed(0); _RT = 0.616  # kcal/mol at 37 °C
_reps = vh_reps.copy(); _reps["Samples"] = _reps["code"]
OCC = (filterData_std(_reps).groupby("code")
       .agg(Tm=('Tm','median'), dG37=('dG37','median')).reset_index())
OCC = OCC[~OCC['code'].str.contains('A|B')].reset_index(drop=True)
def _nnhist(code, nb=5):
    pos = [i for i, ch in enumerate(code) if ch != '0']; h = np.zeros(nb)
    for p in pos:
        d = min(abs(p-q) for q in pos if q != p); h[min(d-1, nb-1)] += 1
    return h
_H = np.array([_nnhist(c) for c in OCC['code']])
_dnm = np.array([1, 2, 3, 4, 5]) * (88.0/10); _labels = ['1', '2', '3', '4', '≥5']
def _boot(resp, nb=3000):
    y = pd.to_numeric(OCC[resp]).values; n = len(y); C = []
    for _ in range(nb):
        idx = np.random.randint(0, n, n); C.append(LinearRegression().fit(_H[idx], y[idx]).coef_)
    C = np.array(C); return C.mean(0), np.percentile(C, 2.5, 0), np.percentile(C, 97.5, 0), C
_gmu, _glo, _ghi, _gC = _boot('dG37'); _tmu, _tlo, _thi, _tC = _boot('Tm')
_theta = np.exp(-(_gmu - _gmu[0]) / _RT)
_thetaB = np.exp(-(_gC - _gC[:, [0]]) / _RT)
_tl, _th = np.percentile(_thetaB, 2.5, 0), np.percentile(_thetaB, 97.5, 0)
print("bonds per bin (NN dist 1,2,3,>=4):", _H.sum(0).astype(int).tolist())
print("relative occupancy θ(d)/θ(9nm):", dict(zip(_labels, _theta.round(2))))

fig, ax = plt.subplots(1, 3, figsize=(19, 5.4))
ax[0].errorbar(_dnm, _gmu-_gmu[0], yerr=[(_gmu-_gmu[0])-(_glo-_gmu[0]), (_ghi-_gmu[0])-(_gmu-_gmu[0])],
               fmt='o-', ms=9, color="tab:red", capsize=4, lw=2); ax[0].axhline(0, color="k", lw=.7)
ax[0].set(xlabel="distance to nearest bond (nm)", ylabel="extra ΔG° penalty per bond (kcal/mol)",
          title="Isolated bonds pay a free-energy penalty")
ax[1].errorbar(_dnm, _theta, yerr=[_theta-_tl, _th-_theta], fmt='o-', ms=10, color="tab:blue", capsize=4, lw=2)
ax[1].axhline(1, color="0.6", ls="--")
ax[1].set(ylim=(0, 1.25), xlabel="distance to nearest bond (nm)", ylabel="relative occupancy θ(d)/θ(9 nm)",
          title="Occupancy decays with bond distance → SUPPORTED")
for d, t in zip(_dnm, _theta): ax[1].annotate("%.2f" % t, (d, t), textcoords="offset points", xytext=(6, -14))
ax[2].errorbar(_dnm, _tmu, yerr=[_tmu-_tlo, _thi-_tmu], fmt='o-', ms=9, color="tab:green", capsize=4, lw=2)
ax[2].set(xlabel="distance to nearest bond (nm)", ylabel="per-bond T$_m$ contribution (°C)",
          title="Isolated bonds contribute less T$_m$")
for a in ax:
    a.secondary_xaxis('top', functions=(lambda v: v/(88.0/10), lambda v: v*(88.0/10))).set_xlabel("distance (bond sites)")
plt.tight_layout(); plt.show()


### 10b. Enthalpy and entropy per bond vs distance

Same decomposition applied to ΔH° and ΔS°. **Both the enthalpic contribution and the entropic
penalty per bond decay with distance, and they track each other almost exactly** (ΔΔH° ≈ T·ΔΔS°):
relative to a clustered bond, an isolated bond (~40 nm) adds ≈16 kcal/mol *less* favourable
enthalpy **and** ≈16 kcal/mol *less* entropic cost, leaving only ≈0.5 kcal/mol of net
destabilisation. That is the fingerprint of fractional occupancy: a bond engaged a fraction θ of
the time contributes θ·(ΔH°, ΔS°), so lowering θ with distance scales enthalpy and entropy down
together in the fixed ratio ΔH°/ΔS° ≈ T$_m$. (Part of this compensation is also imposed by the
single-concentration van't Hoff fit, so the load-bearing point is that **each of ΔH° and ΔS°
individually grows with distance**, both significant at ≥40 nm.)


In [ ]:
# --- same per-bond decomposition applied to ΔH° and ΔS° (are they distance-dependent?) ---
np.random.seed(0); _T = 310.15
_reps2 = vh_reps.copy(); _reps2["Samples"] = _reps2["code"]
OCC2 = (filterData_std(_reps2).groupby("code")
        .agg(dG37=('dG37','median'), dH=('dH','median'), dS=('dS','median')).reset_index())
OCC2 = OCC2[~OCC2['code'].str.contains('A|B')].reset_index(drop=True)
_H2 = np.array([_nnhist(c) for c in OCC2['code']])      # reuse _nnhist from the cell above
_dnm2 = np.array([1, 2, 3, 4, 5]) * (88.0/10)
def _bootC(resp, nb=3000):
    y = pd.to_numeric(OCC2[resp]).values; n = len(y)
    return np.array([LinearRegression().fit(_H2[idx], y[idx]).coef_
                     for idx in [np.random.randint(0, n, n) for _ in range(nb)]])
_CH = _bootC('dH'); _CS = _bootC('dS')*_T/1000.0; _CG = _bootC('dG37')   # kcal/mol (T·ΔS)
def _rel(C):
    R = C - C[:, [0]]; return R.mean(0), np.percentile(R, 2.5, 0), np.percentile(R, 97.5, 0)
_hM,_hL,_hU = _rel(_CH); _sM,_sL,_sU = _rel(_CS); _gM,_gL,_gU = _rel(_CG)
print("per-bond change vs clustered at 40 nm:  ΔΔH=%.1f  T·ΔΔS=%.1f  net ΔΔG=%.2f kcal/mol"%(_hM[-1],_sM[-1],_gM[-1]))

fig, ax = plt.subplots(1, 3, figsize=(19, 5.4))
ax[0].errorbar(_dnm2,_hM,yerr=[_hM-_hL,_hU-_hM],fmt='o-',ms=9,color="tab:red",capsize=4,lw=2); ax[0].axhline(0,color="k",lw=.6)
ax[0].set(xlabel="distance to nearest bond (nm)",ylabel="ΔΔH° per bond vs clustered (kcal/mol)",
          title="Enthalpic contribution decays with distance")
ax[1].errorbar(_dnm2,_sM,yerr=[_sM-_sL,_sU-_sM],fmt='o-',ms=9,color="tab:purple",capsize=4,lw=2); ax[1].axhline(0,color="k",lw=.6)
ax[1].set(xlabel="distance to nearest bond (nm)",ylabel="T·ΔΔS° per bond vs clustered (kcal/mol)",
          title="Entropic penalty decays in lockstep")
ax[2].errorbar(_dnm2,_hM,yerr=[_hM-_hL,_hU-_hM],fmt='o-',color="tab:red",label="ΔΔH°",capsize=3,lw=2)
ax[2].errorbar(_dnm2,_sM,yerr=[_sM-_sL,_sU-_sM],fmt='s--',color="tab:purple",label="T·ΔΔS°",capsize=3,lw=2)
ax[2].errorbar(_dnm2,_gM,yerr=[_gM-_gL,_gU-_gM],fmt='^-',color="k",label="ΔΔG°=ΔΔH°−T·ΔΔS°",capsize=3,lw=2,ms=8)
ax[2].axhline(0,color="0.5",lw=.6)
ax[2].set(xlabel="distance to nearest bond (nm)",ylabel="per-bond term vs clustered (kcal/mol)",
          title="ΔH° and ΔS° compensate → small net ΔG°"); ax[2].legend()
for a in ax:
    a.secondary_xaxis('top', functions=(lambda v: v/(88.0/10), lambda v: v*(88.0/10))).set_xlabel("distance (bond sites)")
plt.tight_layout(); plt.show()


### 10c. Is the bootstrap inference sound? (clustering / experiment structure)

The bootstrap in 10 resamples **constructs** as if independent, but each construct's 3 replicates
sit on **one plate** and constructs share **9 experiments** — so constructs are clustered. Two checks:

- A cluster bootstrap that resamples **experiments** widens the ≥40 nm CI to include zero — but that
  is over-conservative, because with only 9 plates the between-plate ΔG scatter dominates.
- The correct fix is to **control for plate** (experiment fixed effects) and use **cluster-robust
  SEs**. Doing so removes the between-plate offsets and isolates the *within-plate* distance effect,
  which is **significant and monotonic** (per-bond ΔG penalty +0.22 → +0.82 kcal/mol, every bin's
  CI excludes zero).

So the naive construct-level bootstrap was **under-justified** (it ignored clustering; it happened
to match the analytic homoskedastic SE), but the underlying occupancy-vs-distance trend **survives
— and sharpens — under the appropriate design-aware inference.**


In [ ]:
# --- Is the inference sound? The bootstrap above resamples CONSTRUCTS, treating them as
#     independent; but each construct's 3 replicates live on ONE plate, and constructs share
#     plates (9 experiments). Proper inference controls for plate offsets (experiment fixed
#     effects) and uses cluster-robust standard errors clustered by experiment. ---
_R2 = vh_reps.copy(); _R2["Samples"] = _R2["code"]
_AA = (filterData_std(_R2).groupby("code").agg(dG37=('dG37','median'), exp=('exp','first')).reset_index())
_AA = _AA[~_AA['code'].str.contains('A|B')].reset_index(drop=True)
_HH = np.array([_nnhist(c) for c in _AA['code']]); _yy = pd.to_numeric(_AA['dG37']).values
_E = pd.get_dummies(_AA['exp'], drop_first=True).values.astype(float)
def _ols_cluster(X, y, grp):
    beta = np.linalg.lstsq(X, y, rcond=None)[0]; r = y - X @ beta; XtXi = np.linalg.inv(X.T @ X)
    meat = sum(np.outer(X[grp == g].T @ r[grp == g], X[grp == g].T @ r[grp == g]) for g in np.unique(grp))
    G = len(np.unique(grp)); n, k = X.shape
    cov = (G/(G-1))*((n-1)/(n-k)) * XtXi @ meat @ XtXi
    return beta, np.sqrt(np.diag(cov))
_lab = ['d1', 'd2', 'd3', 'd4', 'd≥5']
for _title, _X in [("distance bins only (cluster-robust SE)", np.column_stack([np.ones(len(_AA)), _HH])),
                   ("distance bins + EXPERIMENT fixed effects (cluster-robust SE)", np.column_stack([np.ones(len(_AA)), _HH, _E]))]:
    _b, _se = _ols_cluster(_X, _yy, _AA['exp'].values)
    print("\n" + _title + ":")
    for _k, _L in enumerate(_lab):
        print("   %-4s per-bond ΔG penalty = %+.2f  [%+.2f, %+.2f] kcal/mol"
              % (_L, _b[1+_k], _b[1+_k]-1.96*_se[1+_k], _b[1+_k]+1.96*_se[1+_k]))
print("\n-> Without controlling for plate, between-plate ΔG scatter inflates the SE and the effect")
print("   looks non-significant. Removing plate offsets isolates the WITHIN-plate distance effect,")
print("   which is significant and monotonic -> the occupancy-vs-distance trend holds under proper inference.")


In [ ]:
# === Same per-bond decomposition, ANALYTIC inference (no bootstrap) for ΔH°, ΔS°, ΔG° ===
# Ordinary least squares gives the coefficients in closed form; their standard errors come
# straight from the covariance matrix σ²(XᵀX)⁻¹ (classical) or a cluster-robust sandwich
# (clustered by experiment). 95% CI = β ± t·SE.  No resampling.
from scipy.stats import t as _tdist
_RA = vh_reps.copy(); _RA["Samples"] = _RA["code"]
_AN = (filterData_std(_RA).groupby("code")
       .agg(dH=('dH','median'), dS=('dS','median'), dG37=('dG37','median'), exp=('exp','first')).reset_index())
_AN = _AN[~_AN['code'].str.contains('A|B')].reset_index(drop=True)
_AN = _AN[~_AN['code'].str.contains('6|9')].reset_index(drop=True)
_AN = _AN[~_AN['code'].str.contains('55')].reset_index(drop=True)
_HA = np.array([_nnhist(c) for c in _AN['code']])          # reuse _nnhist from Section 10
_Xa = np.column_stack([np.ones(len(_AN)), _HA]); _lab = ['d1', 'd2', 'd3', 'd4', 'd≥5']; _dnm = np.array([1, 2, 3, 4, 5])*(88.0/10)
#_Xa = np.column_stack([np.ones(len(_AN)), _HA]); _lab = ['d1', 'd2']; _dnm = np.array([1,2])*(88.0/10)
def _ols(X, y, cluster=None):
    beta = np.linalg.lstsq(X, y, rcond=None)[0]; r = y - X@beta; XtXi = np.linalg.inv(X.T@X); n, k = X.shape
    if cluster is None:
        cov = (r@r/(n-k))*XtXi; df = n-k
    else:
        meat = sum(np.outer(X[cluster==c].T@r[cluster==c], X[cluster==c].T@r[cluster==c]) for c in np.unique(cluster))
        G = len(np.unique(cluster)); cov = (G/(G-1))*((n-1)/(n-k))*XtXi@meat@XtXi; df = G-1
    return beta, np.sqrt(np.diag(cov)), df

_specs = [('dH', "ΔH° (kcal/mol)", 'tab:red'), ('dS', "ΔS° (cal/mol/K)", 'tab:purple'), ('dG37', "ΔG° (kcal/mol)", 'k')]
fig, ax = plt.subplots(1, 3, figsize=(19, 5.4))
for a, (resp, ylab, col) in zip(ax, _specs):
    y = pd.to_numeric(_AN[resp]).values
    bC, seC, dfC = _ols(_Xa, y);                       tcC = _tdist.ppf(0.975, dfC)   # classical
    bR, seR, dfR = _ols(_Xa, y, _AN['exp'].values);    tcR = _tdist.ppf(0.975, dfR)   # cluster-robust
    print(f"\n{resp} per-bond contribution by NN distance (analytic 95% CI):")
    for k, L in enumerate(_lab):
        print("   %-4s = %+.2f  classical[%+.2f,%+.2f]  cluster-robust[%+.2f,%+.2f]"
              % (L, bC[1+k], bC[1+k]-tcC*seC[1+k], bC[1+k]+tcC*seC[1+k], bR[1+k]-tcR*seR[1+k], bR[1+k]+tcR*seR[1+k]))
    a.errorbar(_dnm, bC[1:], yerr=tcC*seC[1:], fmt='o-', ms=9, color=col, capsize=4, lw=2, label="classical 95% CI")
    a.errorbar(_dnm+0.6, bR[1:], yerr=tcR*seR[1:], fmt='s', ms=7, color=col, alpha=.45, capsize=4, label="cluster-robust 95% CI")
    a.axhline(0, color="0.6", lw=.7)
    a.set(xlabel="distance to nearest bond (nm)", ylabel="per-bond contribution — "+ylab,
          title=f"{ylab.split(' ')[0]} contribution vs distance (OLS, analytic)")
    a.legend(); a.secondary_xaxis('top', functions=(lambda v: v/(88.0/10), lambda v: v*(88.0/10))).set_xlabel("distance (bond sites)")
plt.tight_layout(); plt.show()


### 10e. Occupancy prediction, tested without bootstrap — SUPPORTED

The occupancy curve θ(d)/θ(9 nm) = exp(−(β_d−β_d1)/RT) computed from closed-form OLS: the
contrast β_d−β_d1 and its SE come straight from the coefficient covariance matrix, and the CI is
mapped through the monotone exp() — no resampling.

- **Classical OLS:** θ = 1.00, 0.97, 0.86, **0.47** at 9/18/26/40 nm; the 40 nm bin's CI excludes 1.
  (Matches the bootstrap, as expected.)
- **Plate fixed-effects + cluster-robust** (the design-aware model): θ = 1.00, **0.88, 0.75, 0.37** —
  a monotonic decay with **every** bin significantly below 1.
- **Single-parameter continuous trend test** (ΔG° ~ #bonds + Σ bond-distance, plate-FE + cluster-robust):
  each unit of bond isolation adds **+0.135 kcal/mol**, **p = 0.002**.

So the occupancy prediction holds under purely analytic inference; the plain cluster-robust-without-plate
model is the only one that fails to reach significance, and that is its known over-conservatism (9 plates).


In [ ]:
# === Occupancy prediction tested WITHOUT bootstrap (analytic OLS) ===============
# θ(d)/θ(9nm) = exp(-(β_d - β_d1)/RT).  β and its covariance come in closed form from OLS;
# the contrast (β_d - β_d1) and its SE come from the covariance matrix; the CI is mapped
# through the monotone exp(). Shown for the appropriate plate-fixed-effects + cluster-robust
# model, with classical CIs for comparison.  Plus a single-parameter continuous trend test.
from scipy.stats import t as _tdist
_RO = vh_reps.copy(); _RO["Samples"] = _RO["code"]
_AO = (filterData_std(_RO).groupby("code").agg(dG37=('dG37','median'), exp=('exp','first')).reset_index())
_AO = _AO[~_AO['code'].str.contains('A|B')].reset_index(drop=True)
_AO = _AO[~_AO['code'].str.contains('6|8|9')].reset_index(drop=True)
_AO = _AO[~_AO['code'].str.contains('55')].reset_index(drop=True)
#getNum_Bonds_Positions(_AN['code'])
_AO['NumBonds'] =getNum_Bonds_Positions(_AO['code'])[0]
_AO=_AO[_AO['NumBonds'] == 4]
_RT = 0.616; _NM = 88.0/10; _lab = ['d1', 'd2', 'd3', 'd4', 'd≥5']; _dnm = np.array([1, 2, 3, 4, 5])*_NM
def _pbd(code):
    pos=[i for i,ch in enumerate(code) if ch!='0']; return [min(abs(p-q) for q in pos if q!=p) for p in pos]
_H = np.array([[sum(1 for d in _pbd(c) if min(d,5)==b) for b in [1,2,3,4,5]] for c in _AO['code']])
_y = pd.to_numeric(_AO['dG37']).values
_E = pd.get_dummies(_AO['exp'], drop_first=True).values.astype(float)
def _olscov(X, y, cluster=None):
    beta = np.linalg.lstsq(X, y, rcond=None)[0]; r = y - X@beta; XtXi = np.linalg.inv(X.T@X); n, k = X.shape
    if cluster is None: cov = (r@r/(n-k))*XtXi; df = n-k
    else:
        meat = sum(np.outer(X[cluster==c].T@r[cluster==c], X[cluster==c].T@r[cluster==c]) for c in np.unique(cluster))
        G = len(np.unique(cluster)); cov = (G/(G-1))*((n-1)/(n-k))*XtXi@meat@XtXi; df = G-1
    return beta, cov, df

def _occ(X, cluster):
    beta, cov, df = _olscov(X, _y, cluster); tc = _tdist.ppf(0.975, df); th=[]; lo=[]; hi=[]
    for j in range(_H.shape[1]):
        c = np.zeros(X.shape[1]); c[1+j] = 1; c[1] -= 1
        dG = c@beta; se = np.sqrt(c@cov@c)
        th.append(np.exp(-dG/_RT)); lo.append(np.exp(-(dG+tc*se)/_RT)); hi.append(np.exp(-(dG-tc*se)/_RT))
    return np.array(th), np.array(lo), np.array(hi)
_thC,_loC,_hiC = _occ(np.column_stack([np.ones(len(_AO)),_H]), None)                       # classical
_thF,_loF,_hiF = _occ(np.column_stack([np.ones(len(_AO)),_H,_E]), _AO['exp'].values)        # plate-FE + cluster

# single-parameter continuous trend test (plate-FE + cluster-robust)
_nb=np.array([len(_pbd(c)) for c in _AO['code']]); _sd=np.array([sum(_pbd(c)) for c in _AO['code']])
_Xt=np.column_stack([np.ones(len(_AO)),_nb,_sd]); _b,_cov,_df=_olscov(_Xt,_y,_AO['exp'].values)
_se=np.sqrt(_cov[2,2]); _p=2*_tdist.sf(abs(_b[2]/_se),_df)
print("continuous trend (traditional): +%.3f kcal/mol per bond·site, p=%.4f"%(_b[2],_p))
print("occupancy at 44 nm: classical θ=%.2f[%.2f,%.2f] | plate-FE θ=%.2f[%.2f,%.2f]"%(_thC[-1],_loC[-1],_hiC[-1],_thF[-1],_loF[-1],_hiF[-1]))

fig, ax = plt.subplots(figsize=(8.5, 6))
#ax.errorbar(_dnm, _thF, yerr=[_thF-_loF, _hiF-_thF], fmt='o-', ms=11, color="tab:blue", capsize=5, lw=2.5,
#            label="plate-FE + cluster-robust (appropriate)")
ax.errorbar(_dnm+0.7, _thC, yerr=[_thC-_loC, _hiC-_thC], fmt='s--', ms=8, color="0.5", capsize=4, lw=1.5,
            label="classical OLS")
ax.axhline(1, color="0.6", ls=":")
#for d, t in zip(_dnm, _thF): ax.annotate("%.2f"%t, (d, t), textcoords="offset points", xytext=(8,-16), color="tab:blue")
ax.set(ylim=(0, 1.2), xlabel="Nearest Bond Distance (nm)", ylabel="Relative Occupancy")
       #title="Occupancy decays with bond distance — analytic (no bootstrap)\ntrend test p=%.3f"%_p)
#ax.legend(); ax.secondary_xaxis('top', functions=(lambda v: v/_NM, lambda v: v*_NM)).set_xlabel("distance (bond sites)")
plt.tight_layout(); #plt.show()
plt.savefig('RelativeOccupancy.svg', dpi = 600)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

plot_df = pd.DataFrame({
    "distance_nm": _dnm,
    "occupancy": _thC,
    "lower": _loC,
    "upper": _hiC
})

#fig, ax = plt.subplots(figsize=(8.5, 6))

sns.lineplot(
    data=plot_df,
    x="distance_nm",
    y="occupancy",
    marker="o",
    markersize=10,
    #linewidth=2.5,
    #ax=ax
)

# Asymmetric confidence intervals
#ax.errorbar(
#    plot_df["distance_nm"],
#    plot_df["occupancy"],
#    yerr=[
#        plot_df["occupancy"] - plot_df["lower"],
#        plot_df["upper"] - plot_df["occupancy"]
#    ],
#    fmt="none",
#    capsize=5,
#    linewidth=1.5
#)

plt.errorbar(_dnm, _thC, yerr=[_thC-_loC, _hiC-_thC], fmt='none', ms=8, capsize=5, lw=1.5,
            label="classical OLS")

plt.axhline(1, linestyle=":", color = 'red')

#for _, row in plot_df.iterrows():
    #ax.annotate(
        #f'{row["occupancy"]:.2f}',
        #(row["distance_nm"], row["occupancy"]),
        #xytext=(8, -16),
        #textcoords="offset points",
        #
#    )

#ax.set(
#    xlabel="Nearest Bond Distance (nm)",
#    ylabel="Relative occupancy",
    #title=f"Occupancy decays with bond distance\ntrend test p = {_p:.3f}",
#    ylim=(0, 1.2)
#)
plt.xlabel("Nearest Bond Distance (nm)")
plt.ylabel("Relative Occupancy")
plt.ylim(0,1.2)
#ax.secondary_xaxis(
#    "top",
#    functions=(lambda x: x / _NM, lambda x: x * _NM)
#).set_xlabel("Distance (bond sites)")
#plt.xticks()
#plt.tick_params(axis="both", which="both", length=20, color = 'black')
plt.tick_params(
    axis="x",
    which="major",
    bottom=True,
    top=False,
    length=8,
    width=1.5,
    direction="out")

plt.tick_params(
    axis="y",
    which="major",
    left=True,
    right=False,
    length=8,
    width=2,
    direction="out"
)
plt.tight_layout(); #plt.show()
plt.savefig('RelativeOccupancy.svg', dpi = 600)

# --- paper-figure export ---
export_panel("Fig3c_RelativeOccupancy_vs_Distance",
             data=plot_df.rename(columns={"distance_nm":"nearest_bond_distance_nm",
                                          "occupancy":"relative_occupancy",
                                          "lower":"ci_lower","upper":"ci_upper"}),
             note="Fig 3c. Relative occupancy theta(d)/theta(d1) vs nearest-neighbor bond distance (nm), interfaces with 7nt bonds and 4 binding sites. Classical-OLS point estimate with 95% CI.")

### 10f. Does the distance effect conflate number & strength of bonds?

Fair concern — the raw decomposition treats every bond as identical, and the distance bins **do**
have different length mixes (d1 is ~40% 5-nt / 50% 7-nt; d≥4 is ~92% 7-nt). Three checks:

1. **Control for number and strength together** by adding the additive NN prediction `add_dG`
   (which sums the length- and GC-specific energy of *every* bond) as a covariate. The distance
   penalty is unchanged (+0.10 → +0.15 kcal/mol per bond·site) and, with plate fixed effects,
   **significant (p≈0.002)**. Controlling for strength if anything *raises* it — because isolated
   bonds are the *stronger* 7-nt type, so their penalty is more surprising, not less.
2. **Homogeneous subset — 4-bond, 7-nt-only** (both # and strength fixed, exactly your test):
   ΔG° vs total bond-isolation is **ρ=0.49, p<0.001 (n=56)**, coefficient **+0.14** — the *same
   slope* as the full data. So the effect is present with number and strength held constant.
3. **Why your *binned* plot looked weaker there:** a 4-bond set has a compressed geometry range
   (≤24 nm) and very few far-apart bonds (only ~19 at d≥3 across the subset), so the *binned*
   occupancy curve is starved of data in its far bins. The *continuous* distance measure keeps the
   power and shows the trend clearly.

**Bottom line:** the geometry/occupancy effect is not an artefact of valency or bond strength — it
survives controlling for both, and holds within a fixed-number/fixed-length subset. Caveat: it
needs plate control to reach significance, and the homogeneous subset spans only 2 plates.


In [ ]:
# === Does the distance effect conflate number & strength of bonds? ============
# Concern: the raw decomposition treats every bond alike, but the distance bins have
# different bond-length mixes. Test whether the effect survives controlling for both,
# and inside a fully homogeneous subset (fixed # and length).
from scipy.stats import t as _tt, spearmanr as _sp
from collections import Counter as _Ctr
_RC = vh_reps.copy(); _RC["Samples"] = _RC["code"]
_AC = (filterData_std(_RC).groupby("code").agg(dG37=('dG37','median'), exp=('exp','first')).reset_index())
_AC = _AC[~_AC['code'].str.contains('A|B')].reset_index(drop=True)
_AC['add_dG'] = _AC['code'].map(lambda c: additive_from_code(c, 'G_corrected'))
_AC['nb'] = _AC['code'].map(lambda c: sum(ch != '0' for ch in c))
_AC['sumNN'] = _AC['code'].map(lambda c: (lambda pos: sum(min(abs(p-q) for q in pos if q != p) for p in pos))([i for i, ch in enumerate(c) if ch != '0']))
def _only7(c): return set(ch for ch in c if ch != '0') <= {'7', 'X'}
def _bonds(code):
    pos=[i for i,ch in enumerate(code) if ch!='0']
    return [(min(min(abs(p-q) for q in pos if q!=p),4), 5 if code[p]=='X' else int(code[p])) for p in pos]
_E = pd.get_dummies(_AC['exp'], drop_first=True).values.astype(float); _y = pd.to_numeric(_AC['dG37']).values
def _olsc(X, y, cl):
    b=np.linalg.lstsq(X,y,rcond=None)[0]; r=y-X@b; XtXi=np.linalg.inv(X.T@X); n,k=X.shape
    meat=sum(np.outer(X[cl==c].T@r[cl==c], X[cl==c].T@r[cl==c]) for c in np.unique(cl)); G=len(np.unique(cl))
    return b, np.sqrt(np.diag((G/(G-1))*((n-1)/(n-k))*XtXi@meat@XtXi)), G-1

_bl={b:_Ctr() for b in [1,2,3,4]}
for c in _AC['code']:
    for d,L in _bonds(c): _bl[d][L]+=1
print("bond-length mix by distance bin:", {("d%s"%('≥4' if b==4 else b)):{L:"%.0f%%"%(100*n/sum(_bl[b].values())) for L,n in sorted(_bl[b].items())} for b in [1,2,3,4]})

fig, ax = plt.subplots(1, 3, figsize=(19, 5.6))
_lens=[5,6,7,9]; _cols={5:"#f4a582",6:"#fddbc7",7:"#92c5de",9:"#0571b0"}; _bot=np.zeros(4)
for L in _lens:
    v=np.array([_bl[b][L]/sum(_bl[b].values()) for b in [1,2,3,4]]); ax[0].bar(range(4),v,bottom=_bot,color=_cols[L],label="%dnt"%L,ec="w"); _bot+=v
ax[0].set(xticks=range(4),ylabel="fraction of bonds",title="Bin composition differs by length (the confound)")
ax[0].set_xticklabels(['d1\n9nm','d2\n18','d3\n26','d≥4\n40']); ax[0].legend(ncol=2,loc="lower center")

_models=[("distance only",[np.ones(len(_AC)),_AC.sumNN.values]),
         ("+ #bonds",[np.ones(len(_AC)),_AC.sumNN.values,_AC.nb.values]),
         ("+ additive (#+strength)",[np.ones(len(_AC)),_AC.sumNN.values,_AC.add_dG.values]),
         ("+ additive + plate FE",[np.ones(len(_AC)),_AC.sumNN.values,_AC.add_dG.values]+[_E[:,i] for i in range(_E.shape[1])])]
_names=[]; _x=[]; _e=[]; _c=[]
for nm,cols in _models:
    X=np.column_stack(cols); b,se,df=_olsc(X,_y,_AC['exp'].values); tc=_tt.ppf(0.975,df)
    _names.append(nm); _x.append(b[1]); _e.append(tc*se[1]); _c.append("tab:blue" if b[1]-tc*se[1]>0 else "0.6")
_S=_AC[(_AC.nb==4)&_AC['code'].map(_only7)].copy(); _yS=pd.to_numeric(_S['dG37']).values
_ES=pd.get_dummies(_S['exp'],drop_first=True).values.astype(float)
_X=np.column_stack([np.ones(len(_S)),_S.sumNN.values]+[_ES[:,i] for i in range(_ES.shape[1])]); b,se,df=_olsc(_X,_yS,_S['exp'].values); tc=_tt.ppf(0.975,df)
_names.append("4-bond 7nt only\n(#+strength fixed)"); _x.append(b[1]); _e.append(tc*se[1]); _c.append("tab:green")
_yp=np.arange(len(_names))[::-1]
ax[1].errorbar(_x,_yp,xerr=_e,fmt='none',capsize=4,lw=2,ecolor="0.4")
for xx,yp,cc in zip(_x,_yp,_c): ax[1].plot(xx,yp,'o',ms=11,color=cc)
ax[1].axvline(0,color="r",ls="--"); ax[1].set_yticks(_yp); ax[1].set_yticklabels(_names)
ax[1].set(xlabel="ΔG° penalty per bond·site (kcal/mol)",title="Distance effect survives controls\n(coloured = CI excludes 0)")
sns.regplot(x=_S.sumNN,y=_yS,ax=ax[2],scatter_kws=dict(s=45,alpha=.6,color="tab:green"),line_kws=dict(color="k"))
_rho,_p=_sp(_S.sumNN,_yS)
ax[2].set(xlabel="total bond isolation  Σ nearest-neighbour distance (sites)",ylabel="measured ΔG°$_{37}$ (kcal/mol)",
          title="4-bond, 7-nt only (n=%d): ρ=%.2f, p=%.1e\nsame slope as full data"%(len(_S),_rho,_p))
plt.tight_layout(); plt.show()
print("4-bond 7nt subset: coef=%+.3f kcal/mol per bond·site (matches full-data +0.15); n=%d from %d plates"%(b[1],len(_S),_S['exp'].nunique()))


### 10g. Same confound check for ΔH° and ΔS°

Repeating 10f with ΔH° and ΔS° as the response gives a sharp, honest split:

- **ΔG°** — the distance penalty **survives** controlling for number & strength (additive + plate FE:
  +0.15 kcal/mol per bond·site, p=0.0002; 4-bond-7-nt subset ρ=0.49, p<0.001).
- **ΔH°** — coef **−1.1 kcal/mol, p=0.49**; 4-bond-7-nt subset **ρ=−0.03, p=0.82**. No effect.
- **ΔS°** — coef **−4 cal/mol/K, p=0.44**; 4-bond-7-nt subset **ρ=−0.04, p=0.75**. No effect.

**So your concern is exactly right for ΔH°/ΔS°.** The raw ΔH°/ΔS°-vs-distance trends in 10b were
**mostly the number/strength confound** (add_dH/add_dS scale strongly with valency and length) plus
large van't Hoff slope noise; once you control for number and strength — or fix them in the
homogeneous subset — the ΔH°/ΔS° distance effect **disappears**. Only **ΔG°** carries a robust
occupancy-vs-distance signal, because it is far more reproducible (replicate SD ~0.2 kcal/mol) than
ΔH° (~11 kcal/mol) or ΔS° (~35 cal/mol/K), so it has the statistical power the others lack.
The 10b "ΔH°/ΔS° decay together" figure should therefore be read as compensation of the
number/strength effect, **not** an independent distance effect.


In [ ]:
# === Same confound check (Section 10f) repeated for ΔH° and ΔS° =================
# Does the per-bond distance effect on ΔH°/ΔS° survive controlling for number & strength?
from scipy.stats import t as _tt, spearmanr as _sp
_RH = vh_reps.copy(); _RH["Samples"] = _RH["code"]
_AH = (filterData_std(_RH).groupby("code").agg(dH=('dH','median'), dS=('dS','median'),
        dG37=('dG37','median'), exp=('exp','first')).reset_index())
_AH = _AH[~_AH['code'].str.contains('A|B')].reset_index(drop=True)
for _r in ['H','S','G_corrected']: _AH['add_'+_r]=_AH['code'].map(lambda c: additive_from_code(c,_r))
_AH['nb']=_AH['code'].map(lambda c: sum(ch!='0' for ch in c))
_AH['sumNN']=_AH['code'].map(lambda c:(lambda pos:sum(min(abs(p-q) for q in pos if q!=p) for p in pos))([i for i,ch in enumerate(c) if ch!='0']))
def _o7(c): return set(ch for ch in c if ch!='0')<={'7','X'}
_EH=pd.get_dummies(_AH['exp'],drop_first=True).values.astype(float)
def _ols(X,y,cl=None):
    b=np.linalg.lstsq(X,y,rcond=None)[0]; r=y-X@b; XtXi=np.linalg.inv(X.T@X); n,k=X.shape
    if cl is None: cov=(r@r/(n-k))*XtXi; df=n-k
    else:
        meat=sum(np.outer(X[cl==c].T@r[cl==c],X[cl==c].T@r[cl==c]) for c in np.unique(cl)); G=len(np.unique(cl)); cov=(G/(G-1))*((n-1)/(n-k))*XtXi@meat@XtXi; df=G-1
    return b,np.sqrt(np.diag(cov)),df
def _coef(resp,addc,mode):
    y=pd.to_numeric(_AH[resp]).values
    if mode=='fe': X=np.column_stack([np.ones(len(_AH)),_AH.sumNN.values,_AH[addc].values]+[_EH[:,i] for i in range(_EH.shape[1])]); b,se,df=_ols(X,y,None)
    else:
        base=[np.ones(len(_AH)),_AH.sumNN.values]
        if 'nb' in mode: base.append(_AH.nb.values)
        if 'add' in mode: base.append(_AH[addc].values)
        b,se,df=_ols(np.column_stack(base),y,_AH['exp'].values)
    tc=_tt.ppf(0.975,df); return b[1],tc*se[1]
_models=[("dist only","dist"),("+ #bonds","dist+nb"),("+ additive (#+strength)","dist+add"),("+ additive + plate FE","fe")]
_specs=[('dG37','add_G_corrected','ΔG°','kcal/mol','tab:green'),('dH','add_H','ΔH°','kcal/mol','tab:red'),('dS','add_S','ΔS°','cal/mol/K','tab:purple')]
fig,ax=plt.subplots(1,3,figsize=(19,5.6))
for a,(resp,addc,ylab,unit,col) in zip(ax,_specs):
    xs=[];es=[];nm=[]
    for lab,m in _models:
        c,e=_coef(resp,addc,m); xs.append(c); es.append(e); nm.append(lab)
    _S=_AH[(_AH.nb==4)&_AH['code'].map(_o7)]; _yS=pd.to_numeric(_S[resp]).values
    b,se,df=_ols(np.column_stack([np.ones(len(_S)),_S.sumNN.values]),_yS,None); tc=_tt.ppf(0.975,df)
    xs.append(b[1]); es.append(tc*se[1]); nm.append("4-bond 7nt only\n(#+strength fixed)")
    _rho,_p=_sp(_S.sumNN,_yS); yp=np.arange(len(nm))[::-1]
    _c=[col if abs(x)>e else "0.6" for x,e in zip(xs,es)]
    a.errorbar(xs,yp,xerr=es,fmt='none',ecolor="0.4",capsize=4,lw=2)
    for x,y_,cc in zip(xs,yp,_c): a.plot(x,y_,'o',ms=11,color=cc)
    a.axvline(0,color="r",ls="--"); a.set_yticks(yp); a.set_yticklabels(nm)
    a.set(xlabel="distance coef per bond·site (%s)"%unit, title="%s vs isolation\n4-bond-7nt: ρ=%.2f, p=%.2g"%(ylab,_rho,_p))
    print("%s: 4-bond-7nt ρ=%+.2f p=%.3f | +additive+FE coef=%+.3f"%(ylab,_rho,_p,xs[3]))
fig.suptitle("Confound check for ΔH° and ΔS° (ΔG° for reference): coloured dot = CI excludes 0",y=1.02)
plt.tight_layout(); plt.show()


### 10h. Is the 4→5 cooperativity shift valency or composition? (ANCOVA)

Your test, implemented. Per-bond ΔΔG = (measured − additive)/N, within N=4 and N=5.

- The raw N=4 group is **contaminated by the 9-nt constructs** (per-bond ΔΔG ≈ 16 vs ≈ 6 for the rest;
  N=5 has none), which is why the naive within-N GC slopes flipped sign. Restricting to **7-nt-matched**
  constructs fixes it.
- **Within each N, per-bond ΔΔG does track composition** (GC slopes +10 for N=4, +8 for N=5 — parallel,
  i.e. the *same* composition dependence). So some within-N spread is composition-driven.
- **ANCOVA (per-bond ΔΔG ~ GC + N):** the 4→5 gap **survives** adjustment for composition —
  **+0.83 kcal/mol/bond**, and the N=5 regression line sits cleanly above N=4. The two lines are
  parallel and vertically offset by valency.

**Verdict:** the 4→5 shift is **valency — real diminishing returns.** Composition sets the *level within*
each N but does not create the 4→5 gap, and **NN miscalibration is dead as an explanation for the slope.**
(Caveat, as before: the 7-nt subset spans only 2 plates, so read the tiny p-value as "clearly non-zero",
not as its literal value; the load-bearing result is the +0.8 kcal/mol/bond offset with parallel slopes.)


In [ ]:
# === Valency vs composition for the 4→5 cooperativity shift (ANCOVA) ===========
# Per-bond ΔΔG = (measured − additive)/N. Within N=4 and N=5, regress it on composition,
# then ANCOVA to see whether the 4→5 level gap survives adjusting for composition.
from scipy.stats import t as _tt, linregress as _lr
_RV = vh_reps.copy(); _RV["Samples"] = _RV["code"]
_AV = (filterData_std(_RV).groupby("code").agg(dG37=('dG37','median')).reset_index())
_AV = _AV[~_AV['code'].str.contains('A|B')].reset_index(drop=True)
_AV['add_dG'] = _AV['code'].map(lambda c: additive_from_code(c,'G_corrected'))
_AV['N'] = _AV['code'].map(lambda c: sum(ch!='0' for ch in c))
_AV['pb_coop'] = (_AV['dG37'] - _AV['add_dG']) / _AV['N']          # per-bond ΔΔG (sub-additivity)
_seq = {L: pd.read_csv(f"nt{L}.csv", index_col=0)['Sequence'].astype(str).tolist() for L in [5,6,7,9]}
def _gc(s): return sum(c in 'GC' for c in s)/len(s) if s and set(s) <= set('ACGT') else np.nan
def _cgc(code):
    v=[]
    for i,ch in enumerate(code):
        t=_seq[5] if ch=='X' else (_seq.get(int(ch)) if ch in '5679' else None)
        if t is None: continue
        for r in (2*i,2*i+1):
            if r<len(t) and not np.isnan(_gc(t[r])): v.append(_gc(t[r]))
    return np.mean(v) if v else np.nan
_AV['GC'] = _AV['code'].map(_cgc)
def _o7(c): return set(ch for ch in c if ch!='0') <= {'7','X'}
_AV['is7'] = _AV['code'].map(_o7); _AV['has9'] = _AV['code'].str.contains('9')

# 7-nt-matched subset (removes the 9-nt subgroup that sits only in N=4)
_S = _AV[_AV.is7 & _AV.N.isin([4,5])].copy(); _col={4:"tab:blue",5:"tab:orange"}
_X = np.column_stack([np.ones(len(_S)), _S.GC.values, (_S.N==5).astype(float).values]); _y = _S.pb_coop.values
_b = np.linalg.lstsq(_X,_y,rcond=None)[0]; _r=_y-_X@_b; _XtXi=np.linalg.inv(_X.T@_X); _n,_k=_X.shape
_se=np.sqrt(np.diag((_r@_r/(_n-_k))*_XtXi)); _pN=2*_tt.sf(abs(_b[2]/_se[2]),_n-_k)
print("within-N GC slopes:  N4=%.1f  N5=%.1f  (parallel => same composition effect)"%(
      _lr(_S[_S.N==4].GC,_S[_S.N==4].pb_coop).slope, _lr(_S[_S.N==5].GC,_S[_S.N==5].pb_coop).slope))
print("ANCOVA 4→5 valency gap (adjusted for GC) = %+.2f kcal/mol/bond, p=%.1g"%(_b[2],_pN))
print("(9-nt contaminates the raw N=4 group: per-bond ΔΔG %.1f vs %.1f for 5/7-nt)"%(
      _AV[(_AV.N==4)&_AV.has9].pb_coop.mean(), _AV[(_AV.N==4)&~_AV.has9].pb_coop.mean()))

fig,ax=plt.subplots(1,3,figsize=(19,5.6))
for N in [4,5]:
    d=_AV[_AV.N==N]; ax[0].scatter(np.full(len(d),N)+np.random.uniform(-.12,.12,len(d)),d.pb_coop,
                    c=["tab:red" if h else "0.5" for h in d.has9],s=30,alpha=.7)
ax[0].set(xticks=[4,5],xlabel="N (number of bonds)",ylabel="per-bond ΔΔG (kcal/mol)",
          title="Raw N=4 is contaminated by 9-nt\n(red = has 9-nt; N=5 has none)")
_gx=np.linspace(_S.GC.min(),_S.GC.max(),50)
for N in [4,5]:
    d=_S[_S.N==N]; ax[1].scatter(d.GC,d.pb_coop,color=_col[N],s=45,alpha=.7,label="N=%d"%N)
    ax[1].plot(_gx,_b[0]+_b[1]*_gx+_b[2]*(N==5),color=_col[N],lw=2.5)
ax[1].set(xlabel="GC fraction (composition)",ylabel="per-bond ΔΔG (kcal/mol)",
          title="7-nt only: parallel within-N GC slopes;\nN=5 line ABOVE N=4  →  valency")
ax[1].legend(); ax[1].text(.03,.95,"ANCOVA N(4→5)=%+.2f kcal/mol/bond\np=%.1g (adj. for GC)"%(_b[2],_pN),
             transform=ax[1].transAxes,va="top",bbox=dict(fc="w",ec="0.5"))
sns.boxplot(data=_S,x="N",y="pb_coop",hue="N",palette=_col,legend=False,ax=ax[2])
sns.stripplot(data=_S,x="N",y="pb_coop",ax=ax[2],color="0.25",size=4,alpha=.6)
ax[2].set(xlabel="N",ylabel="per-bond ΔΔG (kcal/mol)",
          title="7-nt only: N=4 %.2f vs N=5 %.2f (levels differ)"%(_S[_S.N==4].pb_coop.mean(),_S[_S.N==5].pb_coop.mean()))
plt.tight_layout(); plt.show()


### 10i. Diminishing returns, shown directly (measured vs additive vs N)

The most direct picture of the same result: plot the measured stability and the additive/predicted
stability against the number of bonds.

- **ΔG°:** the additive prediction keeps getting more favourable as bonds are added
  (−41 → −63 kcal/mol), but the **measured ΔG° barely moves** (~−13 to −15). The widening red gap
  *is* the diminishing return — each extra bond adds essentially nothing to the measured stability.
- **T$_m$:** the bonds "should" melt at ~70 °C (additive), yet the **assembly melts at ~40 °C
  regardless of N** — a ~30 °C deficit that does not close as bonds are added.

Read this alongside the boxplot in 10h (per-bond ΔΔG rising with N is the same effect, one step removed).


In [ ]:
# === The visceral diminishing-returns plot: measured vs additive/predicted vs N ==
_RD = vh_reps.copy(); _RD["Samples"] = _RD["code"]
_AD = (filterData_std(_RD).groupby("code").agg(dG37=('dG37','median'), Tm=('Tm','median')).reset_index())
_AD = _AD[~_AD['code'].str.contains('A|B')].reset_index(drop=True)
_AD['Tm'] = pd.to_numeric(_AD['Tm'])
_AD['N']  = _AD['code'].map(lambda c: sum(ch!='0' for ch in c))
_AD['add_dG'] = _AD['code'].map(lambda c: additive_from_code(c,'G_corrected'))
_AD['add_dH'] = _AD['code'].map(lambda c: additive_from_code(c,'H'))
_AD['add_dS'] = _AD['code'].map(lambda c: additive_from_code(c,'S'))
_Rg = 1.987204; _CT = 4.5e-9
_AD['Tm_pred'] = _AD['add_dH']*1000/(_AD['add_dS'] + _Rg*np.log(_CT/4)) - 273.15   # additive-predicted Tm
_Ns = [n for n,s in _AD.groupby('N') if len(s) >= 3]
def _ag(col): m=_AD.groupby('N')[col]; return m.mean().loc[_Ns], m.sem().loc[_Ns]
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
_mg,_mge = _ag('dG37'); _agg,_ = _ag('add_dG')
ax[0].fill_between(_Ns, _mg.values, _agg.values, color="tab:red", alpha=.12)
ax[0].errorbar(_Ns, _agg.values, fmt='s--', color="tab:red", ms=10, lw=2, label="additive (Σ NN bonds)")
ax[0].errorbar(_Ns, _mg.values, yerr=_mge.values, fmt='o-', color="tab:blue", ms=11, lw=2.5, capsize=4, label="measured")
for n in _Ns: ax[0].annotate("gap %.0f"%(_agg[n]-_mg[n]), (n,(_agg[n]+_mg[n])/2), color="tab:red", ha="center")
ax[0].set(xlabel="N (number of bonds)", ylabel="ΔG°$_{37}$ (kcal/mol)",
          title="ΔG°: additive keeps dropping, measured flattens\n→ diminishing returns")
ax[0].legend(loc="center left"); ax[0].set_xticks(_Ns); ax[0].invert_yaxis()
_mt,_mte = _ag('Tm'); _pt,_ = _ag('Tm_pred')
ax[1].fill_between(_Ns, _mt.values, _pt.values, color="tab:red", alpha=.12)
ax[1].errorbar(_Ns, _pt.values, fmt='s--', color="tab:red", ms=10, lw=2, label="predicted (additive)")
ax[1].errorbar(_Ns, _mt.values, yerr=_mte.values, fmt='o-', color="tab:blue", ms=11, lw=2.5, capsize=4, label="measured")
for n in _Ns: ax[1].annotate("gap %.0f°"%(_pt[n]-_mt[n]), (n,(_pt[n]+_mt[n])/2), color="tab:red", ha="center")
ax[1].set(xlabel="N (number of bonds)", ylabel="T$_m$ (°C)",
          title="T$_m$: bonds ‘should’ melt at ~70 °C,\nassembly melts at ~40 °C regardless of N")
ax[1].legend(loc="center left"); ax[1].set_xticks(_Ns)
fig.text(0.5, -0.02, "(N=9,10 are the 9-nt constructs; their lower measured values are partly AT-rich composition — see 10h)",
         ha="center", style="italic")
plt.tight_layout(); #plt.show()
plt.savefig('Diminish.svg', dpi =600)

In [ ]:
# === Does the distance effect conflate number & strength of bonds? ============
# Concern: the raw decomposition treats every bond alike, but the distance bins have
# different bond-length mixes. Test whether the effect survives controlling for both,
# and inside a fully homogeneous subset (fixed # and length).
from scipy.stats import t as _tt, spearmanr as _sp
from collections import Counter as _Ctr
_RC = vh_reps.copy(); _RC["Samples"] = _RC["code"]
#_AC = (filterData_std(_RC).groupby("code").agg(dG37=('dH','median'), exp=('exp','first')).reset_index())
_AC = _AC[~_AC['code'].str.contains('A|B')].reset_index(drop=True)
_AC['add_dG'] = _AC['code'].map(lambda c: additive_from_code(c, 'G_corrected'))
_AC['nb'] = _AC['code'].map(lambda c: sum(ch != '0' for ch in c))
_AC['sumNN'] = _AC['code'].map(lambda c: (lambda pos: sum(min(abs(p-q) for q in pos if q != p) for p in pos))([i for i, ch in enumerate(c) if ch != '0']))
def _only7(c): return set(ch for ch in c if ch != '0') <= {'7', 'X'}
def _bonds(code):
    pos=[i for i,ch in enumerate(code) if ch!='0']
    return [(min(min(abs(p-q) for q in pos if q!=p),4), 5 if code[p]=='X' else int(code[p])) for p in pos]
_E = pd.get_dummies(_AC['exp'], drop_first=True).values.astype(float); _y = pd.to_numeric(_AC['dG37']).values
def _olsc(X, y, cl):
    b=np.linalg.lstsq(X,y,rcond=None)[0]; r=y-X@b; XtXi=np.linalg.inv(X.T@X); n,k=X.shape
    meat=sum(np.outer(X[cl==c].T@r[cl==c], X[cl==c].T@r[cl==c]) for c in np.unique(cl)); G=len(np.unique(cl))
    return b, np.sqrt(np.diag((G/(G-1))*((n-1)/(n-k))*XtXi@meat@XtXi)), G-1

_bl={b:_Ctr() for b in [1,2,3,4]}
for c in _AC['code']:
    for d,L in _bonds(c): _bl[d][L]+=1
print("bond-length mix by distance bin:", {("d%s"%('≥4' if b==4 else b)):{L:"%.0f%%"%(100*n/sum(_bl[b].values())) for L,n in sorted(_bl[b].items())} for b in [1,2,3,4]})

fig, ax = plt.subplots(1, 3, figsize=(19, 5.6))
_lens=[5,6,7,9]; _cols={5:"#f4a582",6:"#fddbc7",7:"#92c5de",9:"#0571b0"}; _bot=np.zeros(4)
for L in _lens:
    v=np.array([_bl[b][L]/sum(_bl[b].values()) for b in [1,2,3,4]]); ax[0].bar(range(4),v,bottom=_bot,color=_cols[L],label="%dnt"%L,ec="w"); _bot+=v
ax[0].set(xticks=range(4),ylabel="fraction of bonds",title="Bin composition differs by length (the confound)")
ax[0].set_xticklabels(['d1\n9nm','d2\n18','d3\n26','d≥4\n40']); ax[0].legend(ncol=2,loc="lower center")

_models=[("distance only",[np.ones(len(_AC)),_AC.sumNN.values]),
         ("+ #bonds",[np.ones(len(_AC)),_AC.sumNN.values,_AC.nb.values]),
         ("+ additive (#+strength)",[np.ones(len(_AC)),_AC.sumNN.values,_AC.add_dG.values]),
         ("+ additive + plate FE",[np.ones(len(_AC)),_AC.sumNN.values,_AC.add_dG.values]+[_E[:,i] for i in range(_E.shape[1])])]
_names=[]; _x=[]; _e=[]; _c=[]
for nm,cols in _models:
    X=np.column_stack(cols); b,se,df=_olsc(X,_y,_AC['exp'].values); tc=_tt.ppf(0.975,df)
    _names.append(nm); _x.append(b[1]); _e.append(tc*se[1]); _c.append("tab:blue" if b[1]-tc*se[1]>0 else "0.6")
_S=_AC[(_AC.nb==4)&_AC['code'].map(_only7)].copy(); _yS=pd.to_numeric(_S['dG37']).values
_ES=pd.get_dummies(_S['exp'],drop_first=True).values.astype(float)
_X=np.column_stack([np.ones(len(_S)),_S.sumNN.values]+[_ES[:,i] for i in range(_ES.shape[1])]); b,se,df=_olsc(_X,_yS,_S['exp'].values); tc=_tt.ppf(0.975,df)
_names.append("4-bond 7nt only\n(#+strength fixed)"); _x.append(b[1]); _e.append(tc*se[1]); _c.append("tab:green")
_yp=np.arange(len(_names))[::-1]
ax[1].errorbar(_x,_yp,xerr=_e,fmt='none',capsize=4,lw=2,ecolor="0.4")
for xx,yp,cc in zip(_x,_yp,_c): ax[1].plot(xx,yp,'o',ms=11,color=cc)
ax[1].axvline(0,color="r",ls="--"); ax[1].set_yticks(_yp); ax[1].set_yticklabels(_names)
ax[1].set(xlabel="ΔG° penalty per bond·site (kcal/mol)",title="Distance effect survives controls\n(coloured = CI excludes 0)")
sns.regplot(x=_S.sumNN,y=_yS,ax=ax[2],scatter_kws=dict(s=45,alpha=.6,color="tab:green"),line_kws=dict(color="k"))
_rho,_p=_sp(_S.sumNN,_yS)
ax[2].set(xlabel="total bond isolation  Σ nearest-neighbour distance (sites)",ylabel="measured ΔG°$_{37}$ (kcal/mol)",
          title="4-bond, 7-nt only (n=%d): ρ=%.2f, p=%.1e\nsame slope as full data"%(len(_S),_rho,_p))
plt.tight_layout(); plt.show()
print("4-bond 7nt subset: coef=%+.3f kcal/mol per bond·site (matches full-data +0.15); n=%d from %d plates"%(b[1],len(_S),_S['exp'].nunique()))


In [ ]:
_AC = (filterData_std_H(vh_reps).groupby("code").agg(dG37=('dS','median'), exp=('exp','first')).reset_index())
_AC['sumNN'] = _AC['code'].map(lambda c: (lambda pos: sum(min(abs(p-q) for q in pos if q != p) for p in pos))([i for i, ch in enumerate(c) if ch != '0']))

In [ ]:
vh1=vh[vh['Num_Bonds'] == 4]
vh1=vh1[vh1['NucleotideBonds'] == 7]

In [ ]:
for _y in ['dH', 'dS']:
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
    sns.regplot(data=vh1, x='MeanGap', y=_y, ax=axs[0]); axs[0].set_xlabel('Mean gap (bond sites)')
    sns.regplot(data=vh1, x='MeanGap_nm', y=_y, ax=axs[1]); axs[1].set_xlabel('Mean gap (nm)')
    plt.tight_layout(); plt.show()


In [ ]:
# === Fig 2b (paper-figure export): NN-predicted dG vs experimental Tm ===
_bx = pd.to_numeric(df_Tm_median["Predicted_deltaG"], errors="coerce")
_by = pd.to_numeric(df_Tm_median["Tm"], errors="coerce")
_mask2b = _bx.notna() & _by.notna()
plt.figure()
sns.regplot(x=_bx[_mask2b], y=_by[_mask2b], scatter_kws={"alpha": 0.4}, line_kws={"color": "red"})
plt.xlabel("NN-predicted ΔG (kcal/mol)"); plt.ylabel("Experimental Tm (°C)"); plt.tight_layout()
from scipy.stats import pearsonr as _pearsonr, spearmanr as _spearmanr
_r2b, _ = _pearsonr(_bx[_mask2b], _by[_mask2b]); _rho2b, _ = _spearmanr(_bx[_mask2b], _by[_mask2b])
print(f"Fig2b: Pearson r = {_r2b:.3f}, Spearman rho = {_rho2b:.3f}")
export_panel("Fig2b_PredicteddG_vs_Tm",
             data=pd.DataFrame({"Code": df_Tm_median.loc[_mask2b, "Code"].values,
                                "NN_Predicted_deltaG_kcalmol": _bx[_mask2b].values,
                                "Experimental_Tm_C": _by[_mask2b].values}),
             note=f"Fig 2b. NN-predicted interface dG (sum of per-bond nearest-neighbor contributions) vs experimental Tm, per construct (authoritative Tm table). Pearson r={_r2b:.3f}, Spearman rho={_rho2b:.3f} (weak correlation).")

# --- Fig2b HIGH-reliability variant ---
_bxh = pd.to_numeric(df_Tm_median_high["Predicted_deltaG"], errors="coerce")
_byh = pd.to_numeric(df_Tm_median_high["Tm"], errors="coerce")
_mh = _bxh.notna() & _byh.notna()
plt.figure()
sns.regplot(x=_bxh[_mh], y=_byh[_mh], scatter_kws={"alpha":0.4}, line_kws={"color":"red"})
plt.xlabel("NN-predicted ΔG (kcal/mol)"); plt.ylabel("Experimental Tm (°C)"); plt.tight_layout()
export_panel_high("Fig2b_PredicteddG_vs_Tm",
                  pd.DataFrame({"Code":df_Tm_median_high.loc[_mh,"Code"].values,
                                "NN_Predicted_deltaG_kcalmol":_bxh[_mh].values,
                                "Experimental_Tm_C":_byh[_mh].values}))
plt.close()

In [ ]:
# === Avidity: effective molarity (EM) & avidity enhancement per interface =====================
# Standard multivalency metrics (Jencks effective concentration; Mammen-Choi-Whitesides 1998).
#   Model:  dG_Exp - dG_Seq(1 init) = (N-1) * RT * ln(1 M / EM)
#     dG_Seq(1 init) = sum of per-bond (NN stacking + salt) + ONE bimolecular initiation
#                      (the tiles dock as one body; bonds 2..N are intramolecular).
#     N = total duplex bonds = 2 per binding site.
#   EM  (vs 1 M standard state):  EM < 1 M  => negative cooperativity (alpha<1).
#   EM / C_exp (vs 4.5 nM bulk):  >> 1       => strong avidity enhancement.
# Uses in-memory NN objects (calc_unified_nn, init_G, constant_1, nt5_1/nt6_1/nt7_1) + Filtered_DF_HS.
# 9nt excluded.  Writes Paper_Figures/Avidity/avidity_raw_datapoints.csv + EM_and_avidity_enhancement.svg
import os, math, numpy as np, pandas as pd
import matplotlib.pyplot as plt
_OUT = "Paper_Figures/Avidity"; os.makedirs(_OUT, exist_ok=True)
_RT = 1.987204e-3 * 310.15; _C_EXP = 4.5e-9
_LEN = {"5":5,"X":5,"6":6,"7":7,"9":9,"A":7,"B":7}
_nt1 = {"5":nt5_1,"6":nt6_1,"7":nt7_1}          # 9nt excluded
_ionic = 3.3 * 0.0114
def _corr(L): return 0.114 * L * math.log(math.sqrt(_ionic))        # salt term, one duplex of length L
def _initG(s): s=s.upper(); return sum(init_G["AT" if b in "AT" else "GC"] for b in (s[0], s[-1]))
def _seqs(code):
    out=[]
    for i,ch in enumerate(code):
        if ch=="0": continue
        L=_LEN.get(ch)
        if L is None or ch=="9": return None
        if ch in "5X67":
            t="5" if ch=="X" else ch
            out.append((str(_nt1[t]["Sequence"][2*i]), str(_nt1[t]["Sequence"][2*i+1]), L))
        elif ch=="A": out.append((str(constant_1["Sequence"][0]), str(constant_1["Sequence"][1]), L))
        elif ch=="B": out.append((str(constant_1["Sequence"][2]), str(constant_1["Sequence"][3]), L))
    return out
def _bondtab(code):
    s=_seqs(code)
    if s is None: return None
    dG=[]; ini=[]
    for s1,s2,L in s:
        for q in (s1,s2):
            dG.append(calc_unified_nn(q)["Delta_G_37 (kcal/mol)"] - _corr(L)); ini.append(_initG(q))
    return np.array(dG), np.array(ini)

_d = pd.read_csv("Filtered_DF_HS.csv"); _d["code"]=_d["Code"].astype(str).str.partition(".")[0]
_sub = _d[(_d["Reliable"]==True) & (_d["Confidence"]=="high")]
_g = (_sub.groupby("code").agg(Exp_dG=("Exp_DeltaG","median"), nt=("NucleotideBonds","first"),
        n_sites=("Num_Bonds","first"), n_reps=("Exp_DeltaG","size")).reset_index())
_g = _g[_g["n_reps"]>=2].dropna(subset=["Exp_dG","n_sites"])
_g = _g[~_g["code"].str.contains("9")].reset_index(drop=True); _g["nt"]=_g["nt"].astype(str)

_rows=[]
for _,r in _g.iterrows():
    bt=_bondtab(r["code"])
    if bt is None: continue
    dG,ini=bt; N=len(dG)
    if N<2: continue
    dG_mono_med=float(np.median(dG))
    seq1=dG.sum() - ini.sum()*(N-1)/N
    short=r["Exp_dG"]-seq1
    EM=math.exp(-short/((N-1)*_RT))
    _rows.append(dict(code=r["code"], nt=r["nt"], n_sites=int(r["n_sites"]), N_bonds=N, n_reps=int(r["n_reps"]),
        dG_Exp_kcal=round(r["Exp_dG"],3), dG_mono_median_kcal=round(dG_mono_med,3),
        dG_Seq_1init=round(seq1,3), shortfall_kcal=round(short,3),
        EM_per_bond_M=EM, EM_mM=EM*1e3, cooperativity_alpha=round((r["Exp_dG"]/N)/dG_mono_med,4),
        avidity_enhancement_fold=EM/_C_EXP, avidity_dG_perbond_kcal=round(_RT*math.log(EM/_C_EXP),3)))
avidity_df = pd.DataFrame(_rows)
avidity_df.to_csv(os.path.join(_OUT,"avidity_raw_datapoints.csv"), index=False)

_emM=avidity_df["EM_per_bond_M"]; _enh=avidity_df["avidity_enhancement_fold"]
print(f"n interfaces = {len(avidity_df)}")
print(f"effective molarity : median={_emM.median()*1e3:.2f} mM  IQR=[{_emM.quantile(.25)*1e3:.2f},{_emM.quantile(.75)*1e3:.2f}] mM  (100% < 1 M => negative cooperativity)")
print(f"avidity enhancement: median={_enh.median():.2e}x  (= {_RT*math.log(_enh.median()):.1f} kcal/mol per extra bond vs 4.5 nM bulk)")

_fig,_ax=plt.subplots(1,2,figsize=(13.2,5.1))
_ax[0].hist(np.log10(_emM),bins=24,color="#4C78A8",alpha=.85,edgecolor="white")
_ax[0].axvline(0,color="k",ls="--",lw=1.3,label="1 M standard state")
_ax[0].axvline(np.log10(_emM.median()),color="C3",lw=2,label=f"median = {_emM.median()*1e3:.2f} mM")
_ax[0].set(xlabel="log10(effective molarity / M)",ylabel="interfaces",
           title="(A) Effective molarity (vs 1 M)\nEM < 1 M -> negative cooperativity"); _ax[0].legend(fontsize=9)
_ax[1].hist(np.log10(_enh),bins=24,color="#54A24B",alpha=.85,edgecolor="white")
_ax[1].axvline(0,color="k",ls="--",lw=1.3,label="C_exp = 4.5 nM (no gain)")
_ax[1].axvline(np.log10(_enh.median()),color="C3",lw=2,label=f"median = {_enh.median():.0e}x")
_ax[1].set(xlabel="log10(avidity enhancement, EM / C_exp)",ylabel="interfaces",
           title="(B) Avidity enhancement (vs 4.5 nM bulk)\nEM >> C_exp -> strong avidity"); _ax[1].legend(fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(_OUT,"EM_and_avidity_enhancement.svg"),bbox_inches="tight"); plt.show()